
**Luồng xử lý:** tin nhắn → **DST** (LLM tùy chọn + Regex + luật điền khung slot) → **Policy** (thiếu slot thì hỏi lại; đủ slot thì **gọi API/Database**) → **NLG** → trả lời.

**Chạy theo từng bước, mỗi bước 1 cell, chờ cell xong rồi mới chạy cell tiếp theo.**

| Bước | Việc | Kết quả mong đợi |
|---|---|---|
| 1 | Cài thư viện | không báo lỗi |
| 2 | Tạo file `supeo_api.py` (FastAPI) | `Writing supeo_api.py` |
| 3 | Chạy API + nạp Excel | `API: vừa khởi động` + số lượng sản phẩm/đơn/voucher |
| 4 | Test 5 endpoint | JSON trả về cho từng chức năng |
| 5-6 | Client + DST, Policy + NLG | `DST sẵn sàng`, `Policy & NLG sẵn sàng` |
| 7 | Bật LLM Gemini (API) | `Đã kết nối Gemini (...)` |
| 8 | Chạy thử hội thoại | 5 đoạn chat mẫu |
| 9 | Kiểm tra rò rỉ + đo trên bộ held-out (có test **ghi nhớ** nhiều lượt) | Intent/Slot/Action + nhớ hội thoại |
| 10 | Giao diện Gradio | link demo |

**Chuẩn bị trên Kaggle:** (1) *Settings → Internet: On*; (2) *Add Data → Upload* file `supeo_data_complete.xlsx`; (3) *Add-ons → Secrets*: thêm `GEMINI_API_KEY`.

## Bước 1 - Cài thư viện

In [1]:
!pip install -q "gradio>=5,<6" fastapi "uvicorn[standard]" requests pandas openpyxl

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dopamine-rl 4.1.2 requires gym<=0.25.2, but you have gym 0.26.2 which is incompatible.
hf-gradio 0.4.1 requires gradio-client<3.0,>=2.0, but you have gradio-client 1.14.0 which is incompatible.
google-genai 2.12.1 requires pydantic<3.0.0,>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
python-fasthtml 0.14.14 requires starlette>=1.0.1, but you have starlette 0.52.1 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.52.1 which is incompatible.


## Bước 2 - Tạo Mock API (FastAPI) đọc từ file Excel
Cell này chỉ ghi file `supeo_api.py` ra đĩa.

In [ ]:
%%writefile supeo_api.py
"""
SUPEO Mock API (FastAPI) - đọc dữ liệu từ supeo_data_complete.xlsx

Chạy độc lập:   SUPEO_XLSX=supeo_data_complete.xlsx uvicorn supeo_api:app --port 8008
Tài liệu API:   http://127.0.0.1:8008/docs

Endpoint:
  GET /v1/products?loai=&q=                 Tra cứu sản phẩm (loại, từ khóa màu/chất liệu/tên)
  GET /v1/products/{id}                     Chi tiết 1 sản phẩm
  GET /v1/size-advice?chieu_cao=&can_nang=  Tư vấn size (Max-Boundary)
  GET /v1/lookbook                          Danh sách các dịp
  GET /v1/lookbook/{occasion}               Set đồ theo dịp
  GET /v1/orders/{order_id}/tracking        Tra cứu đơn hàng (#SPxxxx)
  GET /v1/vouchers/available?cart_total=    Mã còn hạn, hien_thi_chatbot = TRUE
  GET /v1/vouchers/codes                    Danh sách mã (chỉ chuỗi mã, để chatbot nhận diện)
  GET /v1/vouchers/{ma}?cart_total=         Kiểm tra 1 mã khách nhập

Biến môi trường:
  SUPEO_XLSX   đường dẫn file Excel (mặc định tự tìm)
  SUPEO_TODAY  ngày giả lập, định dạng YYYY-MM-DD hoặc dd/mm/yyyy (mặc định: hôm nay)
"""
import glob
import os
import re
import unicodedata
from datetime import date, datetime
from typing import Optional

import pandas as pd
from fastapi import FastAPI, HTTPException, Query

XLSX_NAME = "supeo_data_complete.xlsx"


# ----------------------------------------------------------------------------
# Tiện ích
# ----------------------------------------------------------------------------
def strip_acc(s) -> str:
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("đ", "d").replace("Đ", "D")


def norm(s) -> str:
    return strip_acc(s).lower().strip()


def find_xlsx() -> str:
    cands = [os.getenv("SUPEO_XLSX"), XLSX_NAME, os.path.join("/kaggle/working", XLSX_NAME),
             os.path.join("/content", XLSX_NAME), os.path.join("/mnt/user-data/uploads", XLSX_NAME)]
    for pat in ("/kaggle/input/**/" + XLSX_NAME, "/content/**/" + XLSX_NAME):
        cands += glob.glob(pat, recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return c
    raise FileNotFoundError(
        f"Không tìm thấy {XLSX_NAME}. Hãy upload file (Kaggle: Add Data -> Upload) "
        f"hoặc đặt biến môi trường SUPEO_XLSX.")


def parse_date(x) -> Optional[date]:
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return None
    if isinstance(x, datetime):
        return x.date()
    if isinstance(x, date):
        return x
    s = str(x).strip()
    for fmt in ("%d/%m/%Y", "%Y-%m-%d", "%Y-%m-%d %H:%M:%S"):
        try:
            return datetime.strptime(s, fmt).date()
        except ValueError:
            pass
    return None


def fmt_date(d: Optional[date]) -> Optional[str]:
    return d.strftime("%d/%m/%Y") if d else None


def to_int(x, default=0) -> int:
    try:
        return int(float(x))
    except (TypeError, ValueError):
        return default


def to_bool(x) -> bool:
    return str(x).strip().upper() in ("TRUE", "1", "YES", "CÓ")


def get_today(today: Optional[str] = None) -> date:
    s = today or os.getenv("SUPEO_TODAY")
    if s:
        d = parse_date(s)
        if d:
            return d
    return date.today()


def read_sheet(path: str, sheet: str, first_col: str) -> list:
    """Sheet có dòng tiêu đề + dòng trống phía trên: tự dò dòng header theo tên cột đầu."""
    raw = pd.read_excel(path, sheet_name=sheet, header=None, dtype=object)
    hdr = None
    for i in range(len(raw)):
        if str(raw.iloc[i, 0]).strip() == first_col:
            hdr = i
            break
    if hdr is None:
        raise ValueError(f"Sheet '{sheet}': không thấy cột '{first_col}'")
    cols = [str(c).strip() for c in raw.iloc[hdr]]
    df = raw.iloc[hdr + 1:].copy()
    df.columns = cols
    df = df[df[first_col].notna()]
    rows = []
    for rec in df.to_dict("records"):
        rows.append({k: (None if (v is None or (not isinstance(v, str) and pd.isna(v))) else v)
                     for k, v in rec.items()})
    return rows


def tinh_trang(ton: int) -> str:
    if ton <= 0:
        return "Hết hàng"
    if ton < 20:
        return "Sắp hết hàng"
    return "Còn hàng"


# ----------------------------------------------------------------------------
# Database layer (đọc 1 lần khi khởi động; POST /v1/reload để nạp lại)
# ----------------------------------------------------------------------------
class DB:
    def __init__(self, path: str):
        self.path = path
        self.load()

    def load(self):
        p = self.path
        self.products = []
        for r in read_sheet(p, "products", "id"):
            ton = to_int(r.get("ton_kho"))
            self.products.append({
                "id": str(r["id"]).strip(), "ten": r["ten"], "loai": r["loai"],
                "gia": to_int(r["gia"]),
                "mau": [x.strip() for x in str(r.get("mau") or "").split(",") if x.strip()],
                "chat_lieu": r.get("chat_lieu"),
                "size": [x.strip() for x in str(r.get("size") or "").split(",") if x.strip()],
                "phom_dang": r.get("phom_dang"), "ton_kho": ton, "tinh_trang": tinh_trang(ton),
                "mo_ta": r.get("mo_ta"),
            })
        self.by_id = {x["id"]: x for x in self.products}
        self.categories = list(dict.fromkeys(x["loai"] for x in self.products))

        self.sizes = []
        for r in read_sheet(p, "size_chart", "size"):
            self.sizes.append({
                "size": str(r["size"]).strip(),
                "chieu_cao": [to_int(r["chieu_cao_min"]), to_int(r["chieu_cao_max"])],
                "can_nang": [to_int(r["can_nang_min"]), to_int(r["can_nang_max"])],
                "vong_nguc": to_int(r.get("vong_nguc")), "vong_eo": to_int(r.get("vong_eo")),
                "vong_mong": to_int(r.get("vong_mong")), "loi_khuyen": r.get("loi_khuyen_tu_van"),
            })

        self.lookbook = []
        for r in read_sheet(p, "lookbook", "dip_mac"):
            self.lookbook.append({
                "dip_mac": str(r["dip_mac"]).strip(), "ten_bo": r.get("ten_bo"),
                "product_ids": [x.strip() for x in str(r.get("product_ids") or "").split(",") if x.strip()],
                "tong_gia_excel": to_int(r.get("tong_gia_tien")),
                "phong_cach": r.get("phong_cach"), "ghi_chu": r.get("ghi_chu_tu_van"),
            })

        self.orders = {}
        for r in read_sheet(p, "orders", "ma_don"):
            ma = str(r["ma_don"]).strip().upper()
            self.orders[ma] = {
                "ma_don": ma, "trang_thai": r.get("trang_thai"),
                "ngay_dat": fmt_date(parse_date(r.get("ngay_dat"))),
                "du_kien": fmt_date(parse_date(r.get("du_kien"))),
                "dvvc": r.get("dvvc"), "ma_van_don": r.get("ma_van_don"),
                "san_pham": [x.strip() for x in str(r.get("san_pham") or "").split(";") if x.strip()],
                "tong_tien": to_int(r.get("tong_tien")),
            }

        self.vouchers = []
        for r in read_sheet(p, "vouchers", "ma"):
            mo_ta = str(r.get("mo_ta") or "")
            m = re.search(r"(\d+)\s*%", mo_ta)
            if m:
                loai_giam = "phan_tram"
            elif "vận chuyển" in mo_ta.lower():
                loai_giam = "van_chuyen"
            else:
                loai_giam = "tien_mat"
            self.vouchers.append({
                "ma": str(r["ma"]).strip(), "mo_ta": mo_ta, "giam_gia": to_int(r.get("giam_gia")),
                "don_toi_thieu": to_int(r.get("don_toi_thieu")), "han": parse_date(r.get("han_dung")),
                "doi_tuong": r.get("doi_tuong"), "hien_thi": to_bool(r.get("hien_thi_chatbot")),
                "dieu_khoan": r.get("dieu_khoan"), "loai_giam": loai_giam,
                "pct": int(m.group(1)) if m else None,
            })


db: Optional[DB] = None


def get_db() -> DB:
    global db
    if db is None:
        db = DB(find_xlsx())
    return db


app = FastAPI(title="SUPEO Mock API", version="1.0",
              description="Mock API/Database layer cho chatbot SUPEO (SGP-TOD). Nguồn: supeo_data_complete.xlsx")


# ----------------------------------------------------------------------------
# Health
# ----------------------------------------------------------------------------
@app.get("/")
@app.get("/health")
def health():
    d = get_db()
    return {"status": "ok", "file": os.path.basename(d.path), "today": fmt_date(get_today()),
            "so_luong": {"san_pham": len(d.products), "lookbook": len(d.lookbook),
                         "don_hang": len(d.orders), "voucher": len(d.vouchers)}}


@app.post("/v1/reload")
def reload_db():
    get_db().load()
    return {"status": "reloaded"}


# ----------------------------------------------------------------------------
# 1. Products
# ----------------------------------------------------------------------------
def _match_loai(user_loai: str, cats: list) -> Optional[str]:
    n = norm(user_loai)
    for c in cats:
        if norm(c) == n:
            return c
    for c in cats:
        if n and (n in norm(c) or norm(c) in n):
            return c
    return None


@app.get("/v1/products")
def products(loai: Optional[str] = None, q: Optional[str] = None,
             con_hang: Optional[bool] = None, gia_max: Optional[int] = None,
             limit: int = Query(50, ge=1, le=200)):
    d = get_db()
    items = d.products
    loai_hd = None
    if loai:
        loai_hd = _match_loai(loai, d.categories)
        items = [x for x in items if x["loai"] == loai_hd] if loai_hd else []
    if q:
        toks = [t for t in re.findall(r"[a-z0-9]+", norm(q))]
        def hit(x):
            hay = norm(" ".join([x["ten"], ",".join(x["mau"]), str(x["chat_lieu"]),
                                 str(x["phom_dang"]), str(x["mo_ta"])]))
            return all(re.search(r"\b" + re.escape(t) + r"\b", hay) for t in toks)
        items = [x for x in items if hit(x)]
    if con_hang:
        items = [x for x in items if x["ton_kho"] > 0]
    if gia_max:
        items = [x for x in items if x["gia"] <= gia_max]
    items = items[:limit]
    return {"count": len(items), "loai": loai_hd, "q": q, "danh_muc": d.categories, "items": items}


@app.get("/v1/products/{pid}")
def product_detail(pid: str):
    x = get_db().by_id.get(pid.strip().upper())
    if not x:
        raise HTTPException(404, detail={"loi": "khong_tim_thay_san_pham", "id": pid})
    return x


# ----------------------------------------------------------------------------
# 2. Size (Max-Boundary)
# ----------------------------------------------------------------------------
def _band(value: float, sizes: list, key: str) -> int:
    """Chỉ số dải đầu tiên có value <= max. Thấp hơn min -> dải đầu; vượt max cuối -> dải cuối."""
    for i, s in enumerate(sizes):
        if value <= s[key][1]:
            return i
    return len(sizes) - 1


@app.get("/v1/size-advice")
def size_advice(chieu_cao: float = Query(..., ge=140, le=200), can_nang: float = Query(..., ge=30, le=120)):
    sizes = get_db().sizes
    ih, iw = _band(chieu_cao, sizes, "chieu_cao"), _band(can_nang, sizes, "can_nang")
    chosen = sizes[max(ih, iw)]                      # Max-Boundary: lệch dải -> chọn size lớn hơn
    return {
        "chieu_cao": chieu_cao, "can_nang": can_nang,
        "theo_chieu_cao": {"size": sizes[ih]["size"], "khoang": sizes[ih]["chieu_cao"]},
        "theo_can_nang": {"size": sizes[iw]["size"], "khoang": sizes[iw]["can_nang"]},
        "lech": ih != iw,
        "de_xuat": {"size": chosen["size"], "chieu_cao": chosen["chieu_cao"], "can_nang": chosen["can_nang"],
                    "vong_nguc": chosen["vong_nguc"], "vong_eo": chosen["vong_eo"],
                    "vong_mong": chosen["vong_mong"], "loi_khuyen": chosen["loi_khuyen"]},
        "ngoai_dai": {"chieu_cao": chieu_cao < sizes[0]["chieu_cao"][0] or chieu_cao > sizes[-1]["chieu_cao"][1],
                      "can_nang": can_nang < sizes[0]["can_nang"][0] or can_nang > sizes[-1]["can_nang"][1]},
        "thu_tu_size": [s["size"] for s in sizes],
    }


# ----------------------------------------------------------------------------
# 3. Lookbook
# ----------------------------------------------------------------------------
@app.get("/v1/lookbook")
def lookbook_list():
    return {"dip_mac": [x["dip_mac"] for x in get_db().lookbook]}


@app.get("/v1/lookbook/{occasion}")
def lookbook(occasion: str):
    d = get_db()
    n = norm(occasion)
    found = [x for x in d.lookbook if norm(x["dip_mac"]) == n]
    if not found:
        found = [x for x in d.lookbook if n and (n in norm(x["dip_mac"]) or norm(x["dip_mac"]) in n)]
    if len(found) != 1:
        raise HTTPException(404, detail={"loi": "khong_tim_thay_dip", "dip": occasion,
                                         "goi_y": [x["dip_mac"] for x in d.lookbook]})
    lb = found[0]
    items = [d.by_id[i] for i in lb["product_ids"] if i in d.by_id]
    tong = sum(x["gia"] for x in items)
    return {"dip_mac": lb["dip_mac"], "ten_bo": lb["ten_bo"], "phong_cach": lb["phong_cach"],
            "ghi_chu": lb["ghi_chu"], "items": items, "tong_gia_tien": tong,
            "tong_gia_excel": lb["tong_gia_excel"], "khop_gia": tong == lb["tong_gia_excel"]}


# ----------------------------------------------------------------------------
# 4. Orders
# ----------------------------------------------------------------------------
@app.get("/v1/orders/{order_id}/tracking")
def order_tracking(order_id: str):
    ma = re.sub(r"[\s\-]", "", order_id).upper()
    if not re.fullmatch(r"#?SP\d{3,}", ma):
        raise HTTPException(422, detail={"loi": "sai_dinh_dang", "ma_don": order_id,
                                         "dinh_dang": "#SPxxxx (ví dụ #SP2908)"})
    if not ma.startswith("#"):
        ma = "#" + ma
    o = get_db().orders.get(ma)
    if not o:
        raise HTTPException(404, detail={"loi": "khong_tim_thay_don", "ma_don": ma})
    return {**o, "da_ban_giao_vc": bool(o["ma_van_don"])}


# ----------------------------------------------------------------------------
# 5. Vouchers
# ----------------------------------------------------------------------------
def _estimate(v: dict, amount: int) -> int:
    if v["pct"]:
        return min(round(amount * v["pct"] / 100), v["giam_gia"])
    return v["giam_gia"]


def _view(v: dict, today: date, cart: Optional[int]) -> dict:
    out = {"ma": v["ma"], "mo_ta": v["mo_ta"], "giam_gia": v["giam_gia"], "don_toi_thieu": v["don_toi_thieu"],
           "han_dung": fmt_date(v["han"]), "con_lai_ngay": (v["han"] - today).days if v["han"] else None,
           "doi_tuong": v["doi_tuong"], "dieu_khoan": v["dieu_khoan"], "loai_giam": v["loai_giam"]}
    if cart is not None:
        du = cart >= v["don_toi_thieu"]
        out.update(du_dieu_kien=du, con_thieu=0 if du else v["don_toi_thieu"] - cart,
                   uoc_tinh_giam=_estimate(v, cart) if du else None)
    return out


@app.get("/v1/vouchers/available")
def vouchers_available(cart_total: Optional[int] = Query(None, ge=0), today: Optional[str] = None):
    t = get_today(today)
    vs = [v for v in get_db().vouchers if v["hien_thi"] and v["han"] and v["han"] >= t]
    views = [_view(v, t, cart_total) for v in vs]
    best, upsell = None, None
    if cart_total is None:
        views.sort(key=lambda x: x["don_toi_thieu"])
    else:
        ok = [x for x in views if x["du_dieu_kien"]]
        no = [x for x in views if not x["du_dieu_kien"]]
        ok.sort(key=lambda x: -x["uoc_tinh_giam"])
        no.sort(key=lambda x: x["con_thieu"])
        views = ok + no
        if ok:
            best = ok[0]["ma"]
        if no:
            nxt = no[0]
            src = next(v for v in vs if v["ma"] == nxt["ma"])
            upsell = {"ma": nxt["ma"], "con_thieu": nxt["con_thieu"], "don_toi_thieu": nxt["don_toi_thieu"],
                      "uoc_tinh_giam_neu_du": _estimate(src, nxt["don_toi_thieu"]), "mo_ta": nxt["mo_ta"]}
    return {"today": fmt_date(t), "cart_total": cart_total, "count": len(views),
            "tot_nhat": best, "goi_y_upsell": upsell, "vouchers": views}


@app.get("/v1/vouchers/codes")
def voucher_codes():
    return {"codes": [v["ma"] for v in get_db().vouchers]}


@app.get("/v1/vouchers/{ma}")
def voucher_check(ma: str, cart_total: Optional[int] = Query(None, ge=0), today: Optional[str] = None):
    t = get_today(today)
    v = next((x for x in get_db().vouchers if x["ma"].upper() == ma.strip().upper()), None)
    # Mã nội bộ / cá nhân hóa (hien_thi_chatbot = FALSE) không được tiết lộ ra chatbot
    if v is None or not v["hien_thi"]:
        return {"ma": ma.strip().upper(), "trang_thai": "khong_kha_dung"}
    view = _view(v, t, cart_total)
    if v["han"] and v["han"] < t:
        return {**view, "trang_thai": "het_han"}
    if cart_total is not None and not view["du_dieu_kien"]:
        return {**view, "trang_thai": "chua_du_dieu_kien"}
    return {**view, "trang_thai": "hop_le"}


Writing supeo_api.py


## Bước 3 - Tìm file Excel và chạy API ở thread nền

In [3]:
import os, glob, time, threading, requests, uvicorn

# --- 1) Tìm file dữ liệu Excel ---------------------------------------------------
def ensure_xlsx(name="supeo_data_complete.xlsx"):
    cands = [os.environ.get("SUPEO_XLSX"), name, f"/kaggle/working/{name}", f"/content/{name}"]
    for pat in (f"/kaggle/input/**/{name}", f"/content/**/{name}"):
        cands += glob.glob(pat, recursive=True)
    for c in cands:
        if c and os.path.exists(c):
            return os.path.abspath(c)
    try:                                           # Google Colab: cho upload tay
        from google.colab import files
        print("Chọn file supeo_data_complete.xlsx để upload...")
        files.upload()
        if os.path.exists(name):
            return os.path.abspath(name)
    except ImportError:
        pass
    raise FileNotFoundError(
        "Không thấy supeo_data_complete.xlsx. Trên Kaggle: Add Data -> Upload -> chọn file Excel, rồi chạy lại cell này.")

os.environ["SUPEO_XLSX"] = ensure_xlsx()
# Ngày KHÔNG cố định ở đây: các cell đo (9c, 9d, 9e) tự đặt SUPEO_TODAY=2026-10-28 bằng fixed_today() rồi trả lại giá trị cũ.
print("Dữ liệu:", os.environ["SUPEO_XLSX"])

# --- 2) Chạy FastAPI trong thread nền ---------------------------------------------
API_PORT = 8008
API_URL = f"http://127.0.0.1:{API_PORT}"

def start_api():
    try:
        if requests.get(API_URL + "/health", timeout=1).ok:
            return "đã chạy sẵn"
    except requests.RequestException:
        pass
    import importlib, supeo_api
    importlib.reload(supeo_api)
    server = uvicorn.Server(uvicorn.Config(supeo_api.app, host="127.0.0.1", port=API_PORT, log_level="warning"))
    threading.Thread(target=server.run, daemon=True).start()
    for _ in range(50):
        try:
            if requests.get(API_URL + "/health", timeout=1).ok:
                return "vừa khởi động"
        except requests.RequestException:
            time.sleep(0.2)
    raise RuntimeError("API không khởi động được")

print("API:", start_api())
print(requests.get(API_URL + "/health").json())
print("Swagger UI (khi chạy máy local):", API_URL + "/docs")


Dữ liệu: /kaggle/input/datasets/su6587gy/fashion/supeo_data_complete.xlsx
API: vừa khởi động
{'status': 'ok', 'file': 'supeo_data_complete.xlsx', 'today': '17/11/2026', 'so_luong': {'san_pham': 30, 'lookbook': 12, 'don_hang': 12, 'voucher': 7}}
Swagger UI (khi chạy máy local): http://127.0.0.1:8008/docs


## Bước 4 - Test nhanh các endpoint (4 endpoint đặc tả + size-advice)

In [4]:
# Kiểm tra nhanh 4 endpoint chính + size-advice. Nếu cell này in ra đúng => lớp API/Database đã sẵn sàng.
import requests, json
from urllib.parse import quote
def show(title, path, **params):
    r = requests.get(API_URL + path, params=params)
    print(f"\n### {title}\nGET {path} {params or ''} -> {r.status_code}")
    print(json.dumps(r.json(), ensure_ascii=False)[:420])

show("Chức năng 1 - Tra cứu sản phẩm", "/v1/products", loai="Đầm", q="đen")
show("Chức năng 2 - Tư vấn size (lệch dải -> chọn size lớn)", "/v1/size-advice", chieu_cao=162, can_nang=59)
show("Chức năng 3 - Lookbook", "/v1/lookbook/" + quote("Trà chiều", safe=""))
show("Chức năng 4 - Đơn hàng", "/v1/orders/" + quote("#SP2908", safe="") + "/tracking")
show("Chức năng 4 - Đơn không tồn tại", "/v1/orders/" + quote("#SP9999", safe="") + "/tracking")
show("Chức năng 5 - Voucher theo giỏ hàng 750k", "/v1/vouchers/available", cart_total=750000)



### Chức năng 1 - Tra cứu sản phẩm
GET /v1/products {'loai': 'Đầm', 'q': 'đen'} -> 200
{"count": 3, "loai": "Đầm", "q": "đen", "danh_muc": ["Áo Blazer", "Chân váy Midi", "Áo sơ mi", "Quần ống rộng", "Đầm"], "items": [{"id": "P10", "ten": "Đầm suông dự tiệc", "loai": "Đầm", "gia": 1250000, "mau": ["Đen", "Đỏ đô"], "chat_lieu": "Satin cao cấp", "size": ["S", "M", "L"], "phom_dang": "Suông xẻ tà", "ton_kho": 22, "tinh_trang": "Còn hàng", "mo_ta": "Dáng suông quý phái, chi tiết xẻ tà nhẹ bên đùi tạo bước đ

### Chức năng 2 - Tư vấn size (lệch dải -> chọn size lớn)
GET /v1/size-advice {'chieu_cao': 162, 'can_nang': 59} -> 200
{"chieu_cao": 162.0, "can_nang": 59.0, "theo_chieu_cao": {"size": "M", "khoang": [160, 167]}, "theo_can_nang": {"size": "L", "khoang": [57, 64]}, "lech": true, "de_xuat": {"size": "L", "chieu_cao": [168, 174], "can_nang": [57, 64], "vong_nguc": 88, "vong_eo": 72, "vong_mong": 96, "loi_khuyen": "Dành cho chiều cao từ 1m68 hoặc cân nặng từ 57kg. Form thoải mái khi vận đ

## Bước 5 - Client gọi API + Schema + DST
`api_get()` dùng `requests`. Danh mục, dịp mặc, mã voucher đều **nạp từ API**, không hardcode.

In [5]:
import re, json, time, unicodedata, requests
from urllib.parse import quote

API_URL = globals().get("API_URL", "http://127.0.0.1:8008")
API_LOG = []        # nhật ký các lần gọi API của lượt hội thoại hiện tại (hiện ở panel debug)
USE_LLM = False     # cell LLM bên dưới sẽ bật lên True nếu tải được mô hình

class ApiDown(Exception):
    pass

def strip_acc(s):
    s = unicodedata.normalize("NFD", str(s))
    s = "".join(c for c in s if unicodedata.category(c) != "Mn")
    return s.replace("đ", "d").replace("Đ", "D")

def vnd(n):
    return f"{int(n):,}".replace(",", ".") + "đ"

def api_get(path, **params):
    """GET tới Mock API. Trả (status_code, json). Mất kết nối -> ApiDown."""
    params = {k: v for k, v in params.items() if v is not None and v != ""}
    log = {"GET": path, "params": params}
    try:
        r = requests.get(API_URL + path, params=params, timeout=8)
        log["status"] = r.status_code
        data = r.json()
    except Exception as e:
        log["status"], log["error"] = 0, str(e)[:100]
        API_LOG.append(log)
        raise ApiDown(str(e))
    API_LOG.append(log)
    return r.status_code, data

# --- Nạp "từ vựng" từ chính API (không hardcode danh mục/dịp/mã) ----------------
CATALOG, CATEGORIES, OCCASIONS, VOUCHER_CODES = [], [], [], []
DISTINCT, COLOR_LEX, UNIQUE_TOK = {}, {}, {}

CAT_WORDS = {"ao", "chan", "vay", "midi", "blazer", "so", "mi", "quan", "ong", "rong", "dam"}
STOP = {"co", "la", "cho", "va", "voi", "tay", "dang", "mau", "bo", "can", "cao"}
MATERIAL_LEX = {"linen": "linen", "satin": "satin", "tuyet mua": "tuyết mưa", "kaki": "kaki", "cotton": "cotton", "voan": "voan",
                "tweed": "tweed", "dui": "đũi", "thun": "thun", "chiffon": "chiffon", "poplin": "poplin", "oxford": "oxford", "lua": "lụa"}

def load_lexicons():
    global CATALOG, CATEGORIES, OCCASIONS, VOUCHER_CODES, DISTINCT, COLOR_LEX, UNIQUE_TOK
    _, d = api_get("/v1/products", limit=200)
    CATALOG, CATEGORIES = d["items"], d["danh_muc"]
    _, d = api_get("/v1/lookbook");       OCCASIONS = d["dip_mac"]
    _, d = api_get("/v1/vouchers/codes"); VOUCHER_CODES = d["codes"]
    DISTINCT = {}
    for p in CATALOG:                                  # từ "đặc trưng" của tên từng sản phẩm
        toks = set(re.findall(r"[a-z0-9]+", strip_acc(p["ten"]).lower()))
        DISTINCT[p["id"]] = {t for t in toks - CAT_WORDS - STOP if len(t) > 1}
    cnt = {}                                           # từ chỉ xuất hiện ở ĐÚNG 1 sản phẩm (culottes, babydoll...) -> nhận diện món
    for pid, toks in DISTINCT.items():
        for tk in toks:
            cnt.setdefault(tk, []).append(pid)
    UNIQUE_TOK = {tk: ids[0] for tk, ids in cnt.items() if len(ids) == 1}
    cols = {strip_acc(c).lower(): c.lower() for p in CATALOG for c in p["mau"]}       # {không dấu: có dấu}
    COLOR_LEX = dict(sorted(cols.items(), key=lambda kv: -len(kv[0])))
    API_LOG.clear()
    return {"san_pham": len(CATALOG), "danh_muc": CATEGORIES, "dip": OCCASIONS, "ma_voucher": VOUCHER_CODES}

print(load_lexicons())


{'san_pham': 30, 'danh_muc': ['Áo Blazer', 'Chân váy Midi', 'Áo sơ mi', 'Quần ống rộng', 'Đầm'], 'dip': ['Hẹn hò', 'Trà chiều', 'Công sở', 'Tiệc tối', 'Dạo phố', 'Gặp đối tác', 'Du lịch hè', 'Tiệc cocktail', 'Casual Friday', 'Hẹn hò thơ mộng', 'Dạo phố mùa thu', 'Dự tiệc ngoài trời'], 'ma_voucher': ['SUPEOVIP100', 'FLASHDEAL20', 'FREESHIPMAX', 'SUPEOAUTUMN', 'SUPEOSHIP30', 'WELCOME10', 'SUPEOBIRTHDAY']}


In [ ]:
# ============ BƯỚC 1: KHAI BÁO SCHEMA (SGP-TOD) ============
INTENTS = ["product_info", "size_advice", "lookbook", "order_check", "voucher", "chitchat"]
SCHEMA = {                       # 6 slot cốt lõi + 3 slot phụ trợ
    "loai_do":   "Áo Blazer | Chân váy Midi | Áo sơ mi | Quần ống rộng | Đầm",
    "dip_mac":   "dịp mặc (Hẹn hò, Trà chiều, Công sở, Tiệc tối, Dạo phố, Gặp đối tác...)",
    "chieu_cao": "cm (int)", "can_nang": "kg (int)",
    "ma_don":    "#SPxxxx", "ma_voucher": "mã giảm giá khách nhập",
    "ten_sp": "tên sản phẩm cụ thể (khớp danh mục)", "tu_khoa": "màu / chất liệu", "cart_total": "giá trị giỏ hàng (VNĐ)",
}
NEEDED = {                       # slot bắt buộc theo intent (điều kiện đủ ở hàm missing)
    "product_info": ["loai_do"], "size_advice": ["chieu_cao", "can_nang"],
    "lookbook": ["dip_mac"], "order_check": ["ma_don"], "voucher": [], "chitchat": [],
}
FOLLOW_KEYS = {"product_info": ["loai_do", "ten_sp", "tu_khoa"], "size_advice": ["chieu_cao", "can_nang"],
               "lookbook": ["dip_mac"], "order_check": ["ma_don"]}

def missing(intent, S):
    if intent == "product_info":
        return [] if any(k in S for k in ("loai_do", "ten_sp", "tu_khoa")) else ["loai_do"]
    return [k for k in NEEDED.get(intent, []) if k not in S]

# Từ khóa nhận diện (đã bỏ dấu). Thứ tự = độ ưu tiên: cụ thể trước, chung sau.
CATS = {
    "Đầm": [r"dam"],
    "Áo Blazer": [r"blazer", r"vest"],
    "Áo sơ mi": [r"so mi"],
    "Chân váy Midi": [r"chan vay", r"vay midi", r"midi"],
    "Quần ống rộng": [r"quan(?! trong| ao)", r"culottes"],
}
DIPS = {
    "Hẹn hò thơ mộng": [r"hen ho tho mong", r"tho mong"],
    "Hẹn hò": [r"hen ho"],
    "Gặp đối tác": [r"doi tac"],
    "Công sở": [r"cong so", r"di lam"],
    "Casual Friday": [r"casual"],
    "Du lịch hè": [r"du lich"],
    "Tiệc cocktail": [r"cocktail"],
    "Dự tiệc ngoài trời": [r"ngoai troi", r"garden party"],
    "Tiệc tối": [r"tiec toi", r"du tiec", r"tiec"],
    "Trà chiều": [r"tra chieu"],
    "Dạo phố mùa thu": [r"dao pho mua thu", r"dao pho thu"],   # nhóm chung nhất đặt CUỐI ('cuối tuần đi tiệc' phải ra Tiệc, không phải Dạo phố)
    "Dạo phố": [r"dao pho", r"di choi", r"cuoi tuan"],
}

NEG = re.compile(r"\bchu\s+(?:khong|dung)\b|\bkhong\s+(?:phai|muon|can)\b")

def positive_part(text):
    """Bỏ vế phủ định: 'váy chứ không phải quần ống rộng' -> 'váy'; 'không phải quần, mà là váy' -> 'váy'."""
    nfc = unicodedata.normalize("NFC", text).lower()
    t = strip_acc(nfc)
    m = NEG.search(t) if len(t) == len(nfc) else None
    if not m:
        return nfc
    before = nfc[:m.start()].strip(" ,.;")
    if re.search(r"\w", strip_acc(before)):
        return before
    m2 = re.search(r"\b(?:ma|la)\s+(.*)$", t[m.end():])
    return nfc[m.end():][m2.start(1):] if m2 else nfc

def is_generic_vay(ptxt):
    """'váy' chung (đầm + chân váy). Cần dấu để không nhầm với 'vậy'; không dấu thì phải đi sau mua/xem/tìm..."""
    return bool(re.search(r"\bváy\b", ptxt) or re.search(r"\b(?:mua|xem|tim|muon|can|cho)\s+vay\b", ptxt))

ORD = {"nhat": 1, "dau tien": 1, "dau": 1, "hai": 2, "ba": 3, "bon": 4, "nam": 5, "sau": 6, "bay": 7, "tam": 8, "chin": 9, "muoi": 10}
ORD_RE = r"\b(?:mau|cai|so|thu)\s+(?:thu\s+)?(nhat|dau tien|dau|hai|ba|bon|nam|sau|bay|tam|chin|muoi|\d{1,2})\b"

def pick_item(text, lst):
    """Khách chọn mẫu bằng mã (P03) hoặc số thứ tự trong danh sách bot vừa gửi ('mẫu thứ 2', 'cái đầu tiên')."""
    t = strip_acc(text).lower()
    m = re.search(r"\bp(\d{2})\b", t)
    if m:
        p = next((x for x in CATALOG if x["id"] == "P" + m.group(1)), None)
        if p:
            return p["ten"]
    if lst:
        m = re.search(ORD_RE, t)
        if m:
            k = ORD.get(m.group(1)) or (int(m.group(1)) if m.group(1).isdigit() else None)
            if k and 1 <= k <= len(lst):
                return lst[k - 1]
    return None

def match_key(text, table):
    t = strip_acc(text).lower()
    for key, pats in table.items():
        if any(re.search(r"\b" + p + r"\b", t) for p in pats):
            return key
    return None

def match_product(text, cat):
    """Khớp tên sản phẩm trong danh mục (tải từ API). Ưu tiên tên đầy đủ, sau đó từ đặc trưng trong cùng loại."""
    t = strip_acc(text).lower()
    words = set(re.findall(r"[a-z0-9]+", t))
    best = None
    for p in CATALOG:
        if strip_acc(p["ten"]).lower() in t and (best is None or len(p["ten"]) > len(best["ten"])):
            best = p
    if best:
        return best["ten"]
    uniq = [p for p in CATALOG if (not cat or p["loai"] == cat)
            and any(len(tk) >= 7 and tk in words and UNIQUE_TOK.get(tk) == p["id"] for tk in DISTINCT.get(p["id"], ()))]
    if len(uniq) == 1:                                       # vd "culottes", "babydoll", "croptop"
        return uniq[0]["ten"]
    if not cat:
        return None
    scored = []
    for p in CATALOG:
        d = DISTINCT.get(p["id"])
        if p["loai"] == cat and d:
            hit = len(d & words)
            if hit:
                scored.append((hit / len(d), p["ten"]))
    scored.sort(reverse=True)
    if scored and scored[0][0] >= 0.6 and (len(scored) == 1 or scored[0][0] > scored[1][0]):
        return scored[0][1]
    return None

def match_keywords(text):
    t = strip_acc(text).lower()
    kws = [vi for k, vi in MATERIAL_LEX.items() if re.search(r"\b" + k + r"\b", t)]
    kws += [vi for k, vi in COLOR_LEX.items() if re.search(r"\bmau\s+" + re.escape(k) + r"\b", t)][:1]
    return " ".join(kws) or None

def parse_money(text):
    t = strip_acc(text).lower()
    val = None
    m = re.search(r"\b(\d+)\s?(?:tr|trieu)\s?(\d{1,3})?\b", t)
    if m:
        val = int(m.group(1)) * 1_000_000
        if m.group(2):
            frac = m.group(2)
            val += int(frac) * {1: 100_000, 2: 10_000, 3: 1_000}[len(frac)]
    if val is None:
        m = re.search(r"\b(\d+(?:[.,]\d+)?)\s?(?:k|nghin|ngan)\b", t)
        if m:
            val = int(float(m.group(1).replace(",", ".")) * 1000)
    if val is None:
        m = re.search(r"\b(\d{1,3}(?:[.,]\d{3})+)\b", t) or re.search(r"\b(\d{6,8})\b", t)
        if m:
            val = int(re.sub(r"[.,]", "", m.group(1)))
    return val if val and 50_000 <= val <= 100_000_000 else None

def regex_slots(text):
    t = strip_acc(text).lower()
    out = {}
    m = re.search(r"#?\bsp\s?-?(\d{3,})\b", t)
    if m:
        out["ma_don"] = "#SP" + m.group(1)
    m = re.search(r"\b1\s?m\s?(\d{1,2})\b", t)                               # 1m65, 1m6
    if m:
        n = m.group(1)
        out["chieu_cao"] = 100 + (int(n) * 10 if len(n) == 1 else int(n))
    else:
        m = re.search(r"\b(1[4-9]\d|20\d)\s?(?:cm|xang|phan)\b", t) or re.search(r"cao\s*(?:khoang\s*)?(1[4-9]\d|20\d)\b", t)
        if m:
            out["chieu_cao"] = int(m.group(1))
    m = re.search(r"\b(\d{2,3})\s?(?:kg|ki|kilo|ky)\b", t) or re.search(r"\bnang\s*(?:khoang\s*)?(\d{2,3})\b", t)
    if m:
        out["can_nang"] = int(m.group(1))
    for code in VOUCHER_CODES:                                                # mã đã biết (không phân biệt hoa thường)
        if re.search(r"\b" + code.lower() + r"\b", t):
            out["ma_voucher"] = code
            break
    else:                                                                     # mã lạ viết HOA: SUMMER2026...
        m = re.search(r"\b(?!SP\d)[A-Z][A-Z0-9]{5,}\b", text)
        if m and m.group(0) != "SUPEO":
            out["ma_voucher"] = m.group(0)
    return out

def field_asked(text):
    t = strip_acc(text).lower()
    if re.search(r"\bsize\b", t) and re.search(r"(nhung|may|co) size|size nao|size gi co|du size", t):
        return "size"
    if re.search(r"chat lieu|chat vai|vai gi|vai nao|lam bang", t):
        return "chat_lieu"
    nfc = unicodedata.normalize("NFC", text).lower()
    # "màu" (có dấu) khác "mẫu": bỏ dấu thì "mẫu thứ 2 giá bao nhiêu" bị hiểu nhầm là hỏi màu
    if re.search(r"\bmàu\b", nfc) if strip_acc(nfc) != nfc else re.search(r"\bmau\b", t):
        return "mau"
    if re.search(r"con hang|het hang|ton kho|con khong|con mau", t):
        return "ton_kho"
    if re.search(r"\bgia\b|bao nhieu|bao nhieu tien|nhieu tien", t):
        return "gia"
    if re.search(r"hinh anh|xem hinh|\b(?:xem|gui|coi|thay)\s+(?:duoc\s+)?(?:cai\s+|nhung\s+)?anh\b|\banh\s+(?:san pham|mau|that|thuc te)\b", t):
        return "anh"                                              # khách muốn xem ảnh sản phẩm
    return None

PERSONAL_SIZE = (r"\b(?:minh|em|toi|chi|con)\s+(?:mac|lay|chon|dung)\b|\bnen\s+(?:mac|lay|chon)\b|\bmac size\b"
                 r"|\bsize cua (?:minh|em|toi)\b|\bhop size\b")

def rule_intent(text, new, state_slots):
    t = strip_acc(text).lower()
    S = {**state_slots, **new}
    if "ma_don" in new or re.search(r"don hang|van don|check don|kiem tra don|tra cuu don|theo doi don|giao toi dau|ship toi dau|don cua minh", t):
        return "order_check"
    if "ma_voucher" in new or "cart_total" in new or re.search(r"voucher|ma giam|giam gia|khuyen mai|uu dai|gio hang|ap ma|ma nao|coupon|freeship", t):
        return "voucher"
    asks_size_list = (field_asked(text) == "size" and "chieu_cao" not in new and "can_nang" not in new
                      and not re.search(PERSONAL_SIZE, t))
    if not asks_size_list and ("chieu_cao" in new or "can_nang" in new or re.search(r"\bsize\b|mac vua|co nguoi|nang bao nhieu|cao bao nhieu", t)):
        return "size_advice"
    if "dip_mac" in new or re.search(r"phoi|lookbook|goi y (?:bo|set|do|outfit)|set do|outfit", t):
        return "lookbook"
    if any(k in new for k in ("loai_do", "ten_sp", "tu_khoa")) or field_asked(text) or re.search(r"mau (?:nao|gi) |san pham|mat hang", t):
        return "product_info"
    return "chitchat"

# ============ BƯỚC 2: ĐIỀN KHUNG (DST) bằng LLM (tùy chọn) + Regex + luật ============
DST_SYS = """Bạn là bộ trích xuất thông tin (DST) cho chatbot thời trang SUPEO.
Trả về DUY NHẤT một JSON hợp lệ, không giải thích:
{"intent": <product_info|size_advice|lookbook|order_check|voucher|chitchat>,
 "slots": {"loai_do": <Áo Blazer|Chân váy Midi|Áo sơ mi|Quần ống rộng|Đầm|null>,
           "dip_mac": <tên dịp mặc hoặc null>, "chieu_cao": <số cm hoặc null>, "can_nang": <số kg hoặc null>,
           "ma_don": <#SPxxxx hoặc null>}}
Chỉ lấy thông tin trong tin nhắn mới nhất; thiếu thì để null.
Dùng "Lịch sử gần nhất" và "Đang nói về" để hiểu các từ như 'cái này', 'mẫu đó', 'đơn đó', nhưng chỉ điền slot khi tin nhắn mới có nhắc hoặc ám chỉ rõ."""

def dst_input(text, last_bot, cur_intent, hist="", focus=""):
    return (f"Lịch sử gần nhất:\n{hist or 'không'}\nĐang nói về: {focus or 'chưa có'}\n"
            f"Bot vừa hỏi: {last_bot or 'không'}\nIntent đang xử lý: {cur_intent}\nTin nhắn khách: {text}")

_E = '"loai_do":%s,"dip_mac":%s,"chieu_cao":%s,"can_nang":%s,"ma_don":%s'
FEWSHOT = [      # ví dụ TỰ VIẾT, cố ý KHÔNG trùng câu nào trong dialogue_benchmarks / bộ held-out (xem cell kiểm tra rò rỉ)
    ("Chị cao 1m72 thì lấy size nào nhỉ?", "none", "chitchat",
     '{"intent":"size_advice","slots":{' + _E % ("null", "null", "172", "null", "null") + '}}'),
    ("Tra giúp em đơn SP4455", "none", "chitchat",
     '{"intent":"order_check","slots":{' + _E % ("null", "null", "null", "null", '"#SP4455"') + '}}'),
    ("60 cân", "Nàng cho SUPEO xin cân nặng (kg) nhé!", "size_advice",
     '{"intent":"size_advice","slots":{' + _E % ("null", "null", "null", "60", "null") + '}}'),
    ("Mình muốn xem blazer cho đi làm", "none", "chitchat",
     '{"intent":"product_info","slots":{' + _E % ('"Áo Blazer"', '"Công sở"', "null", "null", "null") + '}}'),
    ("Có freeship không bạn", "none", "chitchat",
     '{"intent":"voucher","slots":{' + _E % ("null", "null", "null", "null", "null") + '}}'),
    ("Hello bạn", "none", "chitchat",
     '{"intent":"chitchat","slots":{' + _E % ("null", "null", "null", "null", "null") + '}}'),
]

def llm_dst(text, state):
    msgs = [{"role": "system", "content": DST_SYS}]
    for u, lb, ci, a in FEWSHOT:
        msgs += [{"role": "user", "content": dst_input(u, lb, ci)}, {"role": "assistant", "content": a}]
    msgs.append({"role": "user", "content": dst_input(text, state["last_bot"], state["intent"],
                                                      recent_history(state), focus_text(state))})
    try:
        raw = llm(msgs, 120)
        return json.loads(re.search(r"\{.*\}", raw, re.S).group(0))
    except Exception:
        return None

FOLLOW_ORDER = r"\b(?:giao|khi nao|bao gio|toi noi|den noi|van chuyen|van don|trang thai|hoan|huy|ship|don nay|don do|don kia)\b"
DEICTIC_ACC = re.compile(r"\b(?:áo|đầm|váy|quần|blazer|mẫu|sản phẩm|món|cái|set|bộ)\s+(?:này|đó|kia|ấy|vừa rồi|trên)\b")
DEICTIC_NOACC = re.compile(r"\b(?:ao|dam|vay|quan|blazer|san pham|mon|cai)\s+(?:nay|kia|ay)\b|\bmau\s+(?:nay|kia)\b|\bcai do\b")

def resolve_reference(text, ptxt, cat, state):
    """Hiểu "áo này", "mẫu đó", hoặc câu hỏi tiếp (giá? còn hàng không?) mà không nêu món -> món đang nói tới (bộ nhớ focus)."""
    f = state.get("focus", {})
    if not f.get("ten_sp"):
        return None
    nfc = unicodedata.normalize("NFC", text).lower()
    na = strip_acc(nfc)
    if DEICTIC_ACC.search(nfc) or DEICTIC_NOACC.search(na):
        if cat and cat != "Váy" and cat != f.get("loai_do"):
            return None                                      # "Áo này" nhưng món đang nói là đầm -> không ép
        return f["ten_sp"]
    if field_asked(text) and not cat and not state["slots"].get("ten_sp") \
            and not re.search(r"voucher|ma giam|gio hang|don hang|ma don|\bsp\d", na):
        return f["ten_sp"]
    return None

def recent_history(state, n=3):
    h = state.get("history", [])[-2 * n:]
    return "\n".join(("Khách: " if r == "user" else "SUPEO: ") + str(m)[:160].replace("\n", " ") for r, m in h)

def focus_text(state):
    f, lab = state.get("focus", {}), {"ten_sp": "sản phẩm", "loai_do": "nhóm đồ", "dip_mac": "dịp", "ma_don": "đơn"}
    return "; ".join(f"{lab[k]}: {f[k]}" for k in lab if f.get(k))

def new_state():
    return {"intent": "chitchat", "slots": {}, "history": [], "last_bot": "", "last_reply": "", "last_text": "",
            "focus": {}, "trace": []}      # focus = "bộ nhớ dài": món/đơn/dịp đang nói tới, KHÔNG bị xóa khi đổi chủ đề

KEEP_ALWAYS = {"chieu_cao", "can_nang"}           # số đo cơ thể dùng lại xuyên suốt phiên

def update_state(state, text):
    state["last_text"] = text
    state["slots"].pop("truong", None)
    t = strip_acc(text).lower()
    llm_intent, llm_slots = None, {}
    if USE_LLM:
        d = llm_dst(text, state)
        if d:
            llm_intent, llm_slots = d.get("intent"), d.get("slots") or {}
    new = {}
    ptxt = positive_part(text)                                    # chỉ lấy vế khẳng định để nhận loại đồ / dịp / sản phẩm
    cat = match_key(ptxt, CATS) or (llm_slots.get("loai_do") if llm_slots.get("loai_do") in CATEGORIES else None)
    if not cat and is_generic_vay(ptxt):
        cat = "Váy"                                               # khách nói "váy" chung -> tìm cả đầm và chân váy
    dip = match_key(ptxt, DIPS) or (llm_slots.get("dip_mac") if llm_slots.get("dip_mac") in OCCASIONS else None)
    if cat: new["loai_do"] = cat
    if dip: new["dip_mac"] = dip
    focus = state.setdefault("focus", {})
    prod = pick_item(text, state["slots"].get("_list") or focus.get("_list")) or match_product(ptxt, cat)
    if not prod:
        prod = resolve_reference(text, ptxt, cat, state)          # "áo này", "mẫu đó", "còn hàng không?"
    if prod:
        new["ten_sp"] = prod
        new["loai_do"] = next(p["loai"] for p in CATALOG if p["ten"] == prod)
    kw = None if prod else match_keywords(ptxt)                   # đã chỉ đích danh món thì không thêm từ khóa lấy từ chính tên món
    if kw: new["tu_khoa"] = kw
    new.update(regex_slots(text))                                 # regex ưu tiên hơn LLM cho số và mã
    if "ma_don" not in new and focus.get("ma_don") and re.search(FOLLOW_ORDER, t) \
            and (state["intent"] == "order_check" or re.search(r"\bdon\s+(?:nay|do|kia)\b", t)):
        new["ma_don"] = focus["ma_don"]                           # "thế khi nào giao tới?" -> vẫn là đơn vừa tra
    for k in ("chieu_cao", "can_nang"):
        if k not in new and isinstance(llm_slots.get(k), (int, float)):
            new[k] = int(llm_slots[k])
    if state["intent"] == "size_advice" and state["last_bot"]:    # khách trả lời trơn "52" hoặc "165 52"
        for n in [int(x) for x in re.findall(r"\b\d{2,3}\b", t)]:
            if 130 <= n <= 210 and "chieu_cao" not in new and "chieu_cao" not in state["slots"]:
                new["chieu_cao"] = n
            elif 30 <= n <= 120 and "can_nang" not in new and "can_nang" not in state["slots"]:
                new["can_nang"] = n
    if re.search(r"gio hang|voucher|ma giam|giam gia|uu dai|ap ma|ma nao|khuyen mai", t) or state["intent"] == "voucher" \
            or "ma_voucher" in new:
        cart = parse_money(text)
        if cart and "ma_don" not in new: new["cart_total"] = cart
    truong = field_asked(text)
    # ---- intent ----
    rule = rule_intent(text, new, state["slots"])
    intent = llm_intent if llm_intent in INTENTS else rule
    if intent == "chitchat" and rule != "chitchat":
        intent = rule
    if "ma_don" in new:
        intent = "order_check"
    if intent == "chitchat" and state["last_bot"] and state["intent"] in FOLLOW_KEYS \
            and any(k in new for k in FOLLOW_KEYS[state["intent"]]):
        intent = state["intent"]                                  # đang trả lời câu hỏi lại của bot
    if intent != state["intent"]:                                 # đổi chủ đề -> bỏ slot cũ không còn liên quan
        keep = KEEP_ALWAYS | ({"ten_sp", "loai_do"} if intent == "size_advice" else set())
        state["slots"] = {k: v for k, v in state["slots"].items() if k in keep}
    if "ten_sp" in new or "loai_do" in new:                       # khách đổi sang món/loại khác
        for k in ("ten_sp", "tu_khoa"):
            if k not in new: state["slots"].pop(k, None)
    state["intent"] = intent
    state["slots"].update(new)
    if truong and intent == "product_info": state["slots"]["truong"] = truong
    return state

print("DST sẵn sàng")


DST sẵn sàng


## Bước 6 - Policy (gọi API) + NLG + `bot_reply`

In [7]:
# ============ BƯỚC 3: POLICY & ACTION - thiếu slot thì hỏi lại, đủ slot thì GỌI API ============
ASK = {
    "chieu_cao": "Dạ nàng cho SUPEO xin thêm chiều cao (cm) nhé!",
    "can_nang": "Dạ nàng chia sẻ thêm cân nặng (kg) để shop tư vấn chuẩn xác nha!",
    "loai_do": "Dạ nàng đang quan tâm nhóm đồ nào ạ? SUPEO có: {cats}.",
    "dip_mac": "Dạ nàng định mặc đi dịp nào ạ? SUPEO có sẵn set cho: {dips}.",
    "ma_don": "Dạ nàng nhập giúp SUPEO mã đơn hàng đúng định dạng #SPxxxx (ví dụ #SP2908) nhé!",
}
HELP = ("Dạ SUPEO có thể giúp nàng: tra cứu sản phẩm, tư vấn size, gợi ý set đồ theo dịp, kiểm tra đơn hàng và tìm mã giảm giá. "
        "Nàng thử nhắn như: 'Có áo blazer linen màu be không?', 'Mình cao 1m62 nặng 52kg mặc size gì?', "
        "'Check giúp đơn #SP2908' hoặc 'Giỏ hàng 750k dùng mã nào?' nhé!")
# Chính sách shop KHÔNG nằm trong Excel. Dưới đây là câu TỰ VIẾT, không chép từ đáp án mẫu của dialogue_benchmarks
# (chép sẽ gây rò rỉ khi đo). Hãy THAY bằng thông tin thật của shop.
FAQ_SHIPPING = ("Dạ về hình thức và thời gian giao hàng, SUPEO chưa có thông tin chính xác trong hệ thống nên chưa dám báo nàng ạ. "
                "Nàng nhắn fanpage/hotline của shop để được báo cụ thể nhé! Nếu nàng đã đặt hàng, gửi SUPEO mã đơn #SPxxxx để SUPEO kiểm tra tình trạng giúp nàng.")
FAQ_THANKS = "Dạ cảm ơn nàng đã tin tưởng SUPEO ạ! Nàng cần hỗ trợ thêm gì cứ nhắn SUPEO nhé."

def ask_text(slot):
    return ASK[slot].format(cats=", ".join(CATEGORIES), dips=", ".join(OCCASIONS))

def norm(s): return strip_acc(s).lower().strip()

# ---------- Chức năng 1: product_info ----------
def lc(x):
    x = str(x)
    return x[:1].lower() + x[1:]

def stock_text(p):
    if p["tinh_trang"] == "Sắp hết hàng":
        return f"sắp hết hàng (chỉ còn {p['ton_kho']} sản phẩm)"
    if p["tinh_trang"] == "Hết hàng":
        return "tạm hết hàng"
    return "đang còn hàng"

def product_card(p, truong=None):
    mau, size = ", ".join(p["mau"]), ", ".join(p["size"])
    ten = f"{p['ten']} ({p['id']})"
    opener = {"size": f"Dạ {ten} hiện có size {size} nè nàng.",
              "chat_lieu": f"Dạ {ten} được may bằng {lc(p['chat_lieu'])} ạ.",
              "mau": f"Dạ {ten} có các màu {mau} nàng nhé.",
              "gia": f"Dạ {ten} có giá niêm yết {vnd(p['gia'])} ạ.",
              "ton_kho": f"Dạ {ten} hiện {stock_text(p)} ạ.",
              "anh": f"Dạ SUPEO chưa gửi được ảnh trong khung chat, nàng tham khảo thông tin {ten} bên dưới nhé."}.get(truong, f"Dạ SUPEO gửi nàng thông tin {ten} nè.")
    parts = {"gia": f"giá niêm yết {vnd(p['gia'])}", "chat_lieu": f"chất liệu {lc(p['chat_lieu'])}",
             "phom": f"phom {str(p['phom_dang']).lower()}", "mau": f"màu {mau}", "size": f"size {size}",
             "ton_kho": f"hiện {stock_text(p)}"}
    detail = ", ".join(v for k, v in parts.items() if k != truong)       # không lặp lại điều đã nói ở câu mở đầu
    detail = detail[0].upper() + detail[1:] + "."
    mota = str(p["mo_ta"]).strip().rstrip(".") + "."
    return (f"{opener}\n{detail}\n{mota}\n"
            "Nàng cho SUPEO xin chiều cao, cân nặng để tư vấn size, hoặc nói dịp mặc để SUPEO gợi ý set phối đồ nhé!")

def act_product(S):
    if S.get("ten_sp"):
        _, d = api_get("/v1/products", q=S["ten_sp"])
        items = [x for x in d["items"] if norm(x["ten"]) == norm(S["ten_sp"])] or d["items"]
    elif S.get("loai_do") == "Váy":                                  # "váy" chung = Đầm + Chân váy Midi
        items = []
        for c in ("Đầm", "Chân váy Midi"):
            items += api_get("/v1/products", loai=c, q=S.get("tu_khoa"))[1]["items"]
    else:
        _, d = api_get("/v1/products", loai=S.get("loai_do"), q=S.get("tu_khoa"))
        items = d["items"]
    if not items:
        return "inform", ("Dạ hiện SUPEO chưa có sản phẩm nào khớp với yêu cầu này ạ. "
                          f"Nàng xem thử các nhóm: {', '.join(CATEGORIES)} hoặc đổi màu/chất liệu khác, SUPEO tìm lại ngay nhé!")
    if len(items) == 1:
        p = items[0]
        S["ten_sp"], S["loai_do"] = p["ten"], p["loai"]               # nhớ món vừa gửi để hiểu "áo này", "còn hàng không?"
        card = product_card(p, S.get("truong"))
        h, w = S.get("chieu_cao"), S.get("can_nang")
        if h and w and 140 <= h <= 200 and 30 <= w <= 120:            # đã biết số đo của khách -> dùng luôn, không hỏi lại
            _, r = api_get("/v1/size-advice", chieu_cao=h, can_nang=w)
            sz = r["de_xuat"]["size"]
            tail = (f"mẫu này có size {sz} nè nàng!" if sz in p["size"]
                    else f"nhưng mẫu này chỉ có size {', '.join(p['size'])}, nàng cân nhắc giúp SUPEO nhé.")
            card = card.rsplit("\n", 1)[0] + f"\nTheo số đo nàng đã chia sẻ ({h}cm, {w}kg), nàng hợp Size {sz}, " + tail
        return "inform", card
    label = ""
    if S.get("loai_do"):
        label = " váy (đầm và chân váy)" if S["loai_do"] == "Váy" else f" {S['loai_do']}"
    if S.get("tu_khoa"):
        label += f" ({S['tu_khoa']})"
    shown = items[:10]
    S["_list"] = [p["ten"] for p in shown]                            # nhớ danh sách để khách chọn 'mẫu thứ 2'
    head = (f"Dạ SUPEO chưa gửi được ảnh trong khung chat, nàng tham khảo các mẫu{label} dưới đây nhé:" if S.get("truong") == "anh"
            else f"Dạ SUPEO đang có {len(items)} mẫu{label} nàng tham khảo nè:")
    lines = [f"{i}. {p['ten']} ({p['id']}) - {vnd(p['gia'])}; màu {', '.join(p['mau'])}; {p['tinh_trang'].lower()}"
             for i, p in enumerate(shown, 1)]
    more = f"\n(còn {len(items) - 10} mẫu khác nữa ạ)" if len(items) > 10 else ""
    return "inform", (head + "\n" + "\n".join(lines) + more +
                      "\nNàng thích mẫu nào cứ nhắn tên hoặc số thứ tự (ví dụ 'mẫu thứ 2'), SUPEO gửi chi tiết ngay nhé! "
                      "Nàng cũng có thể cho SUPEO biết chiều cao, cân nặng để chọn đúng size.")

# ---------- Chức năng 2: size_advice (Max-Boundary nằm ở API) ----------
def act_size(S):
    h, w = S["chieu_cao"], S["can_nang"]
    if not (140 <= h <= 200 and 30 <= w <= 120):
        S.pop("chieu_cao", None); S.pop("can_nang", None)
        return "ask", "Dạ chiều cao và cân nặng nàng nhập chưa hợp lý, nàng gửi lại giúp SUPEO (ví dụ: 1m65, 52kg) nhé!"
    _, r = api_get("/v1/size-advice", chieu_cao=h, can_nang=w)
    de = r["de_xuat"]; kg = r["theo_can_nang"]["khoang"]; cm = r["theo_chieu_cao"]["khoang"]
    if r["lech"]:
        txt = (f"Dạ với chiều cao {h}cm nàng đang ở size {r['theo_chieu_cao']['size']} ({cm[0]}-{cm[1]}cm), "
               f"còn cân nặng {w}kg lại thuộc size {r['theo_can_nang']['size']} ({kg[0]}-{kg[1]}kg). "
               f"Khi hai số đo lệch nhau, SUPEO ưu tiên size lớn hơn để nàng mặc thoải mái, nên SUPEO khuyên nàng chọn Size {de['size']} nha!")
    else:
        txt = (f"Dạ với chiều cao {h}cm và cân nặng {w}kg, nàng mặc chuẩn Size {de['size']} "
               f"({de['chieu_cao'][0]}-{de['chieu_cao'][1]}cm, {de['can_nang'][0]}-{de['can_nang'][1]}kg) là vừa vặn nhất ạ!")
    txt += f"\n({de['loi_khuyen']})"
    if r["ngoai_dai"]["chieu_cao"] or r["ngoai_dai"]["can_nang"]:
        txt += "\nSố đo của nàng nằm ngoài dải size chuẩn, nàng nhắn SUPEO để được tư vấn kỹ hơn nhé."
    if S.get("ten_sp"):                                           # kiểm tra mẫu đó có size này không
        _, d = api_get("/v1/products", q=S["ten_sp"])
        p = next((x for x in d["items"] if norm(x["ten"]) == norm(S["ten_sp"])), None)
        if p and de["size"] not in p["size"]:
            txt += f"\nLưu ý: {p['ten']} ({p['id']}) hiện chỉ có size {', '.join(p['size'])}, nàng cân nhắc giúp SUPEO nhé."
        elif p:
            txt += f"\n{p['ten']} ({p['id']}) có size {de['size']} và {stock_text(p)} nè nàng."
    elif S.get("loai_do") and S["loai_do"] != "Váy":              # gợi ý mẫu còn size phù hợp trong nhóm khách đang quan tâm
        _, d = api_get("/v1/products", loai=S["loai_do"])
        ok = [x for x in d["items"] if de["size"] in x["size"] and x["ton_kho"] > 0][:3]
        if ok:
            txt += (f"\nCác mẫu {S['loai_do']} còn size {de['size']} nàng có thể thử: "
                    + "; ".join(f"{x['ten']} ({x['id']}) {vnd(x['gia'])}" for x in ok) + ".")
    else:
        txt += "\nNàng nói tên mẫu hoặc nhóm đồ đang quan tâm, SUPEO kiểm tra giúp mẫu đó còn size này không nhé!"
    return "inform", txt

# ---------- Chức năng 3: lookbook ----------
def act_lookbook(S):
    st, d = api_get(f"/v1/lookbook/{quote(S['dip_mac'], safe='')}")
    if st == 404:
        S.pop("dip_mac", None)
        return "ask", ask_text("dip_mac")
    lines = []
    for p in d["items"]:
        note = "" if p["tinh_trang"] == "Còn hàng" else f" ({p['tinh_trang'].lower()})"
        lines.append(f"- {p['ten']} ({p['id']}): {vnd(p['gia'])}{note}")
    return "inform", (f"Dạ cho dịp {d['dip_mac']}, SUPEO gợi ý nàng set '{d['ten_bo']}' phong cách {d['phong_cach']} nè:\n"
                      + "\n".join(lines) + f"\nTổng cả set: {vnd(d['tong_gia_tien'])}.\n{d['ghi_chu']}\n"
                      "Nàng cho SUPEO xin chiều cao, cân nặng để chọn size từng món, hoặc nhắn tên món nàng thích để SUPEO gửi chi tiết nhé!")

# ---------- Chức năng 4: order_check ----------
def act_order(S):
    st, d = api_get(f"/v1/orders/{quote(S['ma_don'], safe='')}/tracking")
    if st == 422:
        S.pop("ma_don", None)
        return "ask", ask_text("ma_don")
    if st == 404:
        code = S["ma_don"]
        return "inform", (f"Dạ SUPEO đã kiểm tra nhưng chưa tìm thấy mã đơn {code} trên hệ thống. Nàng vui lòng kiểm tra lại "
                          "tin nhắn xác nhận đơn hoặc gửi số điện thoại đặt hàng để shop hỗ trợ tra cứu nha!")
    sp = "; ".join(d["san_pham"])
    tt = d["trang_thai"]
    txt = f"Dạ SUPEO đã kiểm tra đơn hàng {d['ma_don']} ({sp}) của nàng rồi nè. Đơn đặt ngày {d['ngay_dat']} và hiện đang ở trạng thái '{tt}'."
    if tt == "Đã hủy":
        txt = f"Dạ đơn hàng {d['ma_don']} ({sp}) đặt ngày {d['ngay_dat']} đã ở trạng thái 'Đã hủy'. Nếu cần đặt lại, nàng nhắn SUPEO hỗ trợ tạo đơn mới nha!"
    elif d["da_ban_giao_vc"]:
        txt += f" Đơn qua {d['dvvc']} (mã vận đơn: {d['ma_van_don']})."
    elif d["dvvc"] and d["dvvc"] != "Chưa bàn giao":
        txt += f" Đơn sẽ giao qua {d['dvvc']}, mã vận đơn sẽ có khi bàn giao."
    else:
        txt += " Đơn chưa bàn giao cho đơn vị vận chuyển."
    if tt == "Đã giao thành công":
        txt += " Chúc nàng mặc đồ SUPEO thật xinh nhé!"
    elif tt.startswith("Giao thất bại"):
        txt += f" Đơn đang được hoàn về shop, nàng nhắn SUPEO để được hỗ trợ giao lại nhé (dự kiến cũ: {d['du_kien']})."
    elif d["du_kien"]:
        txt += f" Dự kiến giao/bàn giao đến nàng ngày {d['du_kien']}."
    return "inform", txt + f" Tổng giá trị đơn: {vnd(d['tong_tien'])}. Nàng cần SUPEO hỗ trợ thêm gì cho đơn này không ạ?"

# ---------- Chức năng 5: voucher ----------
def voucher_line(v, cart=None):
    s = f"- {v['ma']}: {v['mo_ta']}"
    if v["doi_tuong"] and v["doi_tuong"] != "Mọi khách hàng":
        s += f" (Đối tượng: {v['doi_tuong']})"
    s += f" - HSD {v['han_dung']}"
    if v["con_lai_ngay"] is not None and v["con_lai_ngay"] <= 7:
        s += f" (sắp hết hạn, còn {v['con_lai_ngay']} ngày)"
    if cart is not None and v.get("uoc_tinh_giam"):
        s += f" => ước tính giảm {vnd(v['uoc_tinh_giam'])}"
    return s

def act_voucher(S):
    cart = S.get("cart_total")
    if S.get("ma_voucher"):
        _, d = api_get(f"/v1/vouchers/{quote(S['ma_voucher'], safe='')}", cart_total=cart)
        tt = d["trang_thai"]
        if tt == "khong_kha_dung":
            return "inform", (f"Dạ SUPEO chưa thấy mã {d['ma']} trong danh sách ưu đãi đang mở. Nàng kiểm tra lại mã, "
                              "hoặc hỏi SUPEO 'có mã giảm giá nào' để xem các mã đang áp dụng nhé!")
        if tt == "het_han":
            return "inform", f"Dạ mã {d['ma']} đã hết hạn từ {d['han_dung']} ạ. Nàng hỏi SUPEO 'có mã giảm giá nào' để xem mã còn hạn nhé!"
        if tt == "chua_du_dieu_kien":
            return "inform", (f"Dạ mã {d['ma']} ({d['mo_ta']}) cần đơn từ {vnd(d['don_toi_thieu'])}. Giỏ hàng nàng đang {vnd(cart)}, "
                              f"nàng thêm {vnd(d['con_thieu'])} nữa là dùng được ạ!")
        txt = f"Dạ mã {d['ma']} dùng được nha: {d['mo_ta']}. HSD {d['han_dung']}."
        if cart is not None: txt += f" Với giỏ {vnd(cart)}, ước tính giảm {vnd(d['uoc_tinh_giam'])}."
        return "inform", txt + f" Lưu ý: {d['dieu_khoan']}"
    _, d = api_get("/v1/vouchers/available", cart_total=cart)
    if not d["vouchers"]:
        return "inform", "Dạ hiện SUPEO chưa có mã ưu đãi nào đang mở, nàng theo dõi shop để cập nhật sớm nhất nhé!"
    if cart is None:
        return "inform", (f"Dạ hiện SUPEO đang có {d['count']} mã ưu đãi:\n" + "\n".join(voucher_line(v) for v in d["vouchers"])
                          + "\nNàng cho SUPEO biết giá trị giỏ hàng để chọn mã tốt nhất nhé!")
    ok = [v for v in d["vouchers"] if v["du_dieu_kien"]]
    if not ok:
        up = d["goi_y_upsell"]
        return "inform", (f"Dạ với giỏ hàng {vnd(cart)} nàng chưa đủ điều kiện mã nào. " +
                          (f"Nàng thêm {vnd(up['con_thieu'])} nữa để dùng {up['ma']} ({up['mo_ta']}) nhé!" if up else ""))
    best = next(v for v in ok if v["ma"] == d["tot_nhat"])
    txt = (f"Dạ với giỏ hàng {vnd(cart)}, nàng áp {best['ma']} là lợi nhất, giảm khoảng {vnd(best['uoc_tinh_giam'])}.\n"
           "Các mã nàng đủ điều kiện:\n" + "\n".join(voucher_line(v, cart) for v in ok))
    up = d["goi_y_upsell"]
    if up:
        txt += (f"\nGợi ý: nàng chọn thêm khoảng {vnd(up['con_thieu'])} để đơn đạt {vnd(up['don_toi_thieu'])} "
                f"và dùng thêm {up['ma']} ({up['mo_ta']}).")
    return "inform", txt

# ---------- Chitchat / FAQ ----------
def act_chitchat(text):
    t = strip_acc(text).lower()
    if re.search(r"cam on|thanks|thank", t): return "chitchat", FAQ_THANKS
    if re.search(r"tam biet|bye|hen gap lai|thoi nhe", t):
        return "chitchat", "Dạ cảm ơn nàng đã ghé SUPEO, chúc nàng một ngày thật xinh và hẹn gặp lại nàng nhé!"
    if re.search(r"hoa toc|giao nhanh|phi ship|phi van chuyen|giao hang|bao lau|thoi gian giao|giao toan quoc", t): return "chitchat", FAQ_SHIPPING
    if re.search(r"doi tra|bao hanh|thanh toan|\bcod\b|hotline|so dien thoai|dia chi|cua hang o dau|ban ship cod", t):
        return "chitchat", ("Dạ phần này SUPEO chưa có thông tin chính xác trong hệ thống nên không dám trả lời sai với nàng. "
                            "Nàng nhắn trực tiếp cho shop (fanpage/hotline) để được hỗ trợ nhanh nhất nhé! "
                            "Còn về sản phẩm, size, set đồ, đơn hàng hay mã giảm giá thì nàng cứ hỏi SUPEO nha.")
    if re.search(r"\b(?:xin chao|chao|hello|hi|alo|shop oi|ban oi)\b", t):
        return "chitchat", "Dạ SUPEO chào nàng ạ! " + HELP[3:]
    return "chitchat", HELP

def policy(state):
    intent, S = state["intent"], state["slots"]
    need = missing(intent, S)
    if intent == "size_advice" and len(need) == 2:
        return "ask", "Dạ nàng cho SUPEO xin chiều cao (cm) và cân nặng (kg) để tư vấn size chuẩn nhất nhé!"
    if need == ["loai_do"] and S.get("truong") == "anh":
        return "ask", "Dạ SUPEO chưa gửi được ảnh trong khung chat ạ. Nàng nói tên mẫu hoặc nhóm đồ (" + ", ".join(CATEGORIES) + ") để SUPEO gửi thông tin chi tiết nhé!"
    if need:
        return "ask", ask_text(need[0])
    if intent == "product_info": return act_product(S)
    if intent == "size_advice": return act_size(S)
    if intent == "lookbook": return act_lookbook(S)
    if intent == "order_check": return act_order(S)
    if intent == "voucher": return act_voucher(S)
    return act_chitchat(state["last_text"])

# ============ NLG: diễn đạt tự nhiên, nhưng KHÔNG được đổi số liệu ============
NLG_SYS = ("Bạn là tư vấn viên thời trang SUPEO, xưng 'SUPEO', gọi khách là 'nàng', lịch sự, thân thiện. "
           "Có thể xem 'Hội thoại gần đây' để câu văn liền mạch, nhưng TUYỆT ĐỐI không lấy số liệu từ đó. "
           "Viết lại thông tin sau cho tự nhiên, ngắn gọn. GIỮ NGUYÊN mọi con số, giá tiền, mã, ngày tháng, tên sản phẩm. "
           "TUYỆT ĐỐI không thêm thông tin không có trong dữ liệu. Giữ nguyên mã sản phẩm dạng (P01) và số thứ tự trong danh sách. Giữ định dạng gạch đầu dòng nếu có.")

def facts_ok(facts, out):
    toks = re.findall(r"[#\w.]*\d[\w.]*", facts)
    return all(tk.rstrip(".") in out for tk in toks)

def nlg(facts, user_text, state=None):
    if not USE_LLM:
        return facts
    try:
        out = llm([{"role": "system", "content": NLG_SYS},
                   {"role": "user", "content": (f"Hội thoại gần đây:\n{recent_history(state)}\n\n" if state and state.get("history") else "")
                    + f"Câu khách: {user_text}\nDữ liệu thật từ hệ thống:\n{facts}"}], 350).strip()
        return out if out and facts_ok(facts, out) else facts      # chốt chặn chống bịa số liệu
    except Exception:
        return facts

FOCUS_KEYS = ("ten_sp", "loai_do", "ma_don", "dip_mac", "_list", "cart_total")

def update_focus(state):
    """Ghi nhớ dài hạn: món / đơn / dịp / danh sách đang nói tới. Khác với slots, focus không bị xóa khi đổi chủ đề."""
    S, f = state["slots"], state.setdefault("focus", {})
    if state["intent"] == "product_info" and "ten_sp" not in S:
        f.pop("ten_sp", None)                                    # đang xem danh sách nhiều mẫu -> chưa có món cụ thể
    for k in FOCUS_KEYS:
        if S.get(k) is not None:
            f[k] = S[k]

def bot_reply(text, state):
    API_LOG.clear()
    try:
        update_state(state, text)
        kind, out = policy(state)
    except ApiDown:
        kind, out = "error", "Dạ hệ thống tra cứu của SUPEO đang bận, nàng thử lại sau ít phút nhé!"
    reply = nlg(out, text, state) if kind == "inform" else out
    state["history"] = (state["history"] + [("user", text), ("bot", reply)])[-40:]     # giữ 20 lượt gần nhất
    state["last_bot"] = reply if kind == "ask" else ""
    state["last_reply"] = reply
    update_focus(state)
    state["trace"].append({"user": text, "intent": state["intent"], "slots": dict(state["slots"]),
                           "action": kind, "api_calls": list(API_LOG)})
    return reply

print("Policy & NLG sẵn sàng")


Policy & NLG sẵn sàng


## Bước 6b - Bản vá v5 (nhận món idf / màu / size / voucher / nhớ đơn / che tên món)
Chỉ **định nghĩa lại hàm** (không thay chuỗi). Chạy ngay sau cell Policy+NLG.

In [ ]:
# ============ BẢN VÁ v5: nhận món / màu / size / voucher / nhớ đơn / che tên món ============
# Đặt cell này NGAY SAU cell "Policy + NLG" (cell 13) và TRƯỚC cell bật LLM. Cell chỉ ĐỊNH NGHĨA LẠI hàm
# (không dùng thay chuỗi nên không thể "không khớp"). Không có luật riêng cho câu test nào.
import re, math, unicodedata

def _nfc(s):
    return unicodedata.normalize("NFC", str(s)).lower()

def _has_acc(s):
    return strip_acc(s) != s

# ---------------------------------------------------------------- 1) Chỉ mục nhận món (idf) ----
GENERIC_NAME_TOK = {"mua", "he", "thu", "nhe", "cap", "dep", "dang"}        # từ trong tên món nhưng không phân biệt
COMMON_VI = {"tien", "bang", "no", "nhieu", "gia", "vua", "xinh", "hop", "mac", "lam", "ban", "cho", "hoi", "ve", "di",
             "mua", "do", "la", "nay", "kia", "nao", "gi", "khong", "co", "con", "het"}   # từ hay gặp khi gõ KHÔNG dấu

NAME_TOK, NAME_TOK_ACC, NON_ID = {}, {}, set()

def build_match_index():
    """Gọi lại sau load_lexicons(). NON_ID = từ chung/loại/màu/dịp/chất liệu: không dùng để nhận diện món."""
    global NAME_TOK, NAME_TOK_ACC, NON_ID
    toks = lambda s: set(re.findall(r"[a-z0-9]+", strip_acc(s).lower()))
    color_tok = set().union(*[toks(k) for k in COLOR_LEX]) if COLOR_LEX else set()
    dip_tok = set().union(*[toks(p) for pats in DIPS.values() for p in pats])
    mat_tok = set().union(*[toks(k) for k in MATERIAL_LEX])
    NON_ID = set(CAT_WORDS) | set(STOP) | color_tok | dip_tok | mat_tok | GENERIC_NAME_TOK
    NAME_TOK = {p["id"]: {t for t in toks(p["ten"]) - NON_ID if len(t) > 1} for p in CATALOG}
    NAME_TOK_ACC = {p["id"]: {t for t in re.findall(r"\w+", _nfc(p["ten"]))
                              if len(t) > 1 and strip_acc(t) not in NON_ID} for p in CATALOG}

def idf_pick(words, group, tokmap, min_score=1.0, margin=1.0, banned=frozenset()):
    n = len(group)
    if n < 2:
        return None
    df = {}
    for p in group:
        for t in tokmap[p["id"]]:
            df[t] = df.get(t, 0) + 1
    scored = []
    for p in group:
        hits = {t for t in tokmap[p["id"]] & words if t not in banned}
        if not hits:
            continue
        if len(hits) == 1 and len(next(iter(hits))) < 3:                  
            continue
        scored.append((sum(math.log(n / df[t]) for t in hits), p["ten"]))
    scored.sort(reverse=True)
    if not scored or scored[0][0] < min_score:
        return None
    if len(scored) > 1 and scored[0][0] - scored[1][0] < margin:
        return None
    return scored[0][1]

def mask_product_names(text):
    """Che TÊN MÓN ĐẦY ĐỦ khỏi câu (trả về chuỗi không dấu) để chữ trong tên ('công sở', 'dự tiệc')
    không bị hiểu là dịp mặc / loại đồ khi khách chỉ nhắc tên món."""
    t = strip_acc(text).lower()
    for p in sorted(CATALOG, key=lambda x: -len(x["ten"])):
        t = t.replace(strip_acc(p["ten"]).lower(), " sanpham ")
    return t

def match_product(text, cat):
    """Tên đầy đủ -> chọn; không có thì chọn theo idf trong cùng nhóm (nếu biết loại) hoặc toàn danh mục."""
    t = strip_acc(text).lower()
    best = None
    for p in CATALOG:
        if strip_acc(p["ten"]).lower() in t and (best is None or len(p["ten"]) > len(best["ten"])):
            best = p
    if best:
        return best["ten"]
    if cat == "Váy":
        group = [p for p in CATALOG if p["loai"] in ("Đầm", "Chân váy Midi")]
    elif cat in CATEGORIES:
        group = [p for p in CATALOG if p["loai"] == cat]
    else:
        group = CATALOG
    nfc = _nfc(text)
    if _has_acc(nfc):                                   # gõ có dấu -> so sánh CÓ DẤU ('tiên' khác 'tiền', 'nơ' khác 'nó')
        return idf_pick(set(re.findall(r"\w+", nfc)), group, NAME_TOK_ACC)
    return idf_pick(set(re.findall(r"[a-z0-9]+", t)), group, NAME_TOK, banned=COMMON_VI)

# ---------------------------------------------------------------- 2) Nhận màu không cần chữ "màu" ----
SHORT_AMBIG = {"be", "do"}                                                  # 'be' dễ nhầm -> chỉ nhận khi đứng ngay sau tên loại đồ
NOUN_BEFORE = r"(?:blazer|vest|ao|dam|vay|quan|so mi|midi|culottes)"

def match_color(text):
    nfc = _nfc(text)
    t = strip_acc(nfc)
    acc = t != nfc
    for k, vi in COLOR_LEX.items():                                         # COLOR_LEX đã sắp theo độ dài giảm dần
        if re.search(r"\bmau\s+" + re.escape(k) + r"\b", t):                # có chữ "màu"
            return vi
    for k, vi in COLOR_LEX.items():
        if acc and k not in SHORT_AMBIG:
            if re.search(r"(?<!\w)" + re.escape(vi) + r"(?!\w)", nfc):      # có dấu: 'đen' khác 'đến'
                return vi
        elif not acc and (len(k) >= 6 or " " in k) and k not in SHORT_AMBIG:
            if re.search(r"\b" + re.escape(k) + r"\b", t):                  # không dấu: chỉ nhận màu dài ('champagne')
                return vi
        if re.search(r"\b" + NOUN_BEFORE + r"\s+" + re.escape(k) + r"\b", t):   # 'đầm be', 'blazer kem'
            return vi
    return None

def match_keywords(text):
    t = strip_acc(text).lower()
    kws = [vi for k, vi in MATERIAL_LEX.items() if re.search(r"\b" + k + r"\b", t)]
    c = match_color(text)
    if c:
        kws.append(c)
    return " ".join(kws) or None

# ---------------------------------------------------------------- 3) Dịp mặc (không nhận từ trong tên món) ----
def detect_dip(ptxt, prod_from_text=None):
    t = mask_product_names(ptxt)
    if prod_from_text:                                  # khách gõ một phần tên ('suông công sở'): bỏ cụm dịp nằm trong tên món
        pn = strip_acc(prod_from_text).lower()
        for pats in DIPS.values():
            for p in pats:
                if re.search(r"\b" + p + r"\b", pn) and len(re.findall(r"\b" + p + r"\b", t)) == 1:
                    t = re.sub(r"\b" + p + r"\b", " ", t)
    return match_key(t, DIPS)

# ---------------------------------------------------------------- 4) Size: hỏi hàng vs xin tư vấn ----
SIZE_CODE = r"(?:xxl|xl|xs|s|m|l)"
PERSONAL_SIZE = (r"\b(?:minh|em|toi|chi|con)\s+(?:mac|lay|chon|dung)\b|\bnen\s+(?:mac|lay|chon)\b|\bmac size\b"
                 r"|\bsize cua (?:minh|em|toi)\b|\bhop size\b|\bsize nao\s+(?:hop|vua|phu hop|dung|nen)\b"
                 r"|\b(?:hop|vua|phu hop)\s+voi\s+(?:minh|em|toi|chi)\b|\bsize\s+(?:gi|nao)\s+(?:cho|voi)\s+(?:minh|em|toi|chi)\b")

def asked_size_code(text):
    """'còn size M không' -> 'M' (hỏi hàng). 'mặc size M được không' / 'size M có vừa không' là xin tư vấn -> None."""
    t = strip_acc(text).lower()
    m = re.search(r"\bsize\s+(" + SIZE_CODE + r")\b", t)
    if not m or re.search(PERSONAL_SIZE, t) or re.search(r"\b(?:vua|hop|mac|duoc|thoai mai|nen)\b", t):
        return None
    return m.group(1).upper() if re.search(r"\b(?:con|het|co|ban|du)\b", t) else None

def field_asked(text):
    t = strip_acc(text).lower()
    if asked_size_code(text):
        return "size"
    if re.search(r"\bsize\b", t) and re.search(r"(nhung|may|co) size|size nao|size gi co|du size", t):
        return "size"
    if re.search(r"chat lieu|chat vai|vai gi|vai nao|lam bang", t):
        return "chat_lieu"
    nfc = _nfc(text)
    if re.search(r"\bmàu\b", nfc) if strip_acc(nfc) != nfc else re.search(r"\bmau\b", t):
        return "mau"
    if re.search(r"con hang|het hang|ton kho|con khong|con mau", t):
        return "ton_kho"
    if re.search(r"\bgia\b|bao nhieu|bao nhieu tien|nhieu tien", t):
        return "gia"
    if re.search(r"hinh anh|xem hinh|\b(?:xem|gui|coi|thay)\s+(?:duoc\s+)?(?:cai\s+|nhung\s+)?anh\b|\banh\s+(?:san pham|mau|that|thuc te)\b", t):
        return "anh"
    return None

# ---------------------------------------------------------------- 5) Voucher + intent ----
VOUCHER_RE = (r"\b(?:voucher|ma giam|giam gia|khuyen mai|uu dai|gio hang|ap ma|ma nao|ma gi|coupon|freeship|"
              r"giam bao nhieu|duoc giam|giam duoc|dung ma|nhap ma)\b")

def rule_intent(text, new, state_slots):
    t = strip_acc(text).lower()
    if "ma_don" in new or re.search(r"don hang|van don|check don|kiem tra don|tra cuu don|theo doi don|giao toi dau|ship toi dau|don cua minh", t):
        return "order_check"
    if "ma_voucher" in new or "cart_total" in new or re.search(VOUCHER_RE, t):
        return "voucher"
    asks_size_list = (field_asked(text) == "size" and "chieu_cao" not in new and "can_nang" not in new
                      and not re.search(PERSONAL_SIZE, t))
    if not asks_size_list and ("chieu_cao" in new or "can_nang" in new or re.search(r"\bsize\b|mac vua|co nguoi|nang bao nhieu|cao bao nhieu", t)):
        return "size_advice"
    if "dip_mac" in new or re.search(r"phoi|lookbook|goi y (?:bo|set|do|outfit)|set do|outfit", t):
        return "lookbook"
    if any(k in new for k in ("loai_do", "ten_sp", "tu_khoa")) or field_asked(text) or re.search(r"mau (?:nao|gi) |san pham|mat hang", t):
        return "product_info"
    return "chitchat"

# ---------------------------------------------------------------- 6) Hỏi tiếp về đơn / tham chiếu ----
FOLLOW_ORDER = (r"\b(?:giao|khi nao|bao gio|bao lau|toi noi|den noi|toi chua|den chua|toi dau|den dau|dang o dau|nhan duoc|"
                r"van chuyen|van don|trang thai|tinh trang|du kien|hoan|huy|ship|don nay|don do|don kia|don day)\b")
DEICTIC_ACC = re.compile(r"\b(?:áo|đầm|váy|quần|blazer|mẫu|sản phẩm|món|cái|set|bộ)\s+(?:này|đó|đấy|kia|ấy|vừa rồi|trên)\b|\bnó\b")
DEICTIC_NOACC = re.compile(r"\b(?:ao|dam|vay|quan|blazer|san pham|mon|cai)\s+(?:nay|kia|ay|do|day)\b|\bmau\s+(?:nay|kia|do|day)\b|\bcai do\b")


# ---------------------------------------------------------------- 6b) resolve_reference: so loại với CHÍNH món đang nhớ ----
def resolve_reference(text, ptxt, cat, state):
    """'áo này', 'nó', 'mẫu đó', 'còn hàng không?' -> món đang nói tới. Loại đồ của món lấy từ danh mục (không từ focus['loai_do'],
    vì giá trị đó có thể bị LLM sai ghi đè)."""
    f = state.get("focus", {})
    if not f.get("ten_sp"):
        return None
    p = next((x for x in CATALOG if x["ten"] == f["ten_sp"]), None)
    nfc = _nfc(text)
    na = strip_acc(nfc)
    if DEICTIC_ACC.search(nfc) or DEICTIC_NOACC.search(na):
        if cat and cat != "Váy" and p and cat != p["loai"]:
            return None                                      # "Áo này" nhưng món đang nói là đầm -> không ép
        return f["ten_sp"]
    if field_asked(text) and not cat and not state["slots"].get("ten_sp") \
            and not re.search(r"voucher|ma giam|gio hang|don hang|ma don|\bsp\d", na):
        return f["ten_sp"]
    return None


# ---------------------------------------------------------------- 6c) "70 cân" = 70 kg (cách nói thường ngày) ----
try:
    _regex_slots_v4
except NameError:
    _regex_slots_v4 = regex_slots            # chỉ bọc 1 lần dù chạy lại cell

def regex_slots(text):
    out = _regex_slots_v4(text)
    if "can_nang" not in out:
        m = re.search(r"\b(\d{2,3})\s?can\b", strip_acc(text).lower())    
        if m and 30 <= int(m.group(1)) <= 120:
            out["can_nang"] = int(m.group(1))
    return out

# ---------------------------------------------------------------- 7) update_state (bản đầy đủ, thay bản cũ) ----
def update_state(state, text):
    state["last_text"] = text
    state["slots"].pop("truong", None)
    state["slots"].pop("_size", None)
    t = strip_acc(text).lower()
    llm_intent, llm_slots = None, {}
    if USE_LLM:
        d = llm_dst(text, state)
        if d:
            llm_intent, llm_slots = d.get("intent"), d.get("slots") or {}
    new = {}
    ptxt = positive_part(text)
    cat_rule = match_key(ptxt, CATS) or ("Váy" if is_generic_vay(ptxt) else None)      # loại đồ do LUẬT nhận từ chính câu khách
    cat = cat_rule or (llm_slots.get("loai_do") if llm_slots.get("loai_do") in CATEGORIES else None)
    focus = state.setdefault("focus", {})
    if cat and not cat_rule and field_asked(text) and (state["slots"].get("ten_sp") or focus.get("ten_sp")):
        cat = None                                                # câu hỏi tiếp về món đang nói: không nghe loại đồ do LLM bịa ra
    prod = pick_item(text, state["slots"].get("_list") or focus.get("_list"))
    prod_text = prod or match_product(ptxt, cat)                  # món nhận từ chính câu khách
    prod = prod_text or resolve_reference(text, ptxt, cat_rule, state)   # "áo này", "nó", "mẫu đó", "còn hàng không?" (LLM sai loại đồ cũng không làm mất ngữ cảnh)
    if prod and not cat_rule:
        cat = None                                                # loại đồ lấy từ món đã xác định, không lấy từ LLM
    dip = detect_dip(ptxt, prod_text) or (llm_slots.get("dip_mac") if llm_slots.get("dip_mac") in OCCASIONS else None)
    if cat: new["loai_do"] = cat
    if dip: new["dip_mac"] = dip
    if prod:
        new["ten_sp"] = prod
        new["loai_do"] = next(p["loai"] for p in CATALOG if p["ten"] == prod)
    kw = None if prod else match_keywords(ptxt)
    if kw: new["tu_khoa"] = kw
    new.update(regex_slots(text))
    if "ma_don" not in new and focus.get("ma_don") and re.search(FOLLOW_ORDER, t) \
            and (state["intent"] == "order_check" or re.search(r"\bdon\s+(?:nay|do|kia|day)\b", t)):
        new["ma_don"] = focus["ma_don"]                           # "bao lâu nữa tới?", "đơn đó giao chưa?" -> vẫn là đơn vừa tra
    for k in ("chieu_cao", "can_nang"):
        if k not in new and isinstance(llm_slots.get(k), (int, float)):
            new[k] = int(llm_slots[k])
    if state["intent"] == "size_advice" and state["last_bot"]:
        for n in [int(x) for x in re.findall(r"\b\d{2,3}\b", t)]:
            if 130 <= n <= 210 and "chieu_cao" not in new and "chieu_cao" not in state["slots"]:
                new["chieu_cao"] = n
            elif 30 <= n <= 120 and "can_nang" not in new and "can_nang" not in state["slots"]:
                new["can_nang"] = n
    if re.search(VOUCHER_RE, t) or state["intent"] == "voucher" or "ma_voucher" in new:
        cart = parse_money(text)
        if cart and "ma_don" not in new: new["cart_total"] = cart
    truong = field_asked(text)
    sz = asked_size_code(text)
    if sz: new["_size"] = sz
    rule = rule_intent(text, new, state["slots"])
    intent = llm_intent if llm_intent in INTENTS else rule
    if intent == "chitchat" and rule != "chitchat":
        intent = rule
    if "ma_don" in new:
        intent = "order_check"
    if intent == "chitchat" and state["last_bot"] and state["intent"] in FOLLOW_KEYS \
            and any(k in new for k in FOLLOW_KEYS[state["intent"]]):
        intent = state["intent"]
    if intent != state["intent"]:
        keep = KEEP_ALWAYS | ({"ten_sp", "loai_do"} if intent == "size_advice" else set())
        state["slots"] = {k: v for k, v in state["slots"].items() if k in keep}
    if "ten_sp" in new or "loai_do" in new:
        for k in ("ten_sp", "tu_khoa"):
            if k not in new: state["slots"].pop(k, None)
    state["intent"] = intent
    state["slots"].update(new)
    if truong and intent == "product_info": state["slots"]["truong"] = truong
    return state

# ---------------------------------------------------------------- 8) Hỏi "còn size M không" -> trả lời đúng size đó ----
try:
    _act_product_v4
except NameError:
    _act_product_v4 = act_product                                   # chỉ bọc 1 lần dù chạy lại cell

def act_product(S):
    kind, out = _act_product_v4(S)
    sz = S.get("_size")
    if sz and kind == "inform" and S.get("ten_sp"):
        p = next((x for x in CATALOG if x["ten"] == S["ten_sp"]), None)
        if p and out.startswith(f"Dạ {p['ten']}") and "\n" in out:
            head = (f"Dạ {p['ten']} ({p['id']}) hiện có size {sz} nè nàng." if sz in p["size"]
                    else f"Dạ {p['ten']} ({p['id']}) hiện không có size {sz} ạ, mẫu này có size {', '.join(p['size'])}.")
            out = head + "\n" + out.split("\n", 1)[1]
    return kind, out

build_match_index()
print("Bản vá v5 sẵn sàng:", len(NAME_TOK), "món được lập chỉ mục")


Bản vá v5 sẵn sàng: 30 món được lập chỉ mục


## Bước 6c - Bản vá v6 (không hỏi lại số đo / phối đồ theo món đang xem / giảm độ trễ Gemini)
Chạy ngay sau bản vá v5, trước cell bật LLM.

In [ ]:
# ============ BẢN VÁ v6: không hỏi lại số đo | phối đồ theo món đang xem | giảm độ trễ khi dùng Gemini ============
# Đặt NGAY SAU cell "Bản vá v5". Cell chỉ định nghĩa lại / bọc hàm, chạy lại nhiều lần vẫn an toàn.
import os, re, copy, time

# ---------------------------------------------------------------- 1) Số đo đã biết: dùng lại, không hỏi lại ----
_SIZE_CACHE = {}

def known_measures(S):
    h, w = S.get("chieu_cao"), S.get("can_nang")
    if isinstance(h, (int, float)) and isinstance(w, (int, float)) and 140 <= h <= 200 and 30 <= w <= 120:
        return int(h), int(w)
    return None

def my_size(S):
    """Size đề xuất theo số đo khách đã nói (gọi API size-advice, có nhớ kết quả)."""
    m = known_measures(S)
    if not m:
        return None
    if m not in _SIZE_CACHE:
        _, r = api_get("/v1/size-advice", chieu_cao=m[0], can_nang=m[1])
        _SIZE_CACHE[m] = r["de_xuat"]["size"]
    return _SIZE_CACHE[m]

def _prod_by_id(pid):
    return next((p for p in CATALOG if p["id"] == pid), None)

def _size_tag(p, sz):
    if not sz or not p:
        return ""
    return f" - có size {sz}" if sz in p["size"] else f" - không có size {sz} (có {', '.join(p['size'])})"

def act_lookbook(S):
    st, d = api_get(f"/v1/lookbook/{quote(S['dip_mac'], safe='')}")
    if st == 404:
        S.pop("dip_mac", None)
        return "ask", ask_text("dip_mac")
    sz = my_size(S)
    lines, thieu = [], 0
    for p in d["items"]:
        note = "" if p["tinh_trang"] == "Còn hàng" else f" ({p['tinh_trang'].lower()})"
        cp = _prod_by_id(p["id"])
        tag = _size_tag(cp, sz)
        thieu += bool(sz and cp and sz not in cp["size"])
        lines.append(f"- {p['ten']} ({p['id']}): {vnd(p['gia'])}{note}{tag}")
    head = f"Dạ cho dịp {d['dip_mac']}, SUPEO gợi ý nàng set '{d['ten_bo']}' phong cách {d['phong_cach']} nè:\n"
    body = "\n".join(lines) + f"\nTổng cả set: {vnd(d['tong_gia_tien'])}.\n{d['ghi_chu']}\n"
    if sz:
        h, w = known_measures(S)
        foot = (f"Theo số đo nàng đã chia sẻ ({h}cm, {w}kg), nàng hợp Size {sz}. "
                + ("Cả set đều có size này, nàng chốt được luôn nhé! " if not thieu else
                   "Món nào chưa có size này SUPEO đã ghi chú ở trên để nàng cân nhắc nhé! ")
                + "Nàng muốn xem chi tiết món nào cứ nhắn tên món, hoặc hỏi SUPEO phối thêm áo/quần với món nàng thích nha!")
    else:
        foot = ("Nàng cho SUPEO xin chiều cao, cân nặng để chọn size từng món, "
                "hoặc nhắn tên món nàng thích để SUPEO gửi chi tiết nhé!")
    return "inform", head + body + foot

# ---------------------------------------------------------------- 2) Danh sách nhiều mẫu: ghi chú size theo số đo đã biết ----
_LIST_ASK = "Nàng cũng có thể cho SUPEO biết chiều cao, cân nặng để chọn đúng size."
try:
    _act_product_v5
except NameError:
    _act_product_v5 = act_product              # act_product ở đây đã gồm bản bọc "còn size M không" của v5

def act_product(S):
    kind, out = _act_product_v5(S)
    sz = my_size(S)
    if sz and kind == "inform" and _LIST_ASK in out:
        h, w = known_measures(S)
        new_lines = []
        for ln in out.split("\n"):
            m = re.match(r"^\d+\. .*\((P\d+)\)", ln)
            new_lines.append(ln + _size_tag(_prod_by_id(m.group(1)), sz) if m else ln)
        out = "\n".join(new_lines).replace(
            _LIST_ASK, f"Theo số đo nàng đã chia sẻ ({h}cm, {w}kg), nàng hợp Size {sz}; SUPEO đã ghi chú size ở từng mẫu rồi nè.")
    return kind, out

# ---------------------------------------------------------------- 3) "phối với áo nào?" -> gợi ý theo món đang nói tới ----
PAIR_RE = (r"\bphoi\b.*\b(?:voi|cung|chung)\b|\b(?:voi|cung)\s+(?:ao|quan|vay|dam|blazer|chan vay|so mi)?\s*nao\b"
           r"|\bmac\s+(?:gi\s+)?(?:voi|cung|kem)\b|\bket hop\b|\b(?:di|hop|dung)\s+voi\s+(?:ao|quan|vay|dam|blazer)\b")
COMPLEMENT = {"Đầm": ["Áo Blazer"], "Áo Blazer": ["Đầm", "Áo sơ mi", "Quần ống rộng", "Chân váy Midi"],
              "Áo sơ mi": ["Quần ống rộng", "Chân váy Midi", "Áo Blazer"],
              "Quần ống rộng": ["Áo sơ mi", "Áo Blazer"], "Chân váy Midi": ["Áo sơ mi", "Áo Blazer"]}

def _is_pairing(text):
    return bool(re.search(PAIR_RE, strip_acc(text).lower()))

def _lookbook_sets_with(pid):
    out = []
    for dip in OCCASIONS:
        st, d = api_get(f"/v1/lookbook/{quote(dip, safe='')}")
        if st == 200 and any(i["id"] == pid for i in d["items"]):
            out.append(d)
    return out

def act_pairing(state):
    S = state["slots"]
    anchor = _prod_by_id(next((p["id"] for p in CATALOG if p["ten"] == S.get("ten_sp")), None))
    if not anchor:
        return "ask", ask_text("dip_mac")
    want = match_key(positive_part(state["last_text"]), CATS)            # khách nói rõ muốn phối với nhóm nào ('phối với áo sơ mi')
    if want == anchor["loai"] and not re.search(r"\bkhac\b|\bnua\b", strip_acc(state["last_text"]).lower()):
        want = None
    sz = my_size(S)
    picks, seen = [], {anchor["id"]}
    for d in _lookbook_sets_with(anchor["id"]):                            # 1) món đã được SUPEO phối sẵn cùng món này
        for it in d["items"]:
            cp = _prod_by_id(it["id"])
            if it["id"] in seen or not cp or (want and cp["loai"] != want):
                continue
            seen.add(it["id"])
            picks.append((cp, f" - SUPEO đã phối sẵn trong set '{d['ten_bo']}' (dịp {d['dip_mac']})"))
    cats = [want] if want else COMPLEMENT.get(anchor["loai"], [])
    for c in cats:                                                          # 2) bổ sung món còn hàng ở nhóm có thể phối
        cand = [p for p in CATALOG if p["loai"] == c and p["id"] not in seen and p["ton_kho"] > 0
                and (not sz or sz in p["size"])]
        for cp in cand[:2 if len(picks) >= 2 else 3]:
            seen.add(cp["id"])
            picks.append((cp, ""))
    picks = picks[:5]
    h = f"Dạ để phối với {anchor['ten']} ({anchor['id']}), SUPEO gợi ý nàng:\n"
    if not picks:
        return "inform", h + "Hiện SUPEO chưa có món phù hợp để phối kèm, nàng nói dịp mặc để SUPEO gợi ý set khác nhé!"
    lines = [f"- {cp['ten']} ({cp['id']}): {vnd(cp['gia'])}{tag}{_size_tag(cp, sz)}" for cp, tag in picks]
    if sz:
        m = known_measures(S)
        foot = f"\nTheo số đo nàng đã chia sẻ ({m[0]}cm, {m[1]}kg), nàng hợp Size {sz}, SUPEO ưu tiên món có size này nhé!"
    else:
        foot = "\nNàng cho SUPEO xin chiều cao, cân nặng để chọn size từng món nhé!"
    return "inform", h + "\n".join(lines) + foot + "\nNàng thích món nào cứ nhắn tên món, SUPEO gửi chi tiết ngay."

def resolve_reference(text, ptxt, cat, state):
    """'áo này', 'nó', 'mẫu đó', 'còn hàng không?', 'phối với áo nào?' -> món đang nói tới (loại đồ so với chính món đang nhớ)."""
    f = state.get("focus", {})
    if not f.get("ten_sp"):
        return None
    p = next((x for x in CATALOG if x["ten"] == f["ten_sp"]), None)
    nfc = _nfc(text)
    na = strip_acc(nfc)
    if re.search(PAIR_RE, na) and not re.search(r"voucher|ma giam|gio hang|don hang|ma don|\bsp\d", na):
        return f["ten_sp"]                                                  # câu hỏi phối đồ luôn xoay quanh món đang xem
    if DEICTIC_ACC.search(nfc) or DEICTIC_NOACC.search(na):
        if cat and cat != "Váy" and p and cat != p["loai"]:
            return None
        return f["ten_sp"]
    if field_asked(text) and not cat and not state["slots"].get("ten_sp") \
            and not re.search(r"voucher|ma giam|gio hang|don hang|ma don|\bsp\d", na):
        return f["ten_sp"]
    return None

_rule_intent_v5 = rule_intent

def rule_intent(text, new, state_slots):
    if "ten_sp" in new and _is_pairing(text) and "dip_mac" not in new:
        return "lookbook"
    return _rule_intent_v5(text, new, state_slots)

_policy_v5 = policy

def policy(state):
    S = state["slots"]
    if state["intent"] == "lookbook" and not S.get("dip_mac") and S.get("ten_sp") and _is_pairing(state["last_text"]):
        return act_pairing(state)
    return _policy_v5(state)

# ---------------------------------------------------------------- 4) Tốc độ khi bật Gemini ----
# (a) DST bằng Gemini chỉ gọi khi luật không hiểu câu (mặc định 'fallback'); 'always' = như cũ; 'off' = không gọi.
# (b) NLG (viết lại câu trả lời bằng Gemini) mặc định TẮT: mẫu câu đã tự nhiên, bớt 1 lượt gọi mạng, không nguy cơ bịa số.
LLM_DST_MODE = os.environ.get("SUPEO_LLM_DST", "fallback")                  # fallback | always | off
LLM_NLG = os.environ.get("SUPEO_NLG", "0") == "1"                           # 1 = bật viết lại bằng Gemini
TIMING = os.environ.get("SUPEO_DEBUG_TIMING", "0") == "1"                   # 1 = in thời gian từng lượt ra console

_FAQ_RE = (r"cam on|thanks|thank|tam biet|\bbye\b|hen gap lai|thoi nhe|hoa toc|giao nhanh|phi ship|phi van chuyen|giao hang|"
           r"bao lau|thoi gian giao|giao toan quoc|doi tra|bao hanh|thanh toan|\bcod\b|hotline|so dien thoai|dia chi|"
           r"cua hang o dau|\b(?:xin chao|chao|hello|hi|alo|shop oi|ban oi)\b")

try:
    _update_state_v5
except NameError:
    _update_state_v5 = update_state

def update_state(state, text):
    global USE_LLM
    if not USE_LLM or LLM_DST_MODE == "always":
        if LLM_DST_MODE == "off" and USE_LLM:
            prev, USE_LLM = USE_LLM, False
            try:
                return _update_state_v5(state, text)
            finally:
                USE_LLM = prev
        return _update_state_v5(state, text)
    if LLM_DST_MODE == "off":
        return _update_state_v5(state, text)
    snap = copy.deepcopy(state)
    prev, USE_LLM = USE_LLM, False                                          
    try:
        _update_state_v5(state, text)
    finally:
        USE_LLM = prev
    if state["intent"] == "chitchat" and not re.search(_FAQ_RE, strip_acc(text).lower()):
        state.clear(); state.update(snap)                                  
        return _update_state_v5(state, text)
    return state

_nlg_v5 = nlg

def nlg(facts, user_text, state=None):
    return _nlg_v5(facts, user_text, state) if LLM_NLG else facts

_bot_reply_v5 = bot_reply

def bot_reply(text, state):
    t0 = time.time()
    n0 = _G["calls"] if "_G" in globals() and "calls" in _G else 0
    reply = _bot_reply_v5(text, state)
    if TIMING:
        n1 = _G["calls"] if "_G" in globals() and "calls" in _G else 0
        print(f"[timing] {1000 * (time.time() - t0):.0f} ms | gọi Gemini {n1 - n0} lần | intent={state['intent']} | {text[:40]!r}")
    return reply

print(f"Bản vá v6 sẵn sàng (DST Gemini: {LLM_DST_MODE}, NLG Gemini: {'bật' if LLM_NLG else 'tắt'})")


Bản vá v6 sẵn sàng (DST Gemini: fallback, NLG Gemini: tắt)


## Bước 7 - Bật LLM (Gemini API) cho DST + NLG
Gemini đảm nhận **DST** (tách intent/slot từ câu tự nhiên) và **NLG** (viết lại câu trả lời); số liệu vẫn lấy từ API/Excel và được `facts_ok` kiểm tra lại. Không có key hoặc API lỗi thì chatbot tự chạy bằng luật + regex.

**Lấy key vào Kaggle:** *Add-ons → Secrets → Add secret*, Label `GEMINI_API_KEY`, dán key, bật *Attach* (cần *Internet: On*). Muốn chạy không cần key: đổi `LLM_BACKEND = "qwen"` (cần GPU T4).


In [10]:
# Bật LLM cho DST + NLG bằng Gemini API (Google AI Studio). Không có key / lỗi kết nối thì chatbot tự chạy bằng luật + regex.
ENABLE_LLM = True
LLM_BACKEND = "gemini"               
GEMINI_MODEL = "gemini-3.5-flash-lite"   # đã kiểm tra: dùng được với tài khoản này
GEMINI_FALLBACKS = ["gemini-3.5-flash-lite", "gemini-3.1-flash-lite", "gemini-flash-lite-latest", "gemini-3.5-flash", "gemini-3-flash-preview"]   # chỉ gồm model đã trả HTTP 200; gemini-2.5-* đã bị Google ngừng cấp cho tài khoản mới
MODEL_ID = "Qwen/Qwen2.5-3B-Instruct" 

# Key: nên lưu ở Kaggle -> Add-ons -> Secrets (Label: GEMINI_API_KEY, bật Attach). Chỉ dán thẳng vào đây khi chạy cá nhân, KHÔNG chia sẻ notebook kèm key.
GEMINI_API_KEY = ""

import os, time, json, requests
USE_LLM = False

if ENABLE_LLM and LLM_BACKEND == "gemini":
    GBASE = "https://generativelanguage.googleapis.com/v1beta"
    _G = {"model": GEMINI_MODEL, "key": "", "fails": 0, "block_until": 0.0, "repicked": False, "tried": set(), "calls": 0, "think": {}}
    _SESSION = requests.Session()          # giữ kết nối (keep-alive): bỏ bước bắt tay TLS mỗi lượt gọi

    def _get_key():
        if GEMINI_API_KEY.strip():
            return GEMINI_API_KEY.strip()
        try:
            from kaggle_secrets import UserSecretsClient
            k = UserSecretsClient().get_secret("GEMINI_API_KEY")
            if k: return k.strip()
        except Exception:
            pass
        return os.environ.get("GEMINI_API_KEY", "").strip()

    def _pick_model(key):
        """Liệt kê model của tài khoản, chọn 1 model flash hỗ trợ generateContent."""
        r = requests.get(f"{GBASE}/models", headers={"x-goog-api-key": key}, params={"pageSize": 200}, timeout=20)
        r.raise_for_status()
        names = [m["name"].split("/", 1)[1] for m in r.json().get("models", [])
                 if "generateContent" in m.get("supportedGenerationMethods", [])]
        bad = ("lite", "image", "tts", "audio", "live", "embedding", "vision", "robotics", "computer")
        cand = [n for n in names if "flash" in n and not any(b in n for b in bad)]
        pool = [n for n in cand if "preview" not in n and "exp" not in n] or cand
        if not pool:
            raise RuntimeError("Không tìm thấy model flash nào hỗ trợ generateContent")
        return sorted(pool)[-1]

    def _next_model(cur):
        """Model kế tiếp chưa thử (để đổi sang khi model hiện tại hết hạn mức hoặc không tồn tại)."""
        _G["tried"].add(cur)
        return next((m for m in [GEMINI_MODEL] + GEMINI_FALLBACKS if m not in _G["tried"]), None)

    def gemini_llm(messages, max_new_tokens=200, timeout=25):
        """Cùng giao diện với llm() cũ: messages = [{role: system|user|assistant, content}], trả về chuỗi văn bản."""
        if time.time() < _G["block_until"]:
            raise RuntimeError("Gemini tạm nghỉ sau nhiều lỗi liên tiếp")
        sys_txt = "\n".join(m["content"] for m in messages if m["role"] == "system")
        contents = []
        for m in messages:
            if m["role"] == "system":
                continue
            role = "model" if m["role"] == "assistant" else "user"
            if contents and contents[-1]["role"] == role:
                contents[-1]["parts"][0]["text"] += "\n" + m["content"]
            else:
                contents.append({"role": role, "parts": [{"text": m["content"]}]})
        last_err = None
        for attempt in range(len(GEMINI_FALLBACKS) + 3):
            model = _G["model"]
            gen = {"temperature": 0, "maxOutputTokens": max_new_tokens + 512}
            tc = _THINK_OPTS[_G["think"].get(model, 0)]                 # giảm/tắt "suy nghĩ" (nguyên nhân chính khiến Gemini 3.x trả lời chậm)
            if tc is not None:
                gen["thinkingConfig"] = tc
            body = {"contents": contents, "generationConfig": gen}
            if sys_txt:
                body["system_instruction"] = {"parts": [{"text": sys_txt}]}
            try:
                _G["calls"] += 1
                r = _SESSION.post(f"{GBASE}/models/{model}:generateContent",
                                  headers={"x-goog-api-key": _G["key"], "Content-Type": "application/json"}, json=body, timeout=timeout)
                if r.status_code == 400 and "hink" in r.text and _G["think"].get(model, 0) < len(_THINK_OPTS) - 1:
                    _G["think"][model] = _G["think"].get(model, 0) + 1   # model không nhận mức suy nghĩ này -> thử mức nhẹ hơn / bỏ cấu hình
                    continue
                if r.status_code in (404, 429):
                    last_err = f"HTTP {r.status_code} (model {model}): {r.text[:300]}"
                    nxt = _next_model(model)
                    if nxt:
                        _G["model"] = nxt                              # model này hết hạn mức / không có -> thử model khác
                        continue
                    if r.status_code == 404 and not _G["repicked"]:
                        _G["repicked"] = True
                        _G["model"] = _pick_model(_G["key"])
                        continue
                    if r.status_code == 429:
                        time.sleep(2 * (attempt + 1))
                        continue
                    break
                if r.status_code in (500, 502, 503, 504):
                    last_err = f"HTTP {r.status_code}"
                    time.sleep(2 * (attempt + 1))
                    continue
                if r.status_code >= 400:
                    raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
                parts = r.json()["candidates"][0]["content"]["parts"]
                _G["fails"] = 0
                _G["tried"] = set()
                return "".join(p.get("text", "") for p in parts)
            except Exception as e:
                last_err = str(e)
                if "HTTP 4" in last_err and "HTTP 429" not in last_err:
                    break                                               # lỗi 4xx (sai key, sai tham số): không thử lại
        _G["fails"] += 1
        if _G["fails"] >= 3:
            _G["block_until"] = time.time() + 60                        # lỗi liên tiếp: nghỉ 60 giây, bot dùng luật + regex
        raise RuntimeError(last_err or "Gemini lỗi")

    # Mức "suy nghĩ" thử lần lượt (nhẹ nhất trước); model nào trả 400 vì không nhận thì tự chuyển sang mức kế tiếp / bỏ cấu hình.
    _THINK_OPTS = [{"thinkingLevel": "minimal"}, {"thinkingLevel": "low"}, {"thinkingBudget": 0}, None]
    _G["key"] = _get_key()
    if not _G["key"]:
        print("Chưa có GEMINI_API_KEY -> chạy chế độ luật + regex.\n"
              "Thêm key: Add-ons -> Secrets -> Add secret (Label: GEMINI_API_KEY), bật Attach, rồi chạy lại cell này.")
    else:
        try:
            llm = gemini_llm
            out = llm([{"role": "system", "content": "Chỉ trả lời đúng một từ."}, {"role": "user", "content": "Hãy nói: OK"}], 10)
            USE_LLM = True
            print(f"Đã kết nối Gemini ({_G['model']}). Phản hồi thử: {out.strip()[:30]!r}")
        except Exception as e:
            print(f"Không kết nối được Gemini -> chạy chế độ luật + regex.\nChi tiết: {e}\n"
                  "Kiểm tra: Internet đã bật chưa, key có đúng không, tên model còn dùng được không.")

elif ENABLE_LLM and LLM_BACKEND == "qwen":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM
    tok = AutoTokenizer.from_pretrained(MODEL_ID)
    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16, device_map="auto")

    def llm(messages, max_new_tokens=200):
        prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
        return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

    USE_LLM = True
    print("Đã tải", MODEL_ID)
else:
    print("Chạy chế độ luật + regex (USE_LLM = False)")


Đã kết nối Gemini (gemini-3.5-flash-lite). Phản hồi thử: 'OK'


## Bước 8 - Chạy thử hội thoại nhiều lượt
Mỗi lượt in ra DST (intent, slots), action và các API đã gọi.

In [11]:
# Chạy thử hội thoại nhiều lượt (mỗi hội thoại dùng 1 state riêng). Câu demo TỰ VIẾT, không lấy từ dialogue_benchmarks.
DEMOS = [
    ["Em cao 1m58, mặc size nào hợp ạ?", "47 cân nhé"],
    ["Gợi ý giúp mình outfit đi làm ngày mai nhé"],
    ["Check giúp mình đơn #SP3260", "Thế bao giờ giao tới nơi?"],                       # ghi nhớ đơn vừa tra
    ["Cho mình hỏi đang có ưu đãi gì không?", "Giỏ hàng mình 900k"],                               # ghi nhớ câu bot hỏi lại
    ["Cho mình xem Đầm voan hoa tay tiên", "Đầm này giá thế nào ạ?",              # ghi nhớ món đang nói tới
     "Vậy với số đo 1m63, 52kg thì chọn size gì?"],                                      # ghi nhớ số đo + món
]
for conv in DEMOS:
    st = new_state()
    print("=" * 70)
    for u in conv:
        print("KHÁCH:", u)
        print("SUPEO:", bot_reply(u, st))
        tr = st["trace"][-1]
        print(f"   [DST] intent={tr['intent']} slots={ {k: v for k, v in tr['slots'].items() if k != '_list'} } | action={tr['action']} | API={[c['GET'] for c in tr['api_calls']]}")
        print(f"   [NHỚ] focus={ {k: v for k, v in st['focus'].items() if k != '_list'} }\n")


KHÁCH: Em cao 1m58, mặc size nào hợp ạ?
SUPEO: Dạ nàng chia sẻ thêm cân nặng (kg) để shop tư vấn chuẩn xác nha!
   [DST] intent=size_advice slots={'chieu_cao': 158} | action=ask | API=[]
   [NHỚ] focus={}

KHÁCH: 47 cân nhé
SUPEO: Dạ với chiều cao 158cm và cân nặng 47kg, nàng mặc chuẩn Size S (150-159cm, 40-48kg) là vừa vặn nhất ạ!
(Phù hợp vóc dáng mảnh mai, nhỏ nhắn. Nếu eo < 65cm mặc form chuẩn ôm đẹp.)
Nàng nói tên mẫu hoặc nhóm đồ đang quan tâm, SUPEO kiểm tra giúp mẫu đó còn size này không nhé!
   [DST] intent=size_advice slots={'chieu_cao': 158, 'can_nang': 47} | action=inform | API=['/v1/size-advice']
   [NHỚ] focus={}

KHÁCH: Gợi ý giúp mình outfit đi làm ngày mai nhé
SUPEO: Dạ cho dịp Công sở, SUPEO gợi ý nàng set 'Gọn gàng chuyên nghiệp' phong cách Smart Casual nè:
- Áo Blazer suông cổ điển (P01): 890.000đ
- Quần suông công sở (P09): 620.000đ
Tổng cả set: 1.510.000đ.
Áo Blazer suông cổ điển (P01) phối Quần suông công sở (P09). Chuẩn phong thái nữ quyền.
Nàng cho SUPEO xin ch

## Bước 9 - Đánh giá (đã sửa lỗi rò rỉ dữ liệu)

**Vì sao trước đây ra 100%?** Sheet `dialogue_benchmarks` vừa là *đề thi* vừa là *tài liệu ôn*: câu few-shot, câu demo, câu ví dụ Gradio và hai câu FAQ đều lấy từ chính sheet đó, và luật/regex được chỉnh cho đến khi 20 lượt đều đúng. Con số 100% vì vậy chỉ cho biết "bot thuộc bài", không cho biết bot hiểu câu mới.

Cách đo mới:
- **9a** bộ held-out độc lập (28 câu 1 lượt + 12 hội thoại nhiều lượt để kiểm tra **trí nhớ**), nhãn viết tay, không dùng để chỉnh bot.
- **9b** kiểm tra rò rỉ tự động: few-shot / demo / ví dụ / FAQ không được trùng benchmark.
- **9c** đo held-out (con số báo cáo); chấm slot theo khớp **chính xác**.
- **9d** benchmark cũ chỉ còn là smoke-test.

**Cập nhật v5:** 9c' tách 3 bộ DEV / TEST / FRESH (chạy FRESH đúng 1 lần, có file khóa), báo cáo theo từng intent; 9e kiểm tra bộ nhớ (kể cả LLM giả trả sai). Bản vá v5 nằm ở cell ngay sau Policy+NLG.


In [12]:
# ============ 9a. BỘ KIỂM TRA ĐỘC LẬP (held-out) ============
# BỘ KIỂM TRA ĐỘC LẬP (held-out): câu mới, KHÔNG nằm trong sheet dialogue_benchmarks,
# KHÔNG dùng làm few-shot / demo / ví dụ. Nhãn do người viết gán tay từ dữ liệu trong Excel.
# Mỗi phần tử: (session, câu khách, intent_đúng, slot_đúng, action_đúng, [chuỗi phải có trong câu trả lời])
HELDOUT_SINGLE = [
 ("H01","Cho mình hỏi áo sơ mi croptop cổ bẻ có những màu gì?","product_info",{"ten_sp":"Áo sơ mi croptop cổ bẻ","loai_do":"Áo sơ mi"},"inform",["Vàng bơ"]),
 ("H02","Đầm slip satin còn hàng không ạ?","product_info",{"ten_sp":"Đầm slip satin","loai_do":"Đầm"},"inform",["sắp hết hàng"]),
 ("H03","Quần culottes bao nhiêu tiền vậy","product_info",{"ten_sp":"Quần lửng culottes ống rộng","loai_do":"Quần ống rộng"},"inform",["540.000đ"]),
 ("H04","Mình cần tìm chân váy màu đen","product_info",{"loai_do":"Chân váy Midi","tu_khoa":"đen"},"inform",[]),
 ("H05","Có đầm nào chất voan không shop","product_info",{"loai_do":"Đầm","tu_khoa":"voan"},"inform",["P11"]),
 ("H06","Em cao 1m70, 60 ký thì mặc size nào?","size_advice",{"chieu_cao":170,"can_nang":60},"inform",["Size L"]),
 ("H07","Mình 1m58 nặng 47kg mặc đầm size gì nhỉ","size_advice",{"chieu_cao":158,"can_nang":47,"loai_do":"Đầm"},"inform",["Size S"]),
 ("H08","Cao 1m75 nặng 70kg","size_advice",{"chieu_cao":175,"can_nang":70},"inform",["Size XL"]),
 ("H09","Tư vấn size giúp mình với","size_advice",{},"ask",[]),
 ("H10","Mai đi làm mặc gì cho lịch sự nhỉ?","lookbook",{"dip_mac":"Công sở"},"inform",[]),
 ("H11","Cuối tuần này mình đi tiệc cocktail, nên mặc gì?","lookbook",{"dip_mac":"Tiệc cocktail"},"inform",[]),
 ("H12","Gợi ý cho mình bộ đồ đi du lịch biển","lookbook",{"dip_mac":"Du lịch hè"},"inform",[]),
 ("H13","Đi hẹn hò thơ mộng thì phối sao shop","lookbook",{"dip_mac":"Hẹn hò thơ mộng"},"inform",[]),
 ("H14","Phối đồ giúp mình với","lookbook",{},"ask",[]),
 ("H15","Đơn #SP3150 đến đâu rồi bạn","order_check",{"ma_don":"#SP3150"},"inform",["Đang chuẩn bị hàng","02/11/2026"]),
 ("H16","tra giúp mình đơn sp3255","order_check",{"ma_don":"#SP3255"},"inform",["31/10/2026"]),
 ("H17","Đơn hàng của mình giao tới đâu rồi?","order_check",{},"ask",[]),
 ("H18","#SP1234 có không shop","order_check",{"ma_don":"#SP1234"},"inform",["chưa tìm thấy"]),
 ("H19","Giỏ hàng của mình 1tr2, dùng mã nào được?","voucher",{"cart_total":1200000},"inform",["FREESHIPMAX"]),
 ("H20","Mã SUPEOAUTUMN còn dùng được không?","voucher",{"ma_voucher":"SUPEOAUTUMN"},"inform",["SUPEOAUTUMN"]),
 ("H21","Mình có mã WELCOME10, đơn 300k dùng được không?","voucher",{"ma_voucher":"WELCOME10","cart_total":300000},"inform",["WELCOME10"]),
 ("H22","Còn mã freeship nào không?","voucher",{},"inform",["FREESHIPMAX"]),
 ("H23","Mã SUMMER2026 dùng được chưa","voucher",{"ma_voucher":"SUMMER2026"},"inform",["SUMMER2026"]),
 ("H24","Shop có đổi trả không?","chitchat",{},"chitchat",[]),
 ("H25","Chào shop nhé","chitchat",{},"chitchat",[]),
 ("H26","Ok cảm ơn bạn, mình hiểu rồi","chitchat",{},"chitchat",[]),
 ("H27","Shop ở đâu vậy?","chitchat",{},"chitchat",[]),
 ("H28","Bạn có bán giày không?","chitchat",{},"chitchat",[]),
]
# Hội thoại nhiều lượt: kiểm tra KHẢ NĂNG GHI NHỚ. Mỗi lượt: (câu khách, [chuỗi phải có trong câu trả lời], [chuỗi KHÔNG được có])
HELDOUT_MULTI = {
 "M01_nho_san_pham": [("Mình quan tâm Áo Blazer linen mùa hè",["P03"],[]),
                      ("Giá bao nhiêu vậy?",["820.000đ"],[]),
                      ("Còn size nào không?",["S, M, L, XL"],[])],
 "M02_nho_so_do_sang_san_pham": [("Mình cao 1m64, nặng 55kg",["Size M"],[]),
                      ("Vậy mặc áo sơ mi lụa tay phồng được không?",["P06","Size M"],[])],
 "M03_chon_theo_so_thu_tu": [("Có những mẫu đầm nào vậy shop?",["P10","P12"],[]),
                      ("Cho mình xem mẫu thứ 3",["P12"],["P10"])],
 "M04_nho_don_hang": [("Check đơn #SP3201 giúp mình",["29/10/2026"],[]),
                      ("Thế khi nào thì giao tới nơi?",["29/10/2026"],[])],
 "M05_bot_hoi_lai_gio_hang": [("Có mã giảm giá nào không?",["giỏ hàng"],[]),
                      ("750k",["FLASHDEAL20"],[])],
 "M06_bot_hoi_lai_can_nang": [("Mình cao 1m60",["cân nặng"],[]),
                      ("49 ký",["Size M"],[])],
 "M07_nho_so_do_qua_chu_de_khac": [("Mình cao 1m68 nặng 58kg",["Size L"],[]),
                      ("Mai đi tiệc tối mặc gì nhỉ?",["Tiệc tối"],[]),
                      ("Vậy mình mặc size nào cho bộ đó?",["Size L"],[])],
 "M08_dai_tu_chi_dinh": [("Cho mình hỏi Đầm babydoll hoa nhí",["P27"],[]),
                      ("Đầm này có màu gì vậy?",["Vàng nhạt"],["P10","P26"])],
 "M09_hai_don_lien_tiep": [("Đơn #SP2740 sao rồi",["Giao thất bại"],[]),
                      ("Còn đơn #SP2650?",["Đã hủy"],[])],
 "M10_nho_san_pham_qua_nhieu_luot": [("Blazer cộc tay vạt chéo giá bao nhiêu",["780.000đ"],[]),
                      ("Còn hàng không?",["còn hàng"],[]),
                      ("Mình cao 1m65 nặng 53kg thì mặc size nào?",["P15","Size M"],[])],
 "M11_chon_tu_danh_sach_vay": [("Mình muốn mua váy",["P10"],[]),
                      ("Cái thứ hai",["P11"],["P10"])],
 "M12_nhac_lai_cau_truoc": [("Áo sơ mi trắng cổ đức giá bao nhiêu?",["520.000đ"],[]),
                      ("Chất liệu gì vậy?",["cotton poplin"],[]),
                      ("Mẫu đó còn hàng không?",["P07","còn hàng"],[])],
}
SLOT_KEYS = ["loai_do","dip_mac","chieu_cao","can_nang","ma_don","ma_voucher","cart_total","ten_sp","tu_khoa"]

def run_heldout(new_state, bot_reply, verbose=True):
    import pandas as pd
    rows = []
    for sid, text, e_int, e_slots, e_act, must in HELDOUT_SINGLE:
        st = new_state(); reply = bot_reply(text, st); tr = st["trace"][-1]
        got = {k: v for k, v in tr["slots"].items() if k in SLOT_KEYS}
        rows.append({"id": sid, "câu khách": text[:50], "intent_ok": tr["intent"] == e_int,
                     "slot_ok": got == e_slots, "action_ok": tr["action"] == e_act,
                     "reply_ok": all(m in reply for m in must),
                     "dự_đoán": f"{tr['intent']}/{tr['action']} {got}", "kỳ_vọng": f"{e_int}/{e_act} {e_slots}"})
    single = pd.DataFrame(rows)
    mrows = []
    for sid, turns in HELDOUT_MULTI.items():
        st = new_state()
        for k, (text, must, mustnot) in enumerate(turns, 1):
            reply = bot_reply(text, st)
            ok = all(m in reply for m in must) and not any(m in reply for m in mustnot)
            mrows.append({"id": f"{sid}-T{k}", "ok": ok, "câu khách": text, "trả lời": reply.replace("\n", " ")[:150]})
    multi = pd.DataFrame(mrows)
    if verbose:
        print(f"[HELD-OUT 1 lượt] {len(single)} câu | Intent {single.intent_ok.mean():.1%} | Slot (khớp CHÍNH XÁC) {single.slot_ok.mean():.1%} | "
              f"Action {single.action_ok.mean():.1%} | Nội dung trả lời {single.reply_ok.mean():.1%}")
        by_s = multi.groupby(multi.id.str.split("-T").str[0]).ok.all()
        print(f"[HELD-OUT nhiều lượt] {len(multi)} lượt | đúng từng lượt {multi.ok.mean():.1%} | đúng CẢ hội thoại {by_s.mean():.1%} ({int(by_s.sum())}/{len(by_s)})")
    return single, multi


In [13]:
# ============ 9b. KIỂM TRA RÒ RỈ DỮ LIỆU - chạy trước khi đo ============
import difflib, re, os
import pandas as pd

def _n(s):
    return re.sub(r"\s+", " ", re.sub(r"[^\w\s#]", " ", strip_acc(str(s)).lower())).strip()

def load_bench():
    raw = pd.read_excel(os.environ["SUPEO_XLSX"], sheet_name="dialogue_benchmarks", header=None, dtype=object)
    hdr = next(i for i in range(len(raw)) if str(raw.iloc[i, 0]).strip() == "session_id")
    b = raw.iloc[hdr + 1:].copy(); b.columns = [str(c).strip() for c in raw.iloc[hdr]]
    return b[b["session_id"].notna()].reset_index(drop=True)

bench = load_bench()

def _sentences(txt):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+|\n", str(txt)) if len(s.strip()) >= 25]

def _ratio(a, b):
    return difflib.SequenceMatcher(None, _n(a), _n(b)).ratio()

def audit_leak(threshold=0.85, strict=True):
    """Văn bản dùng để 'dạy/minh họa' bot (few-shot, demo, câu ví dụ, FAQ) KHÔNG được trùng/gần trùng sheet dialogue_benchmarks;
    bộ held-out cũng không được trùng few-shot/demo/ví dụ. Trùng -> báo lỗi."""
    b_utts = list(bench["user_utterance"])
    b_sents = [s for r in bench["ground_truth_bot_response"] for s in _sentences(r)]
    user_src = {"FEWSHOT": [x[0] for x in globals().get("FEWSHOT", [])],
                "DEMOS": [u for conv in globals().get("DEMOS", []) for u in conv],
                "EXAMPLES (chip Gradio)": list(globals().get("EXAMPLES", [])),
                "HELD-OUT": [x[1] for x in HELDOUT_SINGLE] + [t[0] for v in HELDOUT_MULTI.values() for t in v]}
    bot_src = {"FAQ_SHIPPING": FAQ_SHIPPING, "FAQ_THANKS": FAQ_THANKS, "HELP": HELP}
    hits = []
    for name, texts in user_src.items():
        hits += [(name, t, u, r) for t in texts for u in b_utts if (r := _ratio(t, u)) >= threshold]
    for name, txt in bot_src.items():
        hits += [(name, s, g, r) for s in _sentences(txt) for g in b_sents if (r := _ratio(s, g)) >= threshold]
    pool = user_src["FEWSHOT"] + user_src["DEMOS"] + user_src["EXAMPLES (chip Gradio)"]
    hits += [("HELD-OUT trùng few-shot/demo/ví dụ", t, d, r) for t in user_src["HELD-OUT"] for d in pool if (r := _ratio(t, d)) >= threshold]
    print(f"Đã so: {sum(len(v) for v in user_src.values())} câu khách + {len(bot_src)} đoạn trả lời mẫu với {len(b_utts)} câu / {len(b_sents)} vế đáp án của benchmark")
    if hits:
        for name, a, b, r in hits:
            print(f"  RÒ RỈ [{name}] ({r:.0%}): {a[:70]!r}  ~  {b[:70]!r}")
        if strict:
            raise AssertionError(f"Phát hiện {len(hits)} chỗ trùng với benchmark - hãy viết lại các câu đó.")
    else:
        print("OK: không có câu few-shot / demo / ví dụ / FAQ nào trùng benchmark; held-out độc lập.")
    return hits

_ = audit_leak()


Đã so: 72 câu khách + 3 đoạn trả lời mẫu với 20 câu / 36 vế đáp án của benchmark
OK: không có câu few-shot / demo / ví dụ / FAQ nào trùng benchmark; held-out độc lập.


In [14]:
# ============ BỘ TEST A / B / C cho SUPEO ============
# Định dạng single: (id, câu khách, intent_đúng, {slot_chính_xác}, action_đúng, [chuỗi phải có trong câu trả lời])
# Định dạng multi : {"tên": [(câu khách, [phải có], [không được có]), ...]}
# Nhãn được suy ra từ file Excel (giá, size, trạng thái đơn, mã giảm giá...) với ngày API = 2026-10-28, KHÔNG suy ra từ kết quả chạy của bot.

# ---------------------------------------------------------------- A (DEV): được xem lỗi và sửa theo ----------------
SUITE_A_SINGLE = [
 ("A01","Blazer dáng lửng còn hàng không shop","product_info",{"ten_sp":"Áo Blazer dáng lửng","loai_do":"Áo Blazer"},"inform",["P02"]),
 ("A02","Đầm sơ mi thắt đai eo có những màu nào","product_info",{"ten_sp":"Đầm sơ mi thắt đai eo","loai_do":"Đầm"},"inform",["Xanh pastel","Be"]),
 ("A03","chân váy xòe xếp tầng bao nhiêu thế","product_info",{"ten_sp":"Chân váy Midi xòe xếp tầng","loai_do":"Chân váy Midi"},"inform",["580.000đ"]),
 ("A04","Có áo blazer nào màu xám không","product_info",{"loai_do":"Áo Blazer","tu_khoa":"xám"},"inform",["P01","P13"]),
 ("A05","tìm giúp mình quần màu nâu mocha","product_info",{"loai_do":"Quần ống rộng","tu_khoa":"nâu mocha","ten_sp":"Quần ống rộng xếp ly đôi"},"inform",["P22"]),
 ("A06","Quần suông công sở có những size nào ạ","product_info",{"ten_sp":"Quần suông công sở","loai_do":"Quần ống rộng"},"inform",["S, M, L, XL"]),
 ("A07","Áo Blazer tay lỡ một khuy giá sao shop","product_info",{"ten_sp":"Áo Blazer tay lỡ một khuy","loai_do":"Áo Blazer"},"inform",["860.000đ"]),
 ("A08","Đầm ôm body còn size S không","product_info",{"ten_sp":"Đầm ôm body cổ yếm","loai_do":"Đầm"},"inform",["size S"]),
 ("A09","Có váy nào màu champagne không","product_info",{"loai_do":"Đầm","tu_khoa":"champagne","ten_sp":"Đầm slip satin"},"inform",["P12"]),
 ("A10","Mình cao 1m63 nặng 54kg, mặc size gì nhỉ","size_advice",{"chieu_cao":163,"can_nang":54},"inform",["Size M"]),
 ("A11","cao 1m66, 62kg","size_advice",{"chieu_cao":166,"can_nang":62},"inform",["Size L"]),
 ("A12","Em 1m52 42 ký thì size gì","size_advice",{"chieu_cao":152,"can_nang":42},"inform",["Size S"]),
 ("A13","Tư vấn giúp mình size đầm với, mình nặng 58kg","size_advice",{"can_nang":58,"loai_do":"Đầm"},"ask",["chiều cao"]),
 ("A14","Mình cao 1m77 nặng 66kg thì lấy áo sơ mi croptop cổ bẻ size gì","size_advice",{"chieu_cao":177,"can_nang":66,"ten_sp":"Áo sơ mi croptop cổ bẻ","loai_do":"Áo sơ mi"},"inform",["Size XL","S, M, L"]),
 ("A15","Đi hẹn hò tối nay mặc gì cho xinh","lookbook",{"dip_mac":"Hẹn hò"},"inform",["1.180.000đ"]),
 ("A16","Gợi ý đồ đi dự tiệc ngoài trời","lookbook",{"dip_mac":"Dự tiệc ngoài trời"},"inform",["1.700.000đ"]),
 ("A17","Dạo phố mùa thu mặc gì cho hợp","lookbook",{"dip_mac":"Dạo phố mùa thu"},"inform",["1.370.000đ"]),
 ("A18","Đơn #SP2908 giao tới đâu rồi","order_check",{"ma_don":"#SP2908"},"inform",["Đang giao","GHN84920"]),
 ("A19","tra đơn sp3215 giúp mình","order_check",{"ma_don":"#SP3215"},"inform",["Đang chuẩn bị hàng","1.870.000đ"]),
 ("A20","Đơn #SP9999 sao rồi","order_check",{"ma_don":"#SP9999"},"inform",["chưa tìm thấy"]),
 ("A21","Mã SUPEOVIP100 dùng cho đơn 300k được không","voucher",{"ma_voucher":"SUPEOVIP100","cart_total":300000},"inform",["SUPEOVIP100","200.000đ"]),
 ("A22","Hiện có những mã giảm giá nào vậy shop","voucher",{},"inform",["FLASHDEAL20","SUPEOAUTUMN"]),
 ("A23","Mình mua 1tr5 thì áp mã nào được","voucher",{"cart_total":1500000},"inform",["FLASHDEAL20"]),
 ("A24","Mình có mã SUPEOSHIP30, xài được không?","voucher",{"ma_voucher":"SUPEOSHIP30"},"inform",["SUPEOSHIP30","chưa thấy"]),
 ("A25","Cho mình hỏi bên shop có bảo hành không?","chitchat",{},"chitchat",["chưa có thông tin"]),
 ("A26","Hello shop ơi","chitchat",{},"chitchat",["SUPEO chào"]),
 ("A27","Thanks shop nhiều nha","chitchat",{},"chitchat",["cảm ơn"]),
 ("A28","Áo sơ mi kẻ sọc dáng rộng còn hàng không","product_info",{"ten_sp":"Áo sơ mi kẻ sọc dáng rộng","loai_do":"Áo sơ mi"},"inform",["P20"]),
 ("A29","Đơn 700k dùng mã SUPEOAUTUMN được không","voucher",{"ma_voucher":"SUPEOAUTUMN","cart_total":700000},"inform",["SUPEOAUTUMN"]),
 ("A30","Đầm voan hoa tay tiên mặc size nào vậy","size_advice",{"ten_sp":"Đầm voan hoa tay tiên","loai_do":"Đầm"},"ask",["chiều cao"]),
]
SUITE_A_MULTI = {
 "AM1_nho_mon_gia_size": [("Cho mình xem Áo Blazer dáng lửng",["P02"],[]),
                          ("Mẫu đó bán bao nhiêu thế",["950.000đ"],[]),
                          ("Có size XL không?",["S, M, L"],[])],
 "AM2_chon_theo_thu_tu": [("Có những chân váy midi nào vậy?",["P04","P05","P29"],[]),
                          ("Lấy cái thứ 2 cho mình xem với",["P05"],[]),
                          ("Cái đó giá bao nhiêu?",["450.000đ"],[])],
 "AM3_so_do_sang_mon": [("Số đo của mình là 1m70, 63kg",["Size L"],[]),
                        ("Vậy áo sơ mi trắng cổ đức mặc được size nào",["P07","Size L"],[])],
 "AM4_nho_don": [("Xem giúp mình đơn #SP2908 đang ở đâu",["Đang giao"],[]),
                 ("Bao lâu nữa thì tới?",["#SP2908","30/10/2026"],[]),
                 ("Thế còn đơn #SP1001",["Đã giao thành công"],["GHN84920"])],
 "AM5_bot_hoi_gio_hang": [("Cho mình hỏi mã giảm giá",["giỏ hàng"],[]),
                          ("tầm 1tr",["1.000.000đ","FLASHDEAL20"],[])],
 "AM6_doi_y_so_do": [("Cho mình hỏi, mình cao 1m55 và nặng 45kg nhé",["Size S"],[]),
                     ("Ờ nhầm, mình cao 1m65 nặng 55kg",["Size M"],["Size S"]),
                     ("Cho mình xem Đầm slip satin",["P12","Size M"],[])],
}

# ---------------------------------------------------------------- B (TEST): đóng băng, không xem lỗi chi tiết khi chỉnh ----------------
SUITE_B_SINGLE = [
 ("B01","Áo blazer cộc tay bán bao nhiêu vậy shop","product_info",{"ten_sp":"Áo Blazer cộc tay vạt chéo","loai_do":"Áo Blazer"},"inform",["780.000đ"]),
 ("B02","chữ A dập ly có size nào","product_info",{"ten_sp":"Chân váy Midi chữ A dập ly","loai_do":"Chân váy Midi"},"inform",["S, M, L, XL"]),
 ("B03","áo sơ mi cổ đức còn hàng không","product_info",{"ten_sp":"Áo sơ mi trắng cổ đức","loai_do":"Áo sơ mi"},"inform",["P07"]),
 ("B04","đầm champagne giá sao shop","product_info",{"loai_do":"Đầm","tu_khoa":"champagne","ten_sp":"Đầm slip satin"},"inform",["1.150.000đ"]),
 ("B05","Quần culottes còn size M không","product_info",{"ten_sp":"Quần lửng culottes ống rộng","loai_do":"Quần ống rộng"},"inform",["size M"]),
 ("B06","Cho mình xem chân váy xếp ly","product_info",{"ten_sp":"Chân váy Midi xếp ly","loai_do":"Chân váy Midi"},"inform",["P04"]),
 ("B07","Có áo sơ mi trắng không shop","product_info",{"loai_do":"Áo sơ mi","tu_khoa":"trắng"},"inform",["P07","P19"]),
 ("B08","đầm kem có không","product_info",{"loai_do":"Đầm","tu_khoa":"kem","ten_sp":"Đầm midi cổ vuông"},"inform",["P11"]),
 ("B09","Quần lưng cao chất liệu gì vậy","product_info",{"ten_sp":"Quần ống rộng lưng cao","loai_do":"Quần ống rộng"},"inform",["tuyết mưa co giãn"]),
 ("B10","Size nào hợp với mình nhỉ","size_advice",{},"ask",["chiều cao"]),
 ("B11","Mình mặc size nào nhỉ, cao 1m64 55kg","size_advice",{"chieu_cao":164,"can_nang":55},"inform",["Size M"]),
 ("B12","Cao 1m70, 70 cân thì size gì","size_advice",{"chieu_cao":170,"can_nang":70},"inform",["Size XL"]),
 ("B13","mình 1m60 mà nặng 62kg","size_advice",{"chieu_cao":160,"can_nang":62},"inform",["Size L"]),
 ("B14","Cuối tuần đi cocktail party, phối đồ sao nhỉ","lookbook",{"dip_mac":"Tiệc cocktail"},"inform",["2.030.000đ"]),
 ("B15","Mình muốn bộ đồ đi du lịch mùa hè","lookbook",{"dip_mac":"Du lịch hè"},"inform",["1.570.000đ"]),
 ("B16","Cuối tuần đi dạo phố nên mặc gì","lookbook",{"dip_mac":"Dạo phố"},"inform",["1.070.000đ"]),
 ("B17","Cho mình biết đơn #SP2890 đã giao chưa","order_check",{"ma_don":"#SP2890"},"inform",["Đã giao thành công","SPX99310"]),
 ("B18","đơn sp3201 đi tới đâu rồi","order_check",{"ma_don":"#SP3201"},"inform",["Đang giao","VT883921"]),
 ("B19","Đơn hàng #SP3260 chưa bàn giao à","order_check",{"ma_don":"#SP3260"},"inform",["Đang chuẩn bị hàng","03/11/2026"]),
 ("B20","Có 2 triệu thì được giảm bao nhiêu","voucher",{"cart_total":2000000},"inform",["FLASHDEAL20"]),
 ("B21","Mã FREESHIPMAX áp cho đơn 700k được không","voucher",{"ma_voucher":"FREESHIPMAX","cart_total":700000},"inform",["FREESHIPMAX","100.000đ"]),
 ("B22","mã SUPEOVIP100 còn hạn không","voucher",{"ma_voucher":"SUPEOVIP100"},"inform",["SUPEOVIP100","31/12/2026"]),
 ("B23","Có mã FLASHDEAL20 áp cho đơn 500k không","voucher",{"ma_voucher":"FLASHDEAL20","cart_total":500000},"inform",["FLASHDEAL20","100.000đ"]),
 ("B24","Bên mình có chi nhánh ở Hà Nội không?","chitchat",{},"chitchat",[]),
 ("B25","Mình muốn đổi trả áo thì làm sao","chitchat",{},"chitchat",["chưa có thông tin"]),
 ("B26","Tạm biệt shop nhé","chitchat",{},"chitchat",["hẹn gặp lại"]),
 ("B27","Quần ống rộng xếp ly đôi có màu gì","product_info",{"ten_sp":"Quần ống rộng xếp ly đôi","loai_do":"Quần ống rộng"},"inform",["Nâu mocha"]),
 ("B28","Đầm babydoll còn hàng không","product_info",{"ten_sp":"Đầm babydoll hoa nhí","loai_do":"Đầm"},"inform",["P27"]),
 ("B29","Mình cao 1m58 mặc đầm size gì","size_advice",{"chieu_cao":158,"loai_do":"Đầm"},"ask",["cân nặng"]),
 ("B30","Đơn #SP3150 của mình bao giờ giao","order_check",{"ma_don":"#SP3150"},"inform",["Đang chuẩn bị hàng","02/11/2026"]),
]
SUITE_B_MULTI = {
 "BM1_mau_doi_mon": [("Mình quan tâm đầm suông dự tiệc",["P10"],[]),
                     ("Nó có những màu nào",["Đen","Đỏ đô"],[]),
                     ("Còn đầm ôm body thì sao?",["P26"],["P10"]),
                     ("Cái này bao nhiêu tiền nhỉ",["1.080.000đ"],["1.250.000đ"])],
 "BM2_don_qua_chu_de": [("Check đơn #SP3201",["Đang giao","VT883921"],[]),
                        ("Mình cao 1m61 nặng 50kg, size gì?",["Size M"],[]),
                        ("Đơn đó dự kiến giao ngày nào?",["29/10/2026"],[])],
 "BM3_bot_hoi_can_nang": [("Chiều cao của mình là 1m69",["cân nặng"],[]),
                          ("59kg",["Size L"],[])],
 "BM4_danh_sach_chon_so": [("Có áo sơ mi nào vậy shop?",["P06","P07"],[]),
                           ("Mẫu thứ 4 giá bao nhiêu",["560.000đ"],[]),
                           ("Còn size nào vậy",["S, M, L, XL"],[])],
 "BM5_voucher_doi_gio": [("Giỏ hàng mình 900k dùng mã nào?",["900.000đ"],[]),
                         ("Nếu 1tr3 thì sao",["1.300.000đ"],["900.000đ"])],
 "BM6_lookbook_so_do_mon": [("Tuần này mình có hẹn hò thơ mộng, gợi ý đồ nhé",["P19","P18"],[]),
                            ("Hôm nay mình đo được 1m62, 51kg",["Size M"],[]),
                            ("Áo sơ mi cổ nơ ruy băng còn size M không",["P19","size M"],[])],
}

# ---------------------------------------------------------------- C (FRESH): chạy ĐÚNG 1 LẦN ----------------
SUITE_C_SINGLE = [
 ("C01","Áo Blazer kẻ caro vintage có những size nào vậy","product_info",{"ten_sp":"Áo Blazer kẻ caro vintage","loai_do":"Áo Blazer"},"inform",["S, M, L"]),
 ("C02","Mình đang phân vân đầm ôm body cổ yếm, giá bao nhiêu thế","product_info",{"ten_sp":"Đầm ôm body cổ yếm","loai_do":"Đầm"},"inform",["1.080.000đ"]),
 ("C03","quần ống suông cạp chun sau làm bằng vải gì","product_info",{"ten_sp":"Quần ống suông cạp chun sau","loai_do":"Quần ống rộng"},"inform",["đũi lụa chống nhăn"]),
 ("C04","Chân váy đuôi cá còn hàng không","product_info",{"ten_sp":"Chân váy Midi đuôi cá","loai_do":"Chân váy Midi"},"inform",["P16"]),
 ("C05","Shop còn áo sơ mi hồng nhạt không","product_info",{"loai_do":"Áo sơ mi","tu_khoa":"hồng nhạt","ten_sp":"Áo sơ mi lụa tay phồng"},"inform",["P06"]),
 ("C06","Cho mình xem mấy cái quần ống rộng màu be","product_info",{"loai_do":"Quần ống rộng","tu_khoa":"be"},"inform",["P08","P22"]),
 ("C07","Đầm nào có chất liệu satin vậy","product_info",{"loai_do":"Đầm","tu_khoa":"satin"},"inform",["P10","P12"]),
 ("C08","Quần ống rộng lưng cao còn size XL không","product_info",{"ten_sp":"Quần ống rộng lưng cao","loai_do":"Quần ống rộng"},"inform",["XL"]),
 ("C09","Chị cao 1m68 nặng 60kg mặc size nào ạ","size_advice",{"chieu_cao":168,"can_nang":60},"inform",["Size L"]),
 ("C10","mình 52 ký, cao 1m55","size_advice",{"chieu_cao":155,"can_nang":52},"inform",["Size M"]),
 ("C11","Cho mình hỏi size, mình cao 1m72","size_advice",{"chieu_cao":172},"ask",["cân nặng"]),
 ("C12","Em cao 1m80 nặng 80kg đặt áo blazer dáng lửng size gì","size_advice",{"chieu_cao":180,"can_nang":80,"ten_sp":"Áo Blazer dáng lửng","loai_do":"Áo Blazer"},"inform",["Size XL","S, M, L"]),
 ("C13","Mai mình gặp đối tác làm ăn, nên mặc gì","lookbook",{"dip_mac":"Gặp đối tác"},"inform",["1.730.000đ"]),
 ("C14","Chiều nay mình đi uống trà chiều, phối đồ giúp mình","lookbook",{"dip_mac":"Trà chiều"},"inform",["1.340.000đ"]),
 ("C15","Shop gợi ý set casual friday giúp mình","lookbook",{"dip_mac":"Casual Friday"},"inform",["1.130.000đ"]),
 ("C16","Cho mình hỏi đơn #SP3177 sao rồi","order_check",{"ma_don":"#SP3177"},"inform",["Chờ xác nhận","03/11/2026"]),
 ("C17","đơn sp2740 của mình bị gì vậy","order_check",{"ma_don":"#SP2740"},"inform",["Giao thất bại","25/10/2026"]),
 ("C18","Check hộ mình đơn #SP2650","order_check",{"ma_don":"#SP2650"},"inform",["Đã hủy"]),
 ("C19","đơn #SP3220 đang đổi hàng đúng không","order_check",{"ma_don":"#SP3220"},"inform",["Chờ xử lý đổi hàng","VT774102"]),
 ("C20","Đơn 450k thì mình dùng được mã nào không?","voucher",{"cart_total":450000},"inform",["450.000đ","chưa đủ điều kiện"]),
 ("C21","Mã FLASHDEAL20 hôm nay còn dùng được không?","voucher",{"ma_voucher":"FLASHDEAL20"},"inform",["FLASHDEAL20","dùng được"]),
 ("C22","mình có mã SUPEOBIRTHDAY, áp cho đơn 600k được không","voucher",{"ma_voucher":"SUPEOBIRTHDAY","cart_total":600000},"inform",["SUPEOBIRTHDAY","chưa thấy"]),
 ("C23","Mình đang có 550k trong giỏ, dùng mã nào được nhỉ","voucher",{"cart_total":550000},"inform",["SUPEOVIP100"]),
 ("C24","Shop có nhận ship COD không?","chitchat",{},"chitchat",["chưa có thông tin"]),
]
SUITE_C_MULTI = {
 "CM1_nho_mon_doi_chu_de": [("Mình thích Chân váy Midi chữ A dập ly",["P29"],[]),
                            ("Tối thứ Bảy mình có tiệc cocktail, nên diện gì ta",["Tiệc cocktail"],[]),
                            ("Chân váy đó giá bao nhiêu vậy?",["530.000đ"],["P04"])],
 "CM2_nho_don": [("Đơn #SP3255 của mình đang ở đâu rồi nhỉ",["Đang giao","31/10/2026"],[]),
                 ("Đơn đó là của hãng vận chuyển nào vậy?",["Viettel Post"],[]),
                 ("Mã vận đơn là gì?",["VT992014"],[]),
                 ("À còn đơn #SP2908 thì sao?",["#SP2908","30/10/2026"],["VT992014"])],
 "CM3_nho_so_do": [("Chiều cao 1m58, cân nặng 46kg",["Size S"],[]),
                   ("Cho mình xem Đầm midi cổ vuông",["P11","Size S"],[]),
                   ("Còn chân váy midi xòe xếp tầng thì sao?",["P18","Size S"],["xin chiều cao"])],
 "CM4_bot_hoi_gio_hang": [("Shop đang có ưu đãi giảm giá gì vậy?",["SUPEOVIP100","giỏ hàng"],[]),
                          ("khoảng 850k",["850.000đ"],[]),
                          ("Còn nếu thêm nữa lên 1 triệu thì sao?",["1.000.000đ"],["850.000đ"])],
 "CM5_doi_y_nho_mon": [("Cho mình xem Áo Blazer kẻ caro vintage",["P13"],[]),
                       ("Nó có màu gì?",["Nâu tây","Xám ghi"],[]),
                       ("Thôi đổi sang Áo Blazer lụa chiết eo, giá sao nhỉ",["1.050.000đ"],["920.000đ"]),
                       ("Mẫu vừa rồi còn hàng không vậy",["P14"],["P13"])],
}


In [15]:
# ============ 9c'. KHUNG ĐO MỚI: DEV / TEST / FRESH ============
# Thay cell 9c cũ bằng cell này. Mỗi bộ gồm 2 biến cùng định dạng với HELDOUT_SINGLE / HELDOUT_MULTI (cell 9a):
#   single: [(id, câu, intent, {slot}, action, [chuỗi phải có]), ...]
#   multi : {"tên": [(câu, [phải có], [không được có]), ...], ...}
# Dán bộ A, B, C của bạn vào 3 cell riêng (SUITE_A_SINGLE/SUITE_A_MULTI, SUITE_B_..., SUITE_C_...) TRƯỚC cell này.
import os, json, time, contextlib
import pandas as pd

pd.set_option("display.width", 220, "display.max_colwidth", 140)
MEASURE_DATE = "2026-10-28"
LLM_AVAILABLE = bool(USE_LLM)      # True chỉ khi cell 'Bật LLM' đã kết nối Gemini thành công

@contextlib.contextmanager
def fixed_today(day=MEASURE_DATE):
    """Cố định ngày cho API (voucher có hạn), xong trả lại giá trị cũ. API đọc SUPEO_TODAY ở MỖI request nên có hiệu lực ngay."""
    old = os.environ.get("SUPEO_TODAY")
    os.environ["SUPEO_TODAY"] = day
    try:
        yield
    finally:
        if old is None: os.environ.pop("SUPEO_TODAY", None)
        else: os.environ["SUPEO_TODAY"] = old

@contextlib.contextmanager
def llm_mode(on):
    global USE_LLM
    old = USE_LLM
    USE_LLM = bool(on)
    try:
        yield
    finally:
        USE_LLM = old

def _g(name, default):
    return globals().get(name, default)

SUITES = {                                   # tên -> (single, multi, vai trò)
    "HELDOUT_CU (hồi quy)": (_g("HELDOUT_SINGLE", []), _g("HELDOUT_MULTI", {}), "hồi quy"),
    "A (DEV, hồi quy)":     (_g("SUITE_A_SINGLE", []), _g("SUITE_A_MULTI", {}), "hồi quy - đã dùng để chỉnh"),
    "B (TEST, hồi quy)":    (_g("SUITE_B_SINGLE", []), _g("SUITE_B_MULTI", {}), "hồi quy - đã xem lỗi"),
    "C (FRESH)":            (_g("SUITE_C_SINGLE", []), _g("SUITE_C_MULTI", {}), "khách quan - chạy 1 lần"),
}

def run_suite(single, multi):
    rows = []
    for sid, text, e_int, e_slots, e_act, must in single:
        st = new_state(); reply = bot_reply(text, st); tr = st["trace"][-1]
        got = {k: v for k, v in tr["slots"].items() if k in SLOT_KEYS}
        rows.append({"id": sid, "câu khách": text, "intent_gt": e_int, "intent_ok": tr["intent"] == e_int,
                     "slot_ok": got == e_slots, "action_ok": tr["action"] == e_act, "reply_ok": all(m in reply for m in must),
                     "dự_đoán": f"{tr['intent']}/{tr['action']} {got}", "kỳ_vọng": f"{e_int}/{e_act} {e_slots}"})
    mrows = []
    for sid, turns in multi.items():
        st = new_state()
        for k, (text, must, mustnot) in enumerate(turns, 1):
            reply = bot_reply(text, st)
            ok = all(m in reply for m in must) and not any(m in reply for m in mustnot)
            mrows.append({"conv": sid, "id": f"{sid}-T{k}", "ok": ok, "câu khách": text, "trả lời": reply.replace("\n", " ")[:170]})
    return pd.DataFrame(rows), pd.DataFrame(mrows)

def report(name, single, multi, mode):
    s, m = run_suite(single, multi)
    print("=" * 8, name, "|", mode, "|", f"ngày API {MEASURE_DATE}", "=" * 8)
    if len(s):
        s["all_ok"] = s.intent_ok & s.slot_ok & s.action_ok & s.reply_ok
        print(f"[1 lượt] {len(s)} câu | Intent {s.intent_ok.mean():.1%} | Slot (chính xác) {s.slot_ok.mean():.1%} | "
              f"Action {s.action_ok.mean():.1%} | Nội dung {s.reply_ok.mean():.1%} | ĐÚNG HẾT 4 tiêu chí {s.all_ok.mean():.1%}")
        by = s.groupby("intent_gt").agg(n=("id", "size"), intent=("intent_ok", "mean"), slot=("slot_ok", "mean"),
                                         action=("action_ok", "mean"), reply=("reply_ok", "mean"), all_ok=("all_ok", "mean"))
        fmt = lambda v: f"{v:.0%}" if isinstance(v, float) else v
        print((by.map(fmt) if hasattr(by, "map") else by.applymap(fmt)).to_string())
        bad = s[~s.all_ok]
        if len(bad):
            print("\nCâu chưa đạt:"); print(bad[["id", "câu khách", "dự_đoán", "kỳ_vọng"]].to_string(index=False))
    if len(m):
        by_c = m.groupby("conv").ok.all()
        print(f"\n[nhiều lượt] {len(m)} lượt | đúng từng lượt {m.ok.mean():.1%} | đúng CẢ hội thoại {by_c.mean():.1%} ({int(by_c.sum())}/{len(by_c)})")
        badm = m[~m.ok]
        if len(badm):
            print(badm[["id", "câu khách", "trả lời"]].to_string(index=False))
    return s, m

RESULTS = {}

def run_regression(use_llm=False):
    """Chạy các bộ ĐÃ dùng để chỉnh (hồi quy, không khách quan)."""
    if use_llm and not LLM_AVAILABLE:
        print("Bỏ qua hồi quy Gemini: chưa kết nối Gemini."); return
    with fixed_today(), llm_mode(use_llm):
        for name, (s, m, role) in SUITES.items():
            if name.startswith("C ") or not (s or m):
                continue
            RESULTS[(name, use_llm)] = report(name, s, m, (f"Gemini({_G['model']})+luật" if use_llm else "luật+regex") + " | " + role)

FRESH_LOCK = "fresh_C_ran.json"

def run_fresh_once(use_llm=False, force=False):
    """Chạy bộ C ĐÚNG 1 LẦN mỗi chế độ (luật / Gemini). Có file khóa: chạy lần 2 sẽ bị từ chối để không 'đẩy điểm'."""
    if use_llm and not LLM_AVAILABLE:
        raise RuntimeError("Chưa kết nối Gemini (cell 'Bật LLM' chưa có key hoặc lỗi kết nối) - KHÔNG chạy để khỏi ghi nhầm "
                           "kết quả luật thành 'Gemini'. Lượt chạy chưa bị tính.")
    s, m, _ = SUITES["C (FRESH)"]
    if not (s or m):
        print("Chưa có SUITE_C_SINGLE / SUITE_C_MULTI. Dán bộ C vào cell riêng phía trên rồi chạy lại."); return None
    mode = "gemini" if use_llm else "luat"
    lock = json.load(open(FRESH_LOCK)) if os.path.exists(FRESH_LOCK) else {}
    if mode in lock and not force:
        raise RuntimeError(f"Bộ C đã chạy ở chế độ '{mode}' lúc {lock[mode]['time']}. Không chạy lại để lấy điểm. "
                           f"Nếu đã sửa bot theo lỗi của C, hãy viết bộ D mới.")
    with fixed_today(), llm_mode(use_llm):
        res = report("C (FRESH)", s, m, (f"Gemini({_G['model']})+luật" if use_llm else "luật+regex") + " | khách quan, chạy 1 lần")
    RESULTS[("C (FRESH)", use_llm)] = res
    lock[mode] = {"time": time.strftime("%Y-%m-%d %H:%M:%S")}
    json.dump(lock, open(FRESH_LOCK, "w"))
    return res

# ---- audit_leak mở rộng: kiểm tra TẤT CẢ các bộ test với few-shot/demo/ví dụ và benchmark ----
def audit_leak(threshold=0.85, strict=True):
    b_utts = list(bench["user_utterance"])
    b_sents = [s for r in bench["ground_truth_bot_response"] for s in _sentences(r)]
    test_utts = {}
    for name, (s, m, _) in SUITES.items():
        test_utts[name] = [x[1] for x in s] + [t[0] for v in m.values() for t in v]
    user_src = {"FEWSHOT": [x[0] for x in globals().get("FEWSHOT", [])],
                "DEMOS": [u for conv in globals().get("DEMOS", []) for u in conv],
                "EXAMPLES (chip Gradio)": list(globals().get("EXAMPLES", []))}
    bot_src = {"FAQ_SHIPPING": FAQ_SHIPPING, "FAQ_THANKS": FAQ_THANKS, "HELP": HELP}
    hits = []
    for name, texts in {**user_src, **test_utts}.items():                       # dạy-bot & mọi bộ test đều không được trùng benchmark
        hits += [(name, t, u, r) for t in texts for u in b_utts if (r := _ratio(t, u)) >= threshold]
    for name, txt in bot_src.items():
        hits += [(name, s, g, r) for s in _sentences(txt) for g in b_sents if (r := _ratio(s, g)) >= threshold]
    pool = [t for v in user_src.values() for t in v]
    for name, texts in test_utts.items():                                        # test không được trùng few-shot/demo/ví dụ
        hits += [(f"{name} trùng few-shot/demo/ví dụ", t, d, r) for t in texts for d in pool if (r := _ratio(t, d)) >= threshold]
    names = list(test_utts)                                                      # các bộ test không được trùng nhau
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            hits += [(f"{names[i]} ~ {names[j]}", a, b, r) for a in test_utts[names[i]] for b in test_utts[names[j]]
                     if (r := _ratio(a, b)) >= threshold]
    print(f"Đã so {sum(len(v) for v in user_src.values())} câu dạy-bot + {sum(len(v) for v in test_utts.values())} câu test + "
          f"{len(bot_src)} đoạn trả lời mẫu với benchmark ({len(b_utts)} câu / {len(b_sents)} vế đáp án).")
    if hits:
        for name, a, b, r in hits:
            print(f"  RÒ RỈ [{name}] ({r:.0%}): {a[:70]!r}  ~  {b[:70]!r}")
        if strict:
            raise AssertionError(f"Phát hiện {len(hits)} chỗ trùng - viết lại các câu đó.")
    else:
        print("OK: audit_leak pass.")
    return hits

_ = audit_leak()
run_regression(use_llm=False)                  # hồi quy chế độ luật + regex
run_regression(use_llm=True)                   # hồi quy Gemini (tự bỏ qua nếu chưa kết nối Gemini)
# ---- BỘ C (FRESH): bỏ comment ĐÚNG 1 LẦN mỗi dòng, sau khi mọi việc sửa xong ----
# run_fresh_once(use_llm=True)                 # Gemini + luật  (cần Gemini đã kết nối)
run_fresh_once(use_llm=False)                  # luật + regex   (khách quan, chạy 1 lần)


Đã so 16 câu dạy-bot + 190 câu test + 3 đoạn trả lời mẫu với benchmark (20 câu / 36 vế đáp án).
OK: audit_leak pass.
======== HELDOUT_CU (hồi quy) | luật+regex | hồi quy | ngày API 2026-10-28 ========
[1 lượt] 28 câu | Intent 100.0% | Slot (chính xác) 100.0% | Action 100.0% | Nội dung 100.0% | ĐÚNG HẾT 4 tiêu chí 100.0%
              n intent  slot action reply all_ok
intent_gt                                       
chitchat      5   100%  100%   100%  100%   100%
lookbook      5   100%  100%   100%  100%   100%
order_check   4   100%  100%   100%  100%   100%
product_info  5   100%  100%   100%  100%   100%
size_advice   4   100%  100%   100%  100%   100%
voucher       5   100%  100%   100%  100%   100%

[nhiều lượt] 28 lượt | đúng từng lượt 100.0% | đúng CẢ hội thoại 100.0% (12/12)
======== A (DEV, hồi quy) | luật+regex | hồi quy - đã dùng để chỉnh | ngày API 2026-10-28 ========
[1 lượt] 30 câu | Intent 100.0% | Slot (chính xác) 100.0% | Action 100.0% | Nội dung 100.0% | ĐÚNG HẾT 4 t

In [ ]:
# ============ BỘ D (HARD REAL-WORLD BLIND TEST - 30 câu đơn + 6 hội thoại nhiều lượt) ============
# Kiểm thử thực tế nhiễu tự nhiên (viết tắt "sz", "ko", số đo "1.62m", "m62", hỏi ghép 2 ý, từ vựng ngoài tập luật).
# Tuyệt đối KHÔNG chỉnh sửa code của chatbot theo bộ này.

SUITE_D_SINGLE = [
    # 1. Product Info (10 câu: gồm câu chuẩn, viết tắt "sz M" + phủ định "ko lấy màu đen", và hỏi ghép 2 ý giá + phí ship)
    ("D01", "Mẫu áo blazer lụa chiết eo hiện đang bán giá bao nhiêu tiền ạ?", "product_info", {"ten_sp": "Áo Blazer lụa chiết eo", "loai_do": "Áo Blazer"}, "inform", ["1.050.000đ"]),
    ("D02", "Mẫu chân váy kaki xẻ tà bên shop có mấy màu tất cả?", "product_info", {"ten_sp": "Chân váy Midi kaki xẻ tà", "loai_do": "Chân váy Midi"}, "inform", ["Be đậm", "Rêu"]),
    ("D03", "Chiếc đầm midi cổ vuông được may từ chất vải gì thế bạn?", "product_info", {"ten_sp": "Đầm midi cổ vuông", "loai_do": "Đầm"}, "inform", ["voan lót lụa"]),
    ("D04", "Hiện tại áo sơ mi cổ nơ ruy băng trong kho còn hàng không?", "product_info", {"ten_sp": "Áo sơ mi cổ nơ ruy băng", "loai_do": "Áo sơ mi"}, "inform", ["P19", "còn hàng"]),
    ("D05", "Shop ơi có mẫu áo blazer nào may bằng vải tweed không?", "product_info", {"loai_do": "Áo Blazer", "tu_khoa": "tweed", "ten_sp": "Áo Blazer suông cổ điển"}, "inform", ["P01"]),
    ("D06", "Kiếm giùm mình mẫu chân váy midi tông màu nâu hạt dẻ nha", "product_info", {"loai_do": "Chân váy Midi", "tu_khoa": "nâu hạt dẻ", "ten_sp": "Chân váy Midi chữ A dập ly"}, "inform", ["P29"]),
    ("D07", "Lấy cho mình xem các mẫu quần ống rộng chứ đừng lấy chân váy nha", "product_info", {"loai_do": "Quần ống rộng"}, "inform", ["P08", "P09"]),
    ("D08", "Kiểm tra giùm mình đầm voan hoa tay tiên còn size M không nhé", "product_info", {"ten_sp": "Đầm voan hoa tay tiên", "loai_do": "Đầm"}, "inform", ["size M"]),
    ("D09", "Shop tìm giúp mình đầm sz M chứ ko lấy màu đen nha", "product_info", {"loai_do": "Đầm"}, "inform", ["P11", "P25"]),
    ("D10", "Báo giá giúp mình đầm slip satin luôn với tiền ship về Đà Nẵng là bao nhiêu?", "product_info", {"ten_sp": "Đầm slip satin", "loai_do": "Đầm"}, "inform", ["1.150.000đ"]),

    # 2. Size Advice (5 câu: gồm chuẩn, lệch dải Max-Boundary, thiếu slot, và nhiễu gõ số thập phân "1.62m")
    ("D11", "Số đo của em là cao 1m61 và nặng 51kg thì mặc size mấy vừa?", "size_advice", {"chieu_cao": 161, "can_nang": 51}, "inform", ["Size M"]),
    ("D12", "Em cao có 1m56 nhưng nặng tới 59kg thì chọn size nào mặc thoải mái?", "size_advice", {"chieu_cao": 156, "can_nang": 59}, "inform", ["Size L"]),
    ("D13", "Chị cao 1m67 nặng 54kg muốn đặt Chân váy Midi xòe xếp tầng thì lấy size gì?", "size_advice", {"chieu_cao": 167, "can_nang": 54, "ten_sp": "Chân váy Midi xòe xếp tầng", "loai_do": "Chân váy Midi"}, "inform", ["P18", "Size M"]),
    ("D14", "Mình nặng tầm 50kg thì mặc quần ống rộng bên shop size nào vừa?", "size_advice", {"can_nang": 50, "loai_do": "Quần ống rộng"}, "ask", ["chiều cao"]),
    ("D15", "Bình thường mình cao 1.62m nặng 53kg thì mặc vừa size gì?", "size_advice", {"chieu_cao": 162, "can_nang": 53}, "inform", ["Size M"]),

    # 3. Lookbook (5 câu: gồm chuẩn và từ đồng nghĩa ngữ cảnh "đi biển Nha Trang" thay vì "du lịch hè")
    ("D16", "Thứ sáu mặc theo kiểu casual friday thì phối set nào đẹp?", "lookbook", {"dip_mac": "Casual Friday"}, "inform", ["P21", "P08", "1.130.000đ"]),
    ("D17", "Tối mai mình tham gia buổi tiệc tối, gợi ý set đồ sang trọng giúp mình", "lookbook", {"dip_mac": "Tiệc tối"}, "inform", ["P10", "P02", "2.200.000đ"]),
    ("D18", "Chiều chủ nhật hẹn hội bạn đi uống trà chiều thì diện bộ nào hợp?", "lookbook", {"dip_mac": "Trà chiều"}, "inform", ["P06", "P08", "1.340.000đ"]),
    ("D19", "Shop tư vấn phối đồ theo set giúp mình nhé", "lookbook", {}, "ask", ["dịp nào"]),
    ("D20", "Sắp tới mình đi biển Nha Trang chơi, gợi ý set đồ giúp mình nhé", "lookbook", {"dip_mac": "Du lịch hè"}, "inform", ["Du lịch hè"]),

    # 4. Order Check (4 câu)
    ("D21", "Nhờ shop kiểm tra đơn #SP1001 xem đã giao thành công chưa", "order_check", {"ma_don": "#SP1001"}, "inform", ["Đã giao thành công", "GHN77105"]),
    ("D22", "Mã đơn sp3260 của mình gửi qua đơn vị vận chuyển nào thế?", "order_check", {"ma_don": "#SP3260"}, "inform", ["SPX Express", "03/11/2026"]),
    ("D23", "Check giùm em xem đơn hàng #SP8888 hiện đang ở đâu rồi ạ", "order_check", {"ma_don": "#SP8888"}, "inform", ["chưa tìm thấy"]),
    ("D24", "Kiểm tra giùm mình đơn hàng mới đặt hôm qua khi nào giao tới nơi", "order_check", {}, "ask", ["#SPxxxx"]),

    # 5. Voucher (3 câu)
    ("D25", "Tổng đơn trong giỏ của mình là 650k thì áp mã nào giảm nhiều nhất?", "voucher", {"cart_total": 650000}, "inform", ["FLASHDEAL20", "130.000đ"]),
    ("D26", "Nếu áp mã FREESHIPMAX cho giỏ hàng 850k thì được giảm bao nhiêu tiền?", "voucher", {"ma_voucher": "FREESHIPMAX", "cart_total": 850000}, "inform", ["FREESHIPMAX", "40.000đ"]),
    ("D27", "Cho mình hỏi thăm mã ưu đãi WELCOME10 hiện còn xài được không?", "voucher", {"ma_voucher": "WELCOME10"}, "inform", ["WELCOME10", "chưa thấy"]),

    # 6. Chitchat / FAQ (3 câu)
    ("D28", "Bên shop giao hàng toàn quốc thường mất bao lâu?", "chitchat", {}, "chitchat", ["giao hàng"]),
    ("D29", "Cảm ơn bạn nhân viên SUPEO đã tư vấn rất nhiệt tình nha", "chitchat", {}, "chitchat", ["cảm ơn"]),
    ("D30", "Alo shop ơi cho mình hỏi thăm chút xíu nhé", "chitchat", {}, "chitchat", ["SUPEO chào"]),
]

SUITE_D_MULTI = {
    "DM1_nho_mon_va_size": [
        ("Gửi cho mình xem thông tin mẫu Đầm sơ mi thắt đai eo", ["P25", "890.000đ"], []),
        ("Mẫu vừa rồi may bằng chất vải gì thế?", ["cotton poplin"], []),
        ("Nếu mình cao 1m63 và nặng 52kg thì lấy size nào của mẫu này?", ["P25", "Size M"], []),
    ],
    "DM2_chon_danh_sach_blazer": [
        ("Liệt kê giúp mình các mẫu áo blazer hiện có ở shop nhé", ["P01", "P02", "P03"], []),
        ("Báo chi tiết cho mình mẫu số 3 trong danh sách trên", ["P03", "820.000đ"], ["P01"]),
        ("Chiếc áo đó có những màu sắc nào?", ["Be", "Xanh nhạt"], []),
    ],
    "DM3_don_hang_da_luot": [
        ("Nhờ shop tra cứu tình trạng đơn #SP3220 giùm mình", ["Chờ xử lý đổi hàng", "VT774102"], []),
        ("Vậy đơn này dự kiến ngày nào hoàn tất giao hàng?", ["01/11/2026"], []),
        ("Kiểm tra thêm giùm mình mã đơn #SP2890 luôn nha", ["Đã giao thành công", "SPX99310"], ["VT774102"]),
    ],
    "DM4_phoi_do_theo_mon": [
        ("Báo giá cho mình mẫu Chân váy Midi xếp ly nhé", ["P04", "490.000đ"], []),
        ("Số đo hiện tại của mình là cao 1m60 và nặng 50kg", ["Size M"], []),
        ("Chiếc chân váy này kết hợp cùng mẫu áo nào thì hợp?", ["P06", "Size M"], ["xin chiều cao"]),
    ],
    "DM5_voucher_nang_gio_hang": [
        ("Shop liệt kê giúp mình các chương trình khuyến mãi đang mở nhé", ["SUPEOVIP100", "FLASHDEAL20"], []),
        ("Hiện tại giỏ hàng của mình mới có 400k thôi", ["400.000đ", "chưa đủ điều kiện"], []),
        ("Vậy nếu mình chọn thêm đồ lên 650k thì mã nào lợi nhất?", ["FLASHDEAL20", "130.000đ"], []),
    ],
    "DM6_so_do_viet_tat_m62": [
        ("Shop ơi tư vấn chọn size đầm giúp mình với", ["chiều cao", "cân nặng"], []),
        ("em cao m62 nặng 50 cân nha shop", ["Size M"], []),
        ("Vậy mẫu Đầm ôm body cổ yếm mình mặc size M còn hàng không?", ["P26", "Size M"], []),
    ],
}

# Đăng ký thêm Bộ D vào SUITES và chạy kiểm tra rò rỉ + đánh giá:
SUITES["D (HARD BLIND TEST)"] = (SUITE_D_SINGLE, SUITE_D_MULTI, "kiểm thử thực tế nhiễu - không sửa code")
_ = audit_leak(threshold=0.85, strict=True)
with fixed_today(), llm_mode(False):
    RESULTS[("D (HARD BLIND TEST)", False)] = report("D (HARD BLIND TEST)", SUITE_D_SINGLE, SUITE_D_MULTI, "luật+regex v5/v6 | kiểm thử thực tế nhiễu")


In [16]:
# ============ 9d. DEV SMOKE-TEST trên sheet dialogue_benchmarks (20 lượt) ============
# CHÚ Ý: sheet này đã được nhìn thấy khi xây luật/regex => KHÔNG phải số đo khách quan, chỉ để kiểm tra bot không hỏng.
# Con số báo cáo: dùng bước 9c (held-out). Slot ở đây chấm kiểu 'đủ slot kỳ vọng' vì nhãn benchmark không liệt kê hết slot.
import pandas as pd

_old_today = os.environ.get("SUPEO_TODAY"); os.environ["SUPEO_TODAY"] = "2026-10-28"   # cố định ngày khi đo, trả lại ở cuối cell

raw = pd.read_excel(os.environ["SUPEO_XLSX"], sheet_name="dialogue_benchmarks", header=None, dtype=object)
hdr = next(i for i in range(len(raw)) if str(raw.iloc[i, 0]).strip() == "session_id")
bench = raw.iloc[hdr + 1:].copy(); bench.columns = [str(c).strip() for c in raw.iloc[hdr]]
bench = bench[bench["session_id"].notna()].reset_index(drop=True)

INTENT_MAP = {"product_inquiry": "product_info", "size_consulting": "size_advice", "lookbook_recommendation": "lookbook",
              "order_tracking": "order_check", "voucher_lookup": "voucher", "chit_chat": "chitchat"}
SLOT_MAP = {"product_name": "ten_sp", "category": "loai_do", "height_cm": "chieu_cao", "weight_kg": "can_nang",
            "occasion": "dip_mac", "order_id": "ma_don", "voucher_code": "ma_voucher", "cart_value": "cart_total",
            "query_field": "truong"}
def action_kind(a):
    a = str(a)
    if a.startswith("REQUEST_SLOT"): return "ask"
    if a in ("POLICY_SHIPPING_INFO", "POLICY_THANK_YOU"): return "chitchat"
    return "inform"

FACT_RE = r"\d{1,3}(?:\.\d{3})+đ|#SP\d+|\b[A-Z]{2,4}\d{4,}\b|\d{2}/\d{2}/\d{4}|\bP\d{2}\b"

rows, states = [], {}
for _, r in bench.iterrows():
    st = states.setdefault(r["session_id"], new_state())
    reply = bot_reply(r["user_utterance"], st)
    tr = st["trace"][-1]
    exp_slots = {SLOT_MAP[k]: v for k, v in json.loads(r["dst_extracted_slots_json"]).items() if v is not None and k in SLOT_MAP}
    slot_ok = all(tr["slots"].get(k) == v for k, v in exp_slots.items())
    facts = set(re.findall(FACT_RE, str(r["ground_truth_bot_response"])))
    recall = (len([f for f in facts if f in reply]) / len(facts)) if facts else None
    rows.append({"phiên": f"{r['session_id']}-{r['turn_id']}", "intent_ok": tr["intent"] == INTENT_MAP[r["predicted_intent"]],
                 "slot_ok": slot_ok, "action_ok": tr["action"] == action_kind(r["system_action"]), "fact_recall": recall,
                 "câu khách": r["user_utterance"][:55], "intent_dự_đoán": tr["intent"], "slots_dự_đoán": tr["slots"],
                 "slots_kỳ_vọng": exp_slots})
res = pd.DataFrame(rows)
print(f"[DEV smoke-test - KHÔNG báo cáo] Số lượt: {len(res)}")
print(f"Intent accuracy : {res.intent_ok.mean():.1%}")
print(f"Slot accuracy   : {res.slot_ok.mean():.1%}")
print(f"Action accuracy : {res.action_ok.mean():.1%}")
print(f"Fact recall     : {res.fact_recall.dropna().mean():.1%}  (số liệu/mã/giá trong đáp án mẫu xuất hiện trong câu trả lời, {res.fact_recall.notna().sum()} lượt có số liệu)")
bad = res[~(res.intent_ok & res.slot_ok & res.action_ok)]
if len(bad):
    print("\nCác lượt chưa khớp:")
    print(bad[["phiên", "câu khách", "intent_ok", "slot_ok", "action_ok", "slots_dự_đoán", "slots_kỳ_vọng"]].to_string(index=False))
if _old_today is None: os.environ.pop("SUPEO_TODAY", None)
else: os.environ["SUPEO_TODAY"] = _old_today
res


[DEV smoke-test - KHÔNG báo cáo] Số lượt: 20
Intent accuracy : 100.0%
Slot accuracy   : 100.0%
Action accuracy : 100.0%
Fact recall     : 100.0%  (số liệu/mã/giá trong đáp án mẫu xuất hiện trong câu trả lời, 11 lượt có số liệu)


,phiên,intent_ok,slot_ok,action_ok,fact_recall,câu khách,intent_dự_đoán,slots_dự_đoán,slots_kỳ_vọng
0,S01-T01,True,True,True,1.0,Áo blazer suông cổ điển bên bạn giá bao nhiêu vậy?,product_info,"{'loai_do': 'Áo Blazer', 'ten_sp': 'Áo Blazer suông cổ điển', 'truong': 'gia'}",{'ten_sp': 'Áo Blazer suông cổ điển'}
1,S01-T02,True,True,True,NaN,Áo này có những size nào shop?,product_info,"{'loai_do': 'Áo Blazer', 'ten_sp': 'Áo Blazer suông cổ điển', 'truong': 'size'}","{'ten_sp': 'Áo Blazer suông cổ điển', 'truong': 'size'}"
2,S02-T01,True,True,True,NaN,Mình cao 1m65 thì mặc size gì vừa?,size_advice,{'chieu_cao': 165},{'chieu_cao': 165}
3,S02-T02,True,True,True,NaN,Mình nặng 52kg nhé,size_advice,"{'chieu_cao': 165, 'can_nang': 52}","{'chieu_cao': 165, 'can_nang': 52}"
4,S03-T01,True,True,True,NaN,Mình cao 1m62 nặng 59kg thì mặc blazer lửng size gì?,size_advice,"{'loai_do': 'Áo Blazer', 'ten_sp': 'Áo Blazer dáng lửng', 'chieu_cao': 162, 'can_nang': 59}","{'ten_sp': 'Áo Blazer dáng lửng', 'chieu_cao': 162, 'can_nang': 59}"
5,S04-T01,True,True,True,1.0,"Chiều nay mình đi trà chiều với bạn thân, gợi ý đồ giúp",lookbook,{'dip_mac': 'Trà chiều'},{'dip_mac': 'Trà chiều'}
6,S05-T01,True,True,True,1.0,Shop ơi check hộ mình đơn hàng #SP2908 với,order_check,{'ma_don': '#SP2908'},{'ma_don': '#SP2908'}
7,S06-T01,True,True,True,NaN,Hiện shop có mã giảm giá nào không?,voucher,{},{}
8,S07-T01,True,True,True,NaN,Giỏ hàng của mình đang 750k thì áp được mã nào shop?,voucher,{'cart_total': 750000},{'cart_total': 750000}
9,S08-T01,True,True,True,1.0,Mã đơn #SP9999 có chưa bạn?,order_check,{'ma_don': '#SP9999'},{'ma_don': '#SP9999'}


## Bước 9e - Kiểm tra bộ nhớ hội thoại
Hồi quy tự viết (không phải số khách quan) + ca LLM giả trả sai.

In [17]:
# ============ 9e. KIỂM TRA BỘ NHỚ HỘI THOẠI (hồi quy tự viết, KHÔNG phải số báo cáo khách quan) ============
# Mỗi lượt: (câu khách, {slot phải có}, [chuỗi phải có trong trả lời], [chuỗi KHÔNG được có])
MEM_TESTS = {
 "nho_mon_qua_doi_chu_de": [("Cho mình xem Đầm babydoll hoa nhí", {"ten_sp": "Đầm babydoll hoa nhí"}, ["P27"], []),
                            ("Mai đi tiệc tối mặc gì nhỉ?", {"dip_mac": "Tiệc tối"}, ["Tiệc tối"], []),
                            ("Đầm này giá bao nhiêu?", {"ten_sp": "Đầm babydoll hoa nhí"}, ["750.000đ"], []),
                            ("Nó có màu gì vậy", {"ten_sp": "Đầm babydoll hoa nhí"}, ["Vàng nhạt"], ["P10"])],
 "nho_don_hoi_tiep": [("Check đơn #SP3201 giúp mình", {"ma_don": "#SP3201"}, ["29/10/2026"], []),
                      ("Bao lâu nữa thì tới nơi?", {"ma_don": "#SP3201"}, ["#SP3201"], []),
                      ("Mình cao 1m60 nặng 49kg mặc size gì?", {"chieu_cao": 160}, ["Size M"], []),
                      ("Đơn đó giao chưa vậy?", {"ma_don": "#SP3201"}, ["#SP3201"], [])],
 "nho_so_do_dung_lai": [("Mình cao 1m66 nặng 57kg", {"chieu_cao": 166, "can_nang": 57}, ["Size"], []),
                        ("Cho mình xem áo sơ mi trắng cổ đức", {"ten_sp": "Áo sơ mi trắng cổ đức"}, ["1m66".replace("1m66", "166cm")], ["xin chiều cao"]),
                        ("Size nào hợp với mình?", {"chieu_cao": 166, "can_nang": 57}, ["Size"], ["xin thêm chiều cao"])],
 "doi_y_cap_nhat": [("Mình cao 1m60 nặng 49kg", {"chieu_cao": 160, "can_nang": 49}, ["Size M"], []),
                    ("À nhầm, mình cao 1m72 nặng 70kg", {"chieu_cao": 172, "can_nang": 70}, ["Size"], ["160cm"]),
                    ("Check đơn #SP3150", {"ma_don": "#SP3150"}, ["#SP3150"], []),
                    ("Còn đơn #SP3201?", {"ma_don": "#SP3201"}, ["#SP3201"], ["#SP3150 ("])],
 "doi_mon_moi": [("Áo Blazer cộc tay vạt chéo giá bao nhiêu", {"ten_sp": "Áo Blazer cộc tay vạt chéo"}, ["780.000đ"], []),
                 ("Còn Đầm slip satin thì sao, giá bao nhiêu?", {"ten_sp": "Đầm slip satin"}, ["1.150.000đ"], ["780.000đ"]),
                 ("Còn hàng không?", {"ten_sp": "Đầm slip satin"}, ["Đầm slip satin"], ["Blazer cộc tay"])],
 "khong_hoi_lai_so_do_o_lookbook": [("Mình cao 1m64 nặng 55kg", {"chieu_cao": 164, "can_nang": 55}, ["Size M"], []),
                                    ("tiệc tối", {"dip_mac": "Tiệc tối", "chieu_cao": 164}, ["P10", "P02", "Theo số đo nàng đã chia sẻ (164cm, 55kg)", "có size M"],
                                     ["xin chiều cao", "cho SUPEO xin"])],
 "phoi_voi_mon_dang_xem": [("Cho mình xem Đầm suông dự tiệc", {"ten_sp": "Đầm suông dự tiệc"}, ["P10"], []),
                           ("Mình cao 1m64 nặng 55kg", {"chieu_cao": 164, "can_nang": 55}, ["Size M"], []),
                           ("Đầm này phối với áo nào được?", {"ten_sp": "Đầm suông dự tiệc"},
                            ["P02", "Sang trọng tối giản", "Size M"], ["xin chiều cao", "dịp nào"])],
 "phoi_khi_chua_co_so_do": [("Cho mình xem Áo sơ mi trắng cổ đức", {"ten_sp": "Áo sơ mi trắng cổ đức"}, ["P07"], []),
                            ("Phối với quần nào thì hợp?", {"ten_sp": "Áo sơ mi trắng cổ đức"}, ["Quần"], ["dịp nào"])],
 "danh_sach_ghi_chu_size": [("Mình cao 1m64 nặng 55kg", {"chieu_cao": 164, "can_nang": 55}, ["Size M"], []),
                            ("Có những đầm nào vậy", {"loai_do": "Đầm"}, ["P10", "có size M"], ["cho SUPEO biết chiều cao"])],
 "ten_mon_chua_cong_so": [("Quần suông công sở có những size nào?", {"ten_sp": "Quần suông công sở"}, ["S, M, L, XL"], [])],
}

def run_mem_tests(verbose=True):
    ok_all, n = 0, 0
    for name, turns in MEM_TESTS.items():
        st, ok_conv = new_state(), True
        for k, (text, slots, must, mustnot) in enumerate(turns, 1):
            reply = bot_reply(text, st); tr = st["trace"][-1]
            slot_ok = all(tr["slots"].get(a) == b for a, b in slots.items())
            ok = slot_ok and all(m in reply for m in must) and not any(m in reply for m in mustnot)
            ok_conv &= ok
            if not ok and verbose:
                print(f"  FAIL {name}-T{k}: {text!r}\n     slots={ {a: v for a, v in tr['slots'].items() if a != '_list'} } | trả lời={reply[:160]!r}")
        n += 1; ok_all += ok_conv
    print(f"[BỘ NHỚ] đúng cả hội thoại: {ok_all}/{n}")
    return ok_all == n

with fixed_today():
    with llm_mode(False):
        run_mem_tests()
    # ---- LLM GIẢ trả sai hoàn toàn: luật vẫn phải giữ ngữ cảnh, và prompt phải có lịch sử + "đang nói về" ----
    _seen = []
    def _fake_llm(msgs, n=100):
        if msgs[0]["content"].startswith("Bạn là bộ trích xuất"):
            _seen.append(msgs[-1]["content"])
            return '{"intent":"chitchat","slots":{"loai_do":"Quần ống rộng","dip_mac":null,"chieu_cao":null,"can_nang":null,"ma_don":null}}'
        return ""                                                # NLG trả rỗng -> dùng nguyên dữ liệu thật
    _old_llm = globals().get("llm")
    llm = _fake_llm
    try:
        with llm_mode(True):
            print("Chạy với LLM GIẢ (sai hết):", end=" ")
            run_mem_tests()
    finally:
        if _old_llm is not None: llm = _old_llm
        else: del llm
    # ---- Độ trễ: Gemini chỉ được gọi khi luật không hiểu câu (không gọi cho câu rõ ràng / FAQ) ----
    _seen.clear()
    llm = _fake_llm
    try:
        with llm_mode(True):
            st = new_state()
            for q in ["Mình cao 1m64 nặng 55kg", "tiệc tối", "Cho mình xem Đầm suông dự tiệc", "Đầm này phối với áo nào được?", "xin chào", "cảm ơn shop"]:
                bot_reply(q, st)
            n_clear = len(_seen)
            bot_reply("zxcv qwerty lorem ipsum", st)
            n_vague = len(_seen) - n_clear
        print(f"Số lần gọi DST Gemini: {n_clear} cho 6 câu rõ ràng (kỳ vọng 0) | {n_vague} cho câu luật không hiểu (kỳ vọng 1) | NLG Gemini: {'bật' if LLM_NLG else 'tắt'}")
    finally:
        if _old_llm is not None: llm = _old_llm
        else: del llm
    _seen.append("") if False else None
    last = next((s for s in reversed(_seen) if "Lịch sử gần nhất:\nKhách:" in s), "")
    print("Prompt DST có lịch sử:", "Lịch sử gần nhất:\nKhách:" in last, "| có 'Đang nói về' + món:", "Đang nói về: sản phẩm:" in "\n".join(_seen))
    st = new_state()
    for _ in range(30): bot_reply("xin chào", st)
    print("Giới hạn lịch sử:", len(st["history"]) // 2, "lượt (<= 20)")


[BỘ NHỚ] đúng cả hội thoại: 10/10
Chạy với LLM GIẢ (sai hết): [BỘ NHỚ] đúng cả hội thoại: 10/10
Số lần gọi DST Gemini: 0 cho 6 câu rõ ràng (kỳ vọng 0) | 1 cho câu luật không hiểu (kỳ vọng 1) | NLG Gemini: tắt
Prompt DST có lịch sử: True | có 'Đang nói về' + món: True
Giới hạn lịch sử: 20 lượt (<= 20)


## Bước 10 - Giao diện Gradio
Có panel bên phải hiển thị DST → Policy → API đã gọi của từng lượt, tiện để giải thích khi demo.

In [18]:
import gradio as gr
import os, re

GREETING = ("Xin chào! Mình là stylist AI của SUPEO. Mình có thể gợi ý phối đồ theo dịp, "
            "tư vấn size, cập nhật đơn hàng và mã giảm giá cho nàng.")
EXAMPLES = ["Quần ống suông cạp chun sau có màu gì vậy?", "Mình cao 1m69, 58kg nên lấy size gì ạ?",
            "Đi dự tiệc ngoài trời nên phối đồ thế nào?", "Cho mình tra tình trạng đơn #SP3220 với",
            "Đơn 1tr thì dùng mã giảm giá nào được?", "Mình đang tìm váy"]

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Playfair+Display:wght@500&family=Be+Vietnam+Pro:wght@400;500;600&display=swap');
body, .gradio-container { background:#e2e2e2 !important; font-family:'Be Vietnam Pro',system-ui,sans-serif !important; }
footer { display:none !important; }
#app { max-width:480px; margin:12px auto !important; background:#fff; border-radius:28px; padding:0 !important;
       overflow:hidden; gap:0 !important; border:none !important; box-shadow:none !important; }
#app > .block, #app .form { border:none !important; box-shadow:none !important; background:transparent !important; }
.hd { padding:16px 22px; border-bottom:1px solid #ececec; }
.hd .logo { font-family:'Playfair Display',Georgia,serif; font-size:28px; letter-spacing:.14em; line-height:1.1; color:#111; }
.hd .sub { font-size:11px; letter-spacing:.2em; color:#777; margin-top:2px; }
#exbox { padding:10px 14px 4px !important; background:transparent !important; border:none !important; gap:0 !important; }
#exlabel { font-size:12px; color:#777; margin:0 0 6px; }
#chips { flex-wrap:wrap !important; gap:6px !important; }
#chips button { flex:0 0 auto !important; min-width:0 !important; width:auto !important; background:#fff !important; color:#111 !important;
       border:1px solid #ddd !important; border-radius:16px !important; padding:6px 12px !important; font-size:13px !important;
       font-weight:400 !important; box-shadow:none !important; }
#chips button:hover { border-color:#111 !important; }
#chat, #chat .bubble-wrap { background:#fff !important; border:none !important; }
#chat .message-row { padding:0 10px; }
#chat .message { font-size:15px !important; line-height:1.55; border-radius:20px !important; box-shadow:none !important; }
#chat .bot, #chat [data-testid="bot"] { background:#f4f4f4 !important; border:none !important; color:#111 !important; }
#chat .user, #chat [data-testid="user"] { background:#000 !important; border:none !important; color:#fff !important;
       font-weight:600; border-radius:20px 20px 4px 20px !important; }
#chat .user *, #chat [data-testid="user"] * { color:#fff !important; }
#chat .message ul, #chat .message ol { margin:4px 0 4px 18px; }
#chat .message-buttons, #chat .avatar-container { display:none !important; }
#inrow { padding:6px 14px 0; gap:8px !important; flex-wrap:nowrap !important; align-items:center; }
#box textarea, #box input { border:1px solid #ddd !important; border-radius:22px !important; padding:11px 16px !important;
       font-size:15px !important; box-shadow:none !important; background:#fff !important; }
#box { border:none !important; background:transparent !important; box-shadow:none !important; }
#send { background:#000 !important; color:#fff !important; border:none !important; border-radius:22px !important;
        font-weight:600; min-width:72px !important; max-width:90px; }
#new { background:#fff !important; color:#111 !important; border:1px solid #ddd !important; border-radius:22px !important;
       margin:8px 14px 14px; width:calc(100% - 28px); }
"""

def render(hist):
    """State giữ danh sách tin nhắn dạng đơn giản; trả về nguyên danh sách để hiển thị."""
    return list(hist)

def respond(msg, state):
    msg = (msg or "").strip()
    if callable(state):                                  # phòng khi trang chưa kịp khởi tạo state
        state = make_state()
    hist = state.setdefault("hist", [{"role": "assistant", "content": GREETING}])
    if msg:
        reply = bot_reply(msg, state)
        hist += [{"role": "user", "content": msg}, {"role": "assistant", "content": reply}]
    show_examples = not any(m["role"] == "user" for m in hist)
    return "", render(hist), state, gr.update(visible=show_examples)

def reset():
    st = new_state()
    st["hist"] = [{"role": "assistant", "content": GREETING}]
    return "", render(st["hist"]), st, gr.update(visible=True)

def make_state():
    st = new_state()
    st["hist"] = [{"role": "assistant", "content": GREETING}]
    return st

theme = gr.themes.Base(font=["Be Vietnam Pro", "system-ui", "sans-serif"])

with gr.Blocks(title="SUPEO - AI Stylist", css=CSS, theme=theme) as demo:
    state = gr.State(make_state)
    with gr.Column(elem_id="app"):
        gr.HTML('<div class="hd"><div class="logo">SUPEO</div><div class="sub">AI STYLIST</div></div>')
        with gr.Column(elem_id="exbox") as exbox:                       # câu hỏi mẫu: nằm trên đầu, biến mất sau câu hỏi đầu tiên
            gr.HTML('<div id="exlabel">Câu hỏi mẫu</div>')
            with gr.Row(elem_id="chips"):
                chip_btns = [gr.Button(e, size="sm") for e in EXAMPLES]
        chat = gr.Chatbot(value=render([{"role": "assistant", "content": GREETING}]), type="messages", height=460,
                          show_label=False, container=False, elem_id="chat")
        with gr.Row(elem_id="inrow"):
            box = gr.Textbox(placeholder="Nhập câu hỏi rồi nhấn Enter...", show_label=False, container=False,
                             lines=1, max_lines=1, scale=6, elem_id="box")
            send = gr.Button("Gửi", variant="primary", scale=1, elem_id="send")
        clear = gr.Button("Cuộc trò chuyện mới", elem_id="new")
    outs = [box, chat, state, exbox]
    box.submit(respond, [box, state], outs, show_progress="hidden")
    send.click(respond, [box, state], outs, show_progress="hidden")
    for b, e in zip(chip_btns, EXAMPLES):
        b.click(lambda st, e=e: respond(e, st), [state], outs, api_name=False, show_progress="hidden")
    clear.click(reset, None, outs, show_progress="hidden")

def cloudflare_link(port):
    """Dự phòng khi Gradio không tạo được link share: mở tunnel Cloudflare tới cổng local (cần bật Internet trên Kaggle)."""
    import subprocess, time, urllib.request, stat
    exe = "/kaggle/working/cloudflared" if os.path.isdir("/kaggle/working") else "./cloudflared"
    if not os.path.exists(exe):
        urllib.request.urlretrieve("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64", exe)
        os.chmod(exe, os.stat(exe).st_mode | stat.S_IEXEC)
    log = open("cloudflared.log", "w")
    subprocess.Popen([exe, "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{port}"], stdout=log, stderr=log)
    for _ in range(60):
        time.sleep(1)
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cloudflared.log").read())
        if m:
            return m.group(0)
    return None

LAUNCH = globals().get("LAUNCH", True)
if LAUNCH:
    SHARE = os.environ.get("SUPEO_SHARE", "1") != "0"                    # SUPEO_SHARE=0: chỉ chạy local (http://127.0.0.1:7860), không tạo link công khai
    _, local_url, share_url = demo.launch(share=SHARE, debug=False)      # share=True: link công khai tạm thời. Giữ phiên Kaggle mở khi demo.
    if SHARE and not share_url:
        try:
            link = cloudflare_link(int(local_url.rsplit(":", 1)[1].strip("/")))
        except Exception as e:
            link = None
            print("Không tạo được tunnel dự phòng:", e)
        print("\n>>> LINK CHATBOT:", link or "chưa tạo được link - kiểm tra Internet đã bật chưa (Settings -> Internet on)")

try:
    audit_leak()          # kiểm tra lại rò rỉ với EXAMPLES của giao diện
except NameError:
    pass


/tmp/ipykernel_23/2731463141.py:73: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SUPEO - AI Stylist", css=CSS, theme=theme) as demo:
/tmp/ipykernel_23/2731463141.py:73: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="SUPEO - AI Stylist", css=CSS, theme=theme) as demo:
/tmp/ipykernel_23/2731463141.py:81: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chat = gr.Chatbot(value=render([{"role": "assistant", "content": GREETING}]), type="messages", height=460,
/tmp/ipykernel_23/2731463141.py:92: DeprecationWarning: Setting 'api_name=False' in event listeners will be r

* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://4535776f6e0010001c.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Đã so 22 câu dạy-bot + 190 câu test + 3 đoạn trả lời mẫu với benchmark (20 câu / 36 vế đáp án).
OK: audit_leak pass.


In [19]:
# ============ XUẤT KẾT QUẢ + NÉN ZIP ============
# Chạy SAU khi đã chạy các cell đo (9c'/9d/9e) và, nếu có, run_fresh_once. Tạo thư mục supeo_results/ rồi nén thành supeo_results.zip.
import os, glob, json, shutil, zipfile, time, re
import pandas as pd

OUT_DIR = "/kaggle/working/supeo_results" if os.path.isdir("/kaggle/working") else os.path.abspath("supeo_results")
shutil.rmtree(OUT_DIR, ignore_errors=True)
os.makedirs(OUT_DIR)

def _slug(s):
    import unicodedata
    s = unicodedata.normalize("NFD", str(s).replace("đ", "d").replace("Đ", "D"))
    s = "".join(ch for ch in s if unicodedata.category(ch) != "Mn")          # bỏ dấu tiếng Việt khỏi tên file
    return re.sub(r"[^0-9A-Za-z]+", "_", s).strip("_") or "x"

# 1) Báo cáo từng bộ test (CSV, mở được bằng Excel). RESULTS do run_regression / run_fresh_once ghi lại.
n_csv = 0
for (name, use_llm), (s, m) in globals().get("RESULTS", {}).items():
    mode = "gemini" if use_llm else "luat"
    if len(s):
        s.to_csv(f"{OUT_DIR}/{_slug(name)}__{mode}__mot_luot.csv", index=False, encoding="utf-8-sig"); n_csv += 1
    if len(m):
        m.to_csv(f"{OUT_DIR}/{_slug(name)}__{mode}__nhieu_luot.csv", index=False, encoding="utf-8-sig"); n_csv += 1

# 2) Tóm tắt điểm (một file duy nhất, đọc cho nhanh)
rows = []
for (name, use_llm), (s, m) in globals().get("RESULTS", {}).items():
    r = {"bo": name, "che_do": "Gemini+luat" if use_llm else "luat+regex"}
    if len(s):
        ok = s.intent_ok & s.slot_ok & s.action_ok & s.reply_ok
        r.update(so_cau=len(s), intent=round(s.intent_ok.mean(), 4), slot=round(s.slot_ok.mean(), 4),
                 action=round(s.action_ok.mean(), 4), noi_dung=round(s.reply_ok.mean(), 4), dung_het=round(ok.mean(), 4))
    if len(m):
        byc = m.groupby("conv").ok.all()
        r.update(so_luot=len(m), dung_tung_luot=round(m.ok.mean(), 4), hoi_thoai_dung_het=f"{int(byc.sum())}/{len(byc)}")
    rows.append(r)
if rows:
    pd.DataFrame(rows).to_csv(f"{OUT_DIR}/tom_tat_diem.csv", index=False, encoding="utf-8-sig")

# 3) File mã nguồn / dữ liệu / khóa chạy bộ C (chỉ chép file nào có)
for pattern in ["supeo_app.py", "supeo_api.py", "supeo_data_complete.xlsx", "fresh_C_ran.json", "supeo_chatbot_v5.ipynb",
                "supeo_test_suites.py", "/kaggle/working/supeo_app.py", "/kaggle/working/supeo_api.py", "/kaggle/working/*.ipynb"]:
    for f in glob.glob(pattern):
        dst = os.path.join(OUT_DIR, os.path.basename(f))
        if os.path.isfile(f) and not os.path.exists(dst):
            shutil.copy(f, dst)

# 4) Ghi chú chạy (ngày, chế độ, model) để sau này biết điểm đo ở điều kiện nào
note = {"thoi_gian_xuat": time.strftime("%Y-%m-%d %H:%M:%S"), "ngay_api_khi_do": globals().get("MEASURE_DATE"),
        "gemini_dang_bat": bool(globals().get("USE_LLM")), "gemini_model": (globals().get("_G") or {}).get("model"),
        "dst_gemini": globals().get("LLM_DST_MODE"), "nlg_gemini": globals().get("LLM_NLG")}
json.dump(note, open(f"{OUT_DIR}/ghi_chu_chay.json", "w"), ensure_ascii=False, indent=2)

# 5) Nén
ZIP_PATH = OUT_DIR + ".zip"
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z:
    for root, _, files in os.walk(OUT_DIR):
        for fn in files:
            p = os.path.join(root, fn)
            z.write(p, os.path.relpath(p, os.path.dirname(OUT_DIR)))
print(f"Đã nén {len(os.listdir(OUT_DIR))} file ({n_csv} báo cáo CSV) -> {ZIP_PATH} ({os.path.getsize(ZIP_PATH) / 1024:.0f} KB)")
for f in sorted(os.listdir(OUT_DIR)): print("  -", f)
try:
    from IPython.display import FileLink, display
    display(FileLink(os.path.relpath(ZIP_PATH)))          # Kaggle/Colab/Jupyter: bấm để tải
except Exception:
    pass


Đã nén 16 file (12 báo cáo CSV) -> /kaggle/working/supeo_results.zip (89 KB)
  - A_DEV_hoi_quy__gemini__mot_luot.csv
  - A_DEV_hoi_quy__gemini__nhieu_luot.csv
  - A_DEV_hoi_quy__luat__mot_luot.csv
  - A_DEV_hoi_quy__luat__nhieu_luot.csv
  - B_TEST_hoi_quy__gemini__mot_luot.csv
  - B_TEST_hoi_quy__gemini__nhieu_luot.csv
  - B_TEST_hoi_quy__luat__mot_luot.csv
  - B_TEST_hoi_quy__luat__nhieu_luot.csv
  - HELDOUT_CU_hoi_quy__gemini__mot_luot.csv
  - HELDOUT_CU_hoi_quy__gemini__nhieu_luot.csv
  - HELDOUT_CU_hoi_quy__luat__mot_luot.csv
  - HELDOUT_CU_hoi_quy__luat__nhieu_luot.csv
  - __notebook__.ipynb
  - ghi_chu_chay.json
  - supeo_api.py
  - tom_tat_diem.csv


/kaggle/working/supeo_results.zip